# Ansiedad — Servidor PyTorch | CUDA
Todos los modelos sobre datos de voz. Rutas: `../Clasificación Final/Ansiedad/wav/{0,1}`

In [ ]:
# pip install torch torchaudio librosa scikit-learn seaborn matplotlib numpy

# CNN-1D — VGG19-style Raw Waveform

## 10 s | Sin overlap

In [ ]:
import os
import gc
import numpy as np
import librosa
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (confusion_matrix, classification_report,
                             accuracy_score, precision_score, recall_score,
                             f1_score, roc_curve, auc)
from sklearn.utils import class_weight

# ══════════════════════════════════════════════════════════

BASE_PATH_0 = "../Clasificación Final/Ansiedad/wav/0"
BASE_PATH_1 = "../Clasificación Final/Ansiedad/wav/1"

SAMPLE_RATE         = 16000
SEGMENT_DURATION    = 10
SAMPLES_PER_SEGMENT = int(SEGMENT_DURATION * SAMPLE_RATE)  # auto-calculado
# Sin overlap para 10 s
N_FOLDS             = 5
BATCH_SIZE          = 128
EPOCHS              = 25

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Usando dispositivo: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
    torch.backends.cudnn.benchmark = True

def get_file_lists():
    files_0 = [os.path.join(BASE_PATH_0, f)
               for f in sorted(os.listdir(BASE_PATH_0)) if f.endswith('.wav')]
    files_1 = [os.path.join(BASE_PATH_1, f)
               for f in sorted(os.listdir(BASE_PATH_1)) if f.endswith('.wav')]
    files  = np.array(files_0 + files_1)
    labels = np.array([0]*len(files_0) + [1]*len(files_1), dtype=np.int64)
    return files, labels

all_files, all_labels = get_file_lists()
print(f"\nTotal archivos : {len(all_files)}")
print(f" - Sin Ansiedad (0): {np.sum(all_labels == 0)}")
print(f" - Con Ansiedad (1): {np.sum(all_labels == 1)}")
print(f"Segment: {SEGMENT_DURATION}s | Samples/seg: {SAMPLES_PER_SEGMENT}")

class WaveformDataset(Dataset):
    """Lazy loading — sin overlap para 10 s."""
    def __init__(self, file_list, label_list):
        self.samples = []
        for fp, lbl in zip(file_list, label_list):
            try:
                total = librosa.get_duration(path=fp)
                start = 0.0
                while start < total:
                    if total - start < SEGMENT_DURATION / 2:
                        break
                    self.samples.append((fp, start, int(lbl)))
                    start += SEGMENT_DURATION
            except Exception as e:
                print(f"[AVISO] {fp}: {e}")

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        fp, offset, label = self.samples[idx]
        audio, _ = librosa.load(fp, sr=SAMPLE_RATE, offset=offset,
                                duration=SEGMENT_DURATION)
        if len(audio) < SAMPLES_PER_SEGMENT:
            audio = np.pad(audio, (0, SAMPLES_PER_SEGMENT - len(audio)))
        audio = audio[:SAMPLES_PER_SEGMENT].astype(np.float32)
        audio /= (np.max(np.abs(audio)) + 1e-8)
        return torch.from_numpy(audio).unsqueeze(0), torch.tensor(label, dtype=torch.long)

def oversample_files(files, labels):
    f1 = files[labels == 1]; l1 = labels[labels == 1]
    diff = np.sum(labels == 0) - np.sum(labels == 1)
    if diff > 0:
        idx    = np.random.choice(len(f1), size=diff, replace=True)
        files  = np.concatenate([files,  f1[idx]])
        labels = np.concatenate([labels, l1[idx]])
    return files, labels

MODEL_NAME = "CNN-1D VGG"

class CNN1D_VGG(nn.Module):
    """VGG19-style 1D adaptado a Raw Waveform.
    Block 1 usa stride grande para submuestrear; bloques 2-5 VGG estándar.
    GlobalAveragePooling reemplaza Flatten+FC4096 de VGG original."""

    def __init__(self, input_length):
        super().__init__()
        def vgg_block(in_ch, out_ch, n_convs, k):
            layers = []
            for i in range(n_convs):
                layers += [nn.Conv1d(in_ch if i == 0 else out_ch, out_ch, k, padding=k//2),
                           nn.ReLU(inplace=True), nn.BatchNorm1d(out_ch)]
            layers += [nn.MaxPool1d(4), nn.Dropout(0.2)]
            return layers

        self.blocks = nn.Sequential(
            # Bloque 1 — stride agresivo + una conv de refinamiento
            nn.Conv1d(1, 64, 320, stride=40, padding=160),
            nn.ReLU(inplace=True), nn.BatchNorm1d(64),
            nn.Conv1d(64, 64, 9, padding=4),
            nn.ReLU(inplace=True), nn.BatchNorm1d(64),
            nn.MaxPool1d(4), nn.Dropout(0.2),
            # Bloque 2 — 128
            *vgg_block(64, 128, 2, 9),
            # Bloque 3 — 256 (4 conv como VGG19)
            *vgg_block(128, 256, 4, 7),
            # Bloque 4 — 512
            *vgg_block(256, 512, 4, 5),
            # Bloque 5 — 512
            *vgg_block(512, 512, 4, 3),
        )
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.classifier = nn.Sequential(
            nn.Linear(512, 256), nn.ReLU(inplace=True), nn.Dropout(0.5),
            nn.Linear(256, 128), nn.ReLU(inplace=True), nn.Dropout(0.4),
            nn.Linear(128, 1), nn.Sigmoid(),
        )

    def forward(self, x):
        x = self.blocks(x)          # (B, 512, L)
        x = self.pool(x).squeeze(2) # (B, 512)
        return self.classifier(x).squeeze(1)

def build_model():
    return CNN1D_VGG(SAMPLES_PER_SEGMENT)

def train_one_epoch(model, loader, optimizer, criterion, cw_tensor):
    model.train()
    total_loss = 0.0
    for xb, yb in loader:
        xb = xb.to(DEVICE, non_blocking=True)
        yb = yb.to(DEVICE, non_blocking=True).float()
        optimizer.zero_grad(set_to_none=True)
        preds = model(xb)
        sample_w = cw_tensor[yb.long()]
        loss = (criterion(preds, yb) * sample_w).mean()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

@torch.no_grad()
def predict_file(model, file_path):
    model.eval()
    segs = []
    try:
        audio, _ = librosa.load(file_path, sr=SAMPLE_RATE)
        hop = SAMPLES_PER_SEGMENT
        for start in range(0, len(audio), hop):
            seg = audio[start:start + SAMPLES_PER_SEGMENT]
            if len(seg) < SAMPLES_PER_SEGMENT // 2: continue
            if len(seg) < SAMPLES_PER_SEGMENT:
                seg = np.pad(seg, (0, SAMPLES_PER_SEGMENT - len(seg)))
            seg = seg.astype(np.float32)
            seg /= (np.max(np.abs(seg)) + 1e-8)
            segs.append(seg)
    except Exception as e:
        print(f"  Error inferencia {file_path}: {e}"); return None
    if not segs: return None
    probs = []
    for b in range(0, len(segs), BATCH_SIZE):
        batch = np.stack(segs[b:b + BATCH_SIZE])
        bt = torch.from_numpy(batch).unsqueeze(1).to(DEVICE)
        out = model(bt).cpu().numpy()
        probs.extend(out.tolist())
    return float(np.mean(probs))

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
global_y_true, global_y_pred, global_y_prob = [], [], []

print(f"\nINICIANDO CROSS-VALIDATION {MODEL_NAME} | {SEGMENT_DURATION}s | {N_FOLDS} FOLDS\n")

for fold_idx, (tr_idx, te_idx) in enumerate(skf.split(all_files, all_labels)):
    print(f"\n{'─'*55}\n  FOLD {fold_idx+1}/{N_FOLDS}\n{'─'*55}")

    Xtr_f, Xte_f = all_files[tr_idx], all_files[te_idx]
    ytr_f, yte_f = all_labels[tr_idx], all_labels[te_idx]
    Xtr_f, ytr_f = oversample_files(Xtr_f, ytr_f)

    train_ds = WaveformDataset(Xtr_f, ytr_f)
    train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=8, pin_memory=True, persistent_workers=True)
    print(f"  Segmentos train: {len(train_ds)}")

    weights  = class_weight.compute_class_weight('balanced',
                   classes=np.unique(ytr_f), y=ytr_f)
    cw_tensor = torch.tensor(weights, dtype=torch.float32).to(DEVICE)

    model     = build_model().to(DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=1e-4)
    criterion = nn.BCELoss(reduction='none')
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=3, min_lr=1e-7)

    if fold_idx == 0:
        n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
        print(f"  Parámetros entrenables: {n_params:,}\n")

    best_loss, patience_cnt, PATIENCE, best_state = float('inf'), 0, 5, None
    for epoch in range(EPOCHS):
        loss = train_one_epoch(model, train_dl, optimizer, criterion, cw_tensor)
        scheduler.step(loss)
        if loss < best_loss:
            best_loss = loss; patience_cnt = 0
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            mark = "✓"
        else:
            patience_cnt += 1
            mark = f"({patience_cnt}/{PATIENCE})"
        print(f"  Epoch {epoch+1:02d}/{EPOCHS} — loss: {loss:.4f}  best: {best_loss:.4f}  {mark}")
        if patience_cnt >= PATIENCE:
            print("  ► Early stopping activado."); break

    model.load_state_dict(best_state)

    for i, fp in enumerate(Xte_f):
        avg_prob = predict_file(model, fp)
        if avg_prob is None: continue
        global_y_true.append(int(yte_f[i]))
        global_y_pred.append(1 if avg_prob > 0.33 else 0)
        global_y_prob.append(avg_prob)

    del model, train_ds, train_dl, best_state, cw_tensor
    gc.collect(); torch.cuda.empty_cache()

print("\n" + "="*60)
print(f"RESULTADOS {MODEL_NAME} | {SEGMENT_DURATION}s  ({len(global_y_true)} pacientes)")
print("="*60)

cm = confusion_matrix(global_y_true, global_y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Sin Ansiedad', 'Con Ansiedad'],
            yticklabels=['Sin Ansiedad', 'Con Ansiedad'])
plt.title(f'Confusión {MODEL_NAME} {SEGMENT_DURATION}s ({N_FOLDS}-Fold CV)')
plt.ylabel('Real'); plt.xlabel('Predicción')
plt.tight_layout()
plt.savefig(f'confusion_{MODEL_NAME.replace(" ","_")}_{SEGMENT_DURATION}s.png', dpi=150)
plt.show()

acc      = accuracy_score(global_y_true, global_y_pred)
prec     = precision_score(global_y_true, global_y_pred, zero_division=0)
rec      = recall_score(global_y_true, global_y_pred, zero_division=0)
f1       = f1_score(global_y_true, global_y_pred, zero_division=0)
f1_macro = f1_score(global_y_true, global_y_pred, average='macro', zero_division=0)
print(f"\nAccuracy :  {acc:.4f}")
print(f"Precision:  {prec:.4f}")
print(f"Recall   :  {rec:.4f}")
print(f"F1-Ans   :  {f1:.4f}  (clase positiva)")
print(f"F1-Macro :  {f1_macro:.4f}")
print()
print(classification_report(global_y_true, global_y_pred,
                             target_names=['Sin Ansiedad', 'Con Ansiedad']))

fpr, tpr, _ = roc_curve(global_y_true, global_y_prob)
roc_auc = auc(fpr, tpr)
plt.figure(figsize=(7, 6))
plt.plot(fpr, tpr, lw=2, label=f'AUC = {roc_auc:.2f}')
plt.plot([0,1],[0,1],'navy',lw=2,linestyle='--')
plt.xlabel('Falsos Positivos'); plt.ylabel('Verdaderos Positivos')
plt.title(f'ROC — {MODEL_NAME} {SEGMENT_DURATION}s')
plt.legend(loc='lower right'); plt.tight_layout()
plt.savefig(f'roc_{MODEL_NAME.replace(" ","_")}_{SEGMENT_DURATION}s.png', dpi=150)
plt.show()


## 2 s | Overlap 50%

In [ ]:
import os
import gc
import numpy as np
import librosa
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (confusion_matrix, classification_report,
                             accuracy_score, precision_score, recall_score,
                             f1_score, roc_curve, auc)
from sklearn.utils import class_weight

# ══════════════════════════════════════════════════════════

BASE_PATH_0 = "../Clasificación Final/Ansiedad/wav/0"
BASE_PATH_1 = "../Clasificación Final/Ansiedad/wav/1"

SAMPLE_RATE         = 16000
SEGMENT_DURATION    = 2
SAMPLES_PER_SEGMENT = int(SEGMENT_DURATION * SAMPLE_RATE)  # auto-calculado
HOP_SIZE            = SAMPLES_PER_SEGMENT // 2  # 50% overlap → 22050
N_FOLDS             = 5
BATCH_SIZE          = 128
EPOCHS              = 25

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Usando dispositivo: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
    torch.backends.cudnn.benchmark = True

def get_file_lists():
    files_0 = [os.path.join(BASE_PATH_0, f)
               for f in sorted(os.listdir(BASE_PATH_0)) if f.endswith('.wav')]
    files_1 = [os.path.join(BASE_PATH_1, f)
               for f in sorted(os.listdir(BASE_PATH_1)) if f.endswith('.wav')]
    files  = np.array(files_0 + files_1)
    labels = np.array([0]*len(files_0) + [1]*len(files_1), dtype=np.int64)
    return files, labels

all_files, all_labels = get_file_lists()
print(f"\nTotal archivos : {len(all_files)}")
print(f" - Sin Ansiedad (0): {np.sum(all_labels == 0)}")
print(f" - Con Ansiedad (1): {np.sum(all_labels == 1)}")
print(f"Segment: {SEGMENT_DURATION}s | Samples/seg: {SAMPLES_PER_SEGMENT}")

class WaveformDataset(Dataset):
    """Lazy loading — overlap 50%%."""
    def __init__(self, file_list, label_list):
        self.samples = []
        for fp, lbl in zip(file_list, label_list):
            try:
                total = librosa.get_duration(path=fp)
                n_segs = max(1, int((total * SAMPLE_RATE - SAMPLES_PER_SEGMENT) / HOP_SIZE) + 1)
                for i in range(n_segs):
                    offset = i * HOP_SIZE / SAMPLE_RATE
                    self.samples.append((fp, offset, int(lbl)))
            except Exception as e:
                print(f"[AVISO] {fp}: {e}")

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        fp, offset, label = self.samples[idx]
        audio, _ = librosa.load(fp, sr=SAMPLE_RATE, offset=offset,
                                duration=SEGMENT_DURATION)
        if len(audio) < SAMPLES_PER_SEGMENT:
            audio = np.pad(audio, (0, SAMPLES_PER_SEGMENT - len(audio)))
        audio = audio[:SAMPLES_PER_SEGMENT].astype(np.float32)
        audio /= (np.max(np.abs(audio)) + 1e-8)
        return torch.from_numpy(audio).unsqueeze(0), torch.tensor(label, dtype=torch.long)

def oversample_files(files, labels):
    f1 = files[labels == 1]; l1 = labels[labels == 1]
    diff = np.sum(labels == 0) - np.sum(labels == 1)
    if diff > 0:
        idx    = np.random.choice(len(f1), size=diff, replace=True)
        files  = np.concatenate([files,  f1[idx]])
        labels = np.concatenate([labels, l1[idx]])
    return files, labels

MODEL_NAME = "CNN-1D VGG"

class CNN1D_VGG(nn.Module):
    """VGG19-style 1D adaptado a Raw Waveform.
    Block 1 usa stride grande para submuestrear; bloques 2-5 VGG estándar.
    GlobalAveragePooling reemplaza Flatten+FC4096 de VGG original."""

    def __init__(self, input_length):
        super().__init__()
        def vgg_block(in_ch, out_ch, n_convs, k):
            layers = []
            for i in range(n_convs):
                layers += [nn.Conv1d(in_ch if i == 0 else out_ch, out_ch, k, padding=k//2),
                           nn.ReLU(inplace=True), nn.BatchNorm1d(out_ch)]
            layers += [nn.MaxPool1d(4), nn.Dropout(0.2)]
            return layers

        self.blocks = nn.Sequential(
            # Bloque 1 — stride agresivo + una conv de refinamiento
            nn.Conv1d(1, 64, 80, stride=10, padding=40),
            nn.ReLU(inplace=True), nn.BatchNorm1d(64),
            nn.Conv1d(64, 64, 9, padding=4),
            nn.ReLU(inplace=True), nn.BatchNorm1d(64),
            nn.MaxPool1d(4), nn.Dropout(0.2),
            # Bloque 2 — 128
            *vgg_block(64, 128, 2, 9),
            # Bloque 3 — 256 (4 conv como VGG19)
            *vgg_block(128, 256, 4, 7),
            # Bloque 4 — 512
            *vgg_block(256, 512, 4, 5),
            # Bloque 5 — 512
            *vgg_block(512, 512, 4, 3),
        )
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.classifier = nn.Sequential(
            nn.Linear(512, 256), nn.ReLU(inplace=True), nn.Dropout(0.5),
            nn.Linear(256, 128), nn.ReLU(inplace=True), nn.Dropout(0.4),
            nn.Linear(128, 1), nn.Sigmoid(),
        )

    def forward(self, x):
        x = self.blocks(x)          # (B, 512, L)
        x = self.pool(x).squeeze(2) # (B, 512)
        return self.classifier(x).squeeze(1)

def build_model():
    return CNN1D_VGG(SAMPLES_PER_SEGMENT)

def train_one_epoch(model, loader, optimizer, criterion, cw_tensor):
    model.train()
    total_loss = 0.0
    for xb, yb in loader:
        xb = xb.to(DEVICE, non_blocking=True)
        yb = yb.to(DEVICE, non_blocking=True).float()
        optimizer.zero_grad(set_to_none=True)
        preds = model(xb)
        sample_w = cw_tensor[yb.long()]
        loss = (criterion(preds, yb) * sample_w).mean()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

@torch.no_grad()
def predict_file(model, file_path):
    model.eval()
    segs = []
    try:
        audio, _ = librosa.load(file_path, sr=SAMPLE_RATE)
        hop = HOP_SIZE
        for start in range(0, len(audio), hop):
            seg = audio[start:start + SAMPLES_PER_SEGMENT]
            if len(seg) < SAMPLES_PER_SEGMENT // 2: continue
            if len(seg) < SAMPLES_PER_SEGMENT:
                seg = np.pad(seg, (0, SAMPLES_PER_SEGMENT - len(seg)))
            seg = seg.astype(np.float32)
            seg /= (np.max(np.abs(seg)) + 1e-8)
            segs.append(seg)
    except Exception as e:
        print(f"  Error inferencia {file_path}: {e}"); return None
    if not segs: return None
    probs = []
    for b in range(0, len(segs), BATCH_SIZE):
        batch = np.stack(segs[b:b + BATCH_SIZE])
        bt = torch.from_numpy(batch).unsqueeze(1).to(DEVICE)
        out = model(bt).cpu().numpy()
        probs.extend(out.tolist())
    return float(np.mean(probs))

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
global_y_true, global_y_pred, global_y_prob = [], [], []

print(f"\nINICIANDO CROSS-VALIDATION {MODEL_NAME} | {SEGMENT_DURATION}s | {N_FOLDS} FOLDS\n")

for fold_idx, (tr_idx, te_idx) in enumerate(skf.split(all_files, all_labels)):
    print(f"\n{'─'*55}\n  FOLD {fold_idx+1}/{N_FOLDS}\n{'─'*55}")

    Xtr_f, Xte_f = all_files[tr_idx], all_files[te_idx]
    ytr_f, yte_f = all_labels[tr_idx], all_labels[te_idx]
    Xtr_f, ytr_f = oversample_files(Xtr_f, ytr_f)

    train_ds = WaveformDataset(Xtr_f, ytr_f)
    train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=8, pin_memory=True, persistent_workers=True)
    print(f"  Segmentos train: {len(train_ds)}")

    weights  = class_weight.compute_class_weight('balanced',
                   classes=np.unique(ytr_f), y=ytr_f)
    cw_tensor = torch.tensor(weights, dtype=torch.float32).to(DEVICE)

    model     = build_model().to(DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=1e-4)
    criterion = nn.BCELoss(reduction='none')
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=3, min_lr=1e-7)

    if fold_idx == 0:
        n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
        print(f"  Parámetros entrenables: {n_params:,}\n")

    best_loss, patience_cnt, PATIENCE, best_state = float('inf'), 0, 5, None
    for epoch in range(EPOCHS):
        loss = train_one_epoch(model, train_dl, optimizer, criterion, cw_tensor)
        scheduler.step(loss)
        if loss < best_loss:
            best_loss = loss; patience_cnt = 0
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            mark = "✓"
        else:
            patience_cnt += 1
            mark = f"({patience_cnt}/{PATIENCE})"
        print(f"  Epoch {epoch+1:02d}/{EPOCHS} — loss: {loss:.4f}  best: {best_loss:.4f}  {mark}")
        if patience_cnt >= PATIENCE:
            print("  ► Early stopping activado."); break

    model.load_state_dict(best_state)

    for i, fp in enumerate(Xte_f):
        avg_prob = predict_file(model, fp)
        if avg_prob is None: continue
        global_y_true.append(int(yte_f[i]))
        global_y_pred.append(1 if avg_prob > 0.33 else 0)
        global_y_prob.append(avg_prob)

    del model, train_ds, train_dl, best_state, cw_tensor
    gc.collect(); torch.cuda.empty_cache()

print("\n" + "="*60)
print(f"RESULTADOS {MODEL_NAME} | {SEGMENT_DURATION}s  ({len(global_y_true)} pacientes)")
print("="*60)

cm = confusion_matrix(global_y_true, global_y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Sin Ansiedad', 'Con Ansiedad'],
            yticklabels=['Sin Ansiedad', 'Con Ansiedad'])
plt.title(f'Confusión {MODEL_NAME} {SEGMENT_DURATION}s ({N_FOLDS}-Fold CV)')
plt.ylabel('Real'); plt.xlabel('Predicción')
plt.tight_layout()
plt.savefig(f'confusion_{MODEL_NAME.replace(" ","_")}_{SEGMENT_DURATION}s.png', dpi=150)
plt.show()

acc      = accuracy_score(global_y_true, global_y_pred)
prec     = precision_score(global_y_true, global_y_pred, zero_division=0)
rec      = recall_score(global_y_true, global_y_pred, zero_division=0)
f1       = f1_score(global_y_true, global_y_pred, zero_division=0)
f1_macro = f1_score(global_y_true, global_y_pred, average='macro', zero_division=0)
print(f"\nAccuracy :  {acc:.4f}")
print(f"Precision:  {prec:.4f}")
print(f"Recall   :  {rec:.4f}")
print(f"F1-Ans   :  {f1:.4f}  (clase positiva)")
print(f"F1-Macro :  {f1_macro:.4f}")
print()
print(classification_report(global_y_true, global_y_pred,
                             target_names=['Sin Ansiedad', 'Con Ansiedad']))

fpr, tpr, _ = roc_curve(global_y_true, global_y_prob)
roc_auc = auc(fpr, tpr)
plt.figure(figsize=(7, 6))
plt.plot(fpr, tpr, lw=2, label=f'AUC = {roc_auc:.2f}')
plt.plot([0,1],[0,1],'navy',lw=2,linestyle='--')
plt.xlabel('Falsos Positivos'); plt.ylabel('Verdaderos Positivos')
plt.title(f'ROC — {MODEL_NAME} {SEGMENT_DURATION}s')
plt.legend(loc='lower right'); plt.tight_layout()
plt.savefig(f'roc_{MODEL_NAME.replace(" ","_")}_{SEGMENT_DURATION}s.png', dpi=150)
plt.show()


## 3 s | Overlap 50%

In [ ]:
import os
import gc
import numpy as np
import librosa
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (confusion_matrix, classification_report,
                             accuracy_score, precision_score, recall_score,
                             f1_score, roc_curve, auc)
from sklearn.utils import class_weight

# ══════════════════════════════════════════════════════════

BASE_PATH_0 = "../Clasificación Final/Ansiedad/wav/0"
BASE_PATH_1 = "../Clasificación Final/Ansiedad/wav/1"

SAMPLE_RATE         = 16000
SEGMENT_DURATION    = 3
SAMPLES_PER_SEGMENT = int(SEGMENT_DURATION * SAMPLE_RATE)  # auto-calculado
HOP_SIZE            = SAMPLES_PER_SEGMENT // 2  # 50% overlap → 33075
N_FOLDS             = 5
BATCH_SIZE          = 128
EPOCHS              = 25

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Usando dispositivo: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
    torch.backends.cudnn.benchmark = True

def get_file_lists():
    files_0 = [os.path.join(BASE_PATH_0, f)
               for f in sorted(os.listdir(BASE_PATH_0)) if f.endswith('.wav')]
    files_1 = [os.path.join(BASE_PATH_1, f)
               for f in sorted(os.listdir(BASE_PATH_1)) if f.endswith('.wav')]
    files  = np.array(files_0 + files_1)
    labels = np.array([0]*len(files_0) + [1]*len(files_1), dtype=np.int64)
    return files, labels

all_files, all_labels = get_file_lists()
print(f"\nTotal archivos : {len(all_files)}")
print(f" - Sin Ansiedad (0): {np.sum(all_labels == 0)}")
print(f" - Con Ansiedad (1): {np.sum(all_labels == 1)}")
print(f"Segment: {SEGMENT_DURATION}s | Samples/seg: {SAMPLES_PER_SEGMENT}")

class WaveformDataset(Dataset):
    """Lazy loading — overlap 50%%."""
    def __init__(self, file_list, label_list):
        self.samples = []
        for fp, lbl in zip(file_list, label_list):
            try:
                total = librosa.get_duration(path=fp)
                n_segs = max(1, int((total * SAMPLE_RATE - SAMPLES_PER_SEGMENT) / HOP_SIZE) + 1)
                for i in range(n_segs):
                    offset = i * HOP_SIZE / SAMPLE_RATE
                    self.samples.append((fp, offset, int(lbl)))
            except Exception as e:
                print(f"[AVISO] {fp}: {e}")

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        fp, offset, label = self.samples[idx]
        audio, _ = librosa.load(fp, sr=SAMPLE_RATE, offset=offset,
                                duration=SEGMENT_DURATION)
        if len(audio) < SAMPLES_PER_SEGMENT:
            audio = np.pad(audio, (0, SAMPLES_PER_SEGMENT - len(audio)))
        audio = audio[:SAMPLES_PER_SEGMENT].astype(np.float32)
        audio /= (np.max(np.abs(audio)) + 1e-8)
        return torch.from_numpy(audio).unsqueeze(0), torch.tensor(label, dtype=torch.long)

def oversample_files(files, labels):
    f1 = files[labels == 1]; l1 = labels[labels == 1]
    diff = np.sum(labels == 0) - np.sum(labels == 1)
    if diff > 0:
        idx    = np.random.choice(len(f1), size=diff, replace=True)
        files  = np.concatenate([files,  f1[idx]])
        labels = np.concatenate([labels, l1[idx]])
    return files, labels

MODEL_NAME = "CNN-1D VGG"

class CNN1D_VGG(nn.Module):
    """VGG19-style 1D adaptado a Raw Waveform.
    Block 1 usa stride grande para submuestrear; bloques 2-5 VGG estándar.
    GlobalAveragePooling reemplaza Flatten+FC4096 de VGG original."""

    def __init__(self, input_length):
        super().__init__()
        def vgg_block(in_ch, out_ch, n_convs, k):
            layers = []
            for i in range(n_convs):
                layers += [nn.Conv1d(in_ch if i == 0 else out_ch, out_ch, k, padding=k//2),
                           nn.ReLU(inplace=True), nn.BatchNorm1d(out_ch)]
            layers += [nn.MaxPool1d(4), nn.Dropout(0.2)]
            return layers

        self.blocks = nn.Sequential(
            # Bloque 1 — stride agresivo + una conv de refinamiento
            nn.Conv1d(1, 64, 160, stride=20, padding=80),
            nn.ReLU(inplace=True), nn.BatchNorm1d(64),
            nn.Conv1d(64, 64, 9, padding=4),
            nn.ReLU(inplace=True), nn.BatchNorm1d(64),
            nn.MaxPool1d(4), nn.Dropout(0.2),
            # Bloque 2 — 128
            *vgg_block(64, 128, 2, 9),
            # Bloque 3 — 256 (4 conv como VGG19)
            *vgg_block(128, 256, 4, 7),
            # Bloque 4 — 512
            *vgg_block(256, 512, 4, 5),
            # Bloque 5 — 512
            *vgg_block(512, 512, 4, 3),
        )
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.classifier = nn.Sequential(
            nn.Linear(512, 256), nn.ReLU(inplace=True), nn.Dropout(0.5),
            nn.Linear(256, 128), nn.ReLU(inplace=True), nn.Dropout(0.4),
            nn.Linear(128, 1), nn.Sigmoid(),
        )

    def forward(self, x):
        x = self.blocks(x)          # (B, 512, L)
        x = self.pool(x).squeeze(2) # (B, 512)
        return self.classifier(x).squeeze(1)

def build_model():
    return CNN1D_VGG(SAMPLES_PER_SEGMENT)

def train_one_epoch(model, loader, optimizer, criterion, cw_tensor):
    model.train()
    total_loss = 0.0
    for xb, yb in loader:
        xb = xb.to(DEVICE, non_blocking=True)
        yb = yb.to(DEVICE, non_blocking=True).float()
        optimizer.zero_grad(set_to_none=True)
        preds = model(xb)
        sample_w = cw_tensor[yb.long()]
        loss = (criterion(preds, yb) * sample_w).mean()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

@torch.no_grad()
def predict_file(model, file_path):
    model.eval()
    segs = []
    try:
        audio, _ = librosa.load(file_path, sr=SAMPLE_RATE)
        hop = HOP_SIZE
        for start in range(0, len(audio), hop):
            seg = audio[start:start + SAMPLES_PER_SEGMENT]
            if len(seg) < SAMPLES_PER_SEGMENT // 2: continue
            if len(seg) < SAMPLES_PER_SEGMENT:
                seg = np.pad(seg, (0, SAMPLES_PER_SEGMENT - len(seg)))
            seg = seg.astype(np.float32)
            seg /= (np.max(np.abs(seg)) + 1e-8)
            segs.append(seg)
    except Exception as e:
        print(f"  Error inferencia {file_path}: {e}"); return None
    if not segs: return None
    probs = []
    for b in range(0, len(segs), BATCH_SIZE):
        batch = np.stack(segs[b:b + BATCH_SIZE])
        bt = torch.from_numpy(batch).unsqueeze(1).to(DEVICE)
        out = model(bt).cpu().numpy()
        probs.extend(out.tolist())
    return float(np.mean(probs))

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
global_y_true, global_y_pred, global_y_prob = [], [], []

print(f"\nINICIANDO CROSS-VALIDATION {MODEL_NAME} | {SEGMENT_DURATION}s | {N_FOLDS} FOLDS\n")

for fold_idx, (tr_idx, te_idx) in enumerate(skf.split(all_files, all_labels)):
    print(f"\n{'─'*55}\n  FOLD {fold_idx+1}/{N_FOLDS}\n{'─'*55}")

    Xtr_f, Xte_f = all_files[tr_idx], all_files[te_idx]
    ytr_f, yte_f = all_labels[tr_idx], all_labels[te_idx]
    Xtr_f, ytr_f = oversample_files(Xtr_f, ytr_f)

    train_ds = WaveformDataset(Xtr_f, ytr_f)
    train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=8, pin_memory=True, persistent_workers=True)
    print(f"  Segmentos train: {len(train_ds)}")

    weights  = class_weight.compute_class_weight('balanced',
                   classes=np.unique(ytr_f), y=ytr_f)
    cw_tensor = torch.tensor(weights, dtype=torch.float32).to(DEVICE)

    model     = build_model().to(DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=1e-4)
    criterion = nn.BCELoss(reduction='none')
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=3, min_lr=1e-7)

    if fold_idx == 0:
        n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
        print(f"  Parámetros entrenables: {n_params:,}\n")

    best_loss, patience_cnt, PATIENCE, best_state = float('inf'), 0, 5, None
    for epoch in range(EPOCHS):
        loss = train_one_epoch(model, train_dl, optimizer, criterion, cw_tensor)
        scheduler.step(loss)
        if loss < best_loss:
            best_loss = loss; patience_cnt = 0
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            mark = "✓"
        else:
            patience_cnt += 1
            mark = f"({patience_cnt}/{PATIENCE})"
        print(f"  Epoch {epoch+1:02d}/{EPOCHS} — loss: {loss:.4f}  best: {best_loss:.4f}  {mark}")
        if patience_cnt >= PATIENCE:
            print("  ► Early stopping activado."); break

    model.load_state_dict(best_state)

    for i, fp in enumerate(Xte_f):
        avg_prob = predict_file(model, fp)
        if avg_prob is None: continue
        global_y_true.append(int(yte_f[i]))
        global_y_pred.append(1 if avg_prob > 0.33 else 0)
        global_y_prob.append(avg_prob)

    del model, train_ds, train_dl, best_state, cw_tensor
    gc.collect(); torch.cuda.empty_cache()

print("\n" + "="*60)
print(f"RESULTADOS {MODEL_NAME} | {SEGMENT_DURATION}s  ({len(global_y_true)} pacientes)")
print("="*60)

cm = confusion_matrix(global_y_true, global_y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Sin Ansiedad', 'Con Ansiedad'],
            yticklabels=['Sin Ansiedad', 'Con Ansiedad'])
plt.title(f'Confusión {MODEL_NAME} {SEGMENT_DURATION}s ({N_FOLDS}-Fold CV)')
plt.ylabel('Real'); plt.xlabel('Predicción')
plt.tight_layout()
plt.savefig(f'confusion_{MODEL_NAME.replace(" ","_")}_{SEGMENT_DURATION}s.png', dpi=150)
plt.show()

acc      = accuracy_score(global_y_true, global_y_pred)
prec     = precision_score(global_y_true, global_y_pred, zero_division=0)
rec      = recall_score(global_y_true, global_y_pred, zero_division=0)
f1       = f1_score(global_y_true, global_y_pred, zero_division=0)
f1_macro = f1_score(global_y_true, global_y_pred, average='macro', zero_division=0)
print(f"\nAccuracy :  {acc:.4f}")
print(f"Precision:  {prec:.4f}")
print(f"Recall   :  {rec:.4f}")
print(f"F1-Ans   :  {f1:.4f}  (clase positiva)")
print(f"F1-Macro :  {f1_macro:.4f}")
print()
print(classification_report(global_y_true, global_y_pred,
                             target_names=['Sin Ansiedad', 'Con Ansiedad']))

fpr, tpr, _ = roc_curve(global_y_true, global_y_prob)
roc_auc = auc(fpr, tpr)
plt.figure(figsize=(7, 6))
plt.plot(fpr, tpr, lw=2, label=f'AUC = {roc_auc:.2f}')
plt.plot([0,1],[0,1],'navy',lw=2,linestyle='--')
plt.xlabel('Falsos Positivos'); plt.ylabel('Verdaderos Positivos')
plt.title(f'ROC — {MODEL_NAME} {SEGMENT_DURATION}s')
plt.legend(loc='lower right'); plt.tight_layout()
plt.savefig(f'roc_{MODEL_NAME.replace(" ","_")}_{SEGMENT_DURATION}s.png', dpi=150)
plt.show()


## 4 s | Overlap 50%

In [ ]:
import os
import gc
import numpy as np
import librosa
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (confusion_matrix, classification_report,
                             accuracy_score, precision_score, recall_score,
                             f1_score, roc_curve, auc)
from sklearn.utils import class_weight

# ══════════════════════════════════════════════════════════

BASE_PATH_0 = "../Clasificación Final/Ansiedad/wav/0"
BASE_PATH_1 = "../Clasificación Final/Ansiedad/wav/1"

SAMPLE_RATE         = 16000
SEGMENT_DURATION    = 4
SAMPLES_PER_SEGMENT = int(SEGMENT_DURATION * SAMPLE_RATE)  # auto-calculado
HOP_SIZE            = SAMPLES_PER_SEGMENT // 2  # 50% overlap → 44100
N_FOLDS             = 5
BATCH_SIZE          = 128
EPOCHS              = 25

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Usando dispositivo: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
    torch.backends.cudnn.benchmark = True

def get_file_lists():
    files_0 = [os.path.join(BASE_PATH_0, f)
               for f in sorted(os.listdir(BASE_PATH_0)) if f.endswith('.wav')]
    files_1 = [os.path.join(BASE_PATH_1, f)
               for f in sorted(os.listdir(BASE_PATH_1)) if f.endswith('.wav')]
    files  = np.array(files_0 + files_1)
    labels = np.array([0]*len(files_0) + [1]*len(files_1), dtype=np.int64)
    return files, labels

all_files, all_labels = get_file_lists()
print(f"\nTotal archivos : {len(all_files)}")
print(f" - Sin Ansiedad (0): {np.sum(all_labels == 0)}")
print(f" - Con Ansiedad (1): {np.sum(all_labels == 1)}")
print(f"Segment: {SEGMENT_DURATION}s | Samples/seg: {SAMPLES_PER_SEGMENT}")

class WaveformDataset(Dataset):
    """Lazy loading — overlap 50%%."""
    def __init__(self, file_list, label_list):
        self.samples = []
        for fp, lbl in zip(file_list, label_list):
            try:
                total = librosa.get_duration(path=fp)
                n_segs = max(1, int((total * SAMPLE_RATE - SAMPLES_PER_SEGMENT) / HOP_SIZE) + 1)
                for i in range(n_segs):
                    offset = i * HOP_SIZE / SAMPLE_RATE
                    self.samples.append((fp, offset, int(lbl)))
            except Exception as e:
                print(f"[AVISO] {fp}: {e}")

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        fp, offset, label = self.samples[idx]
        audio, _ = librosa.load(fp, sr=SAMPLE_RATE, offset=offset,
                                duration=SEGMENT_DURATION)
        if len(audio) < SAMPLES_PER_SEGMENT:
            audio = np.pad(audio, (0, SAMPLES_PER_SEGMENT - len(audio)))
        audio = audio[:SAMPLES_PER_SEGMENT].astype(np.float32)
        audio /= (np.max(np.abs(audio)) + 1e-8)
        return torch.from_numpy(audio).unsqueeze(0), torch.tensor(label, dtype=torch.long)

def oversample_files(files, labels):
    f1 = files[labels == 1]; l1 = labels[labels == 1]
    diff = np.sum(labels == 0) - np.sum(labels == 1)
    if diff > 0:
        idx    = np.random.choice(len(f1), size=diff, replace=True)
        files  = np.concatenate([files,  f1[idx]])
        labels = np.concatenate([labels, l1[idx]])
    return files, labels

MODEL_NAME = "CNN-1D VGG"

class CNN1D_VGG(nn.Module):
    """VGG19-style 1D adaptado a Raw Waveform.
    Block 1 usa stride grande para submuestrear; bloques 2-5 VGG estándar.
    GlobalAveragePooling reemplaza Flatten+FC4096 de VGG original."""

    def __init__(self, input_length):
        super().__init__()
        def vgg_block(in_ch, out_ch, n_convs, k):
            layers = []
            for i in range(n_convs):
                layers += [nn.Conv1d(in_ch if i == 0 else out_ch, out_ch, k, padding=k//2),
                           nn.ReLU(inplace=True), nn.BatchNorm1d(out_ch)]
            layers += [nn.MaxPool1d(4), nn.Dropout(0.2)]
            return layers

        self.blocks = nn.Sequential(
            # Bloque 1 — stride agresivo + una conv de refinamiento
            nn.Conv1d(1, 64, 160, stride=20, padding=80),
            nn.ReLU(inplace=True), nn.BatchNorm1d(64),
            nn.Conv1d(64, 64, 9, padding=4),
            nn.ReLU(inplace=True), nn.BatchNorm1d(64),
            nn.MaxPool1d(4), nn.Dropout(0.2),
            # Bloque 2 — 128
            *vgg_block(64, 128, 2, 9),
            # Bloque 3 — 256 (4 conv como VGG19)
            *vgg_block(128, 256, 4, 7),
            # Bloque 4 — 512
            *vgg_block(256, 512, 4, 5),
            # Bloque 5 — 512
            *vgg_block(512, 512, 4, 3),
        )
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.classifier = nn.Sequential(
            nn.Linear(512, 256), nn.ReLU(inplace=True), nn.Dropout(0.5),
            nn.Linear(256, 128), nn.ReLU(inplace=True), nn.Dropout(0.4),
            nn.Linear(128, 1), nn.Sigmoid(),
        )

    def forward(self, x):
        x = self.blocks(x)          # (B, 512, L)
        x = self.pool(x).squeeze(2) # (B, 512)
        return self.classifier(x).squeeze(1)

def build_model():
    return CNN1D_VGG(SAMPLES_PER_SEGMENT)

def train_one_epoch(model, loader, optimizer, criterion, cw_tensor):
    model.train()
    total_loss = 0.0
    for xb, yb in loader:
        xb = xb.to(DEVICE, non_blocking=True)
        yb = yb.to(DEVICE, non_blocking=True).float()
        optimizer.zero_grad(set_to_none=True)
        preds = model(xb)
        sample_w = cw_tensor[yb.long()]
        loss = (criterion(preds, yb) * sample_w).mean()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

@torch.no_grad()
def predict_file(model, file_path):
    model.eval()
    segs = []
    try:
        audio, _ = librosa.load(file_path, sr=SAMPLE_RATE)
        hop = HOP_SIZE
        for start in range(0, len(audio), hop):
            seg = audio[start:start + SAMPLES_PER_SEGMENT]
            if len(seg) < SAMPLES_PER_SEGMENT // 2: continue
            if len(seg) < SAMPLES_PER_SEGMENT:
                seg = np.pad(seg, (0, SAMPLES_PER_SEGMENT - len(seg)))
            seg = seg.astype(np.float32)
            seg /= (np.max(np.abs(seg)) + 1e-8)
            segs.append(seg)
    except Exception as e:
        print(f"  Error inferencia {file_path}: {e}"); return None
    if not segs: return None
    probs = []
    for b in range(0, len(segs), BATCH_SIZE):
        batch = np.stack(segs[b:b + BATCH_SIZE])
        bt = torch.from_numpy(batch).unsqueeze(1).to(DEVICE)
        out = model(bt).cpu().numpy()
        probs.extend(out.tolist())
    return float(np.mean(probs))

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
global_y_true, global_y_pred, global_y_prob = [], [], []

print(f"\nINICIANDO CROSS-VALIDATION {MODEL_NAME} | {SEGMENT_DURATION}s | {N_FOLDS} FOLDS\n")

for fold_idx, (tr_idx, te_idx) in enumerate(skf.split(all_files, all_labels)):
    print(f"\n{'─'*55}\n  FOLD {fold_idx+1}/{N_FOLDS}\n{'─'*55}")

    Xtr_f, Xte_f = all_files[tr_idx], all_files[te_idx]
    ytr_f, yte_f = all_labels[tr_idx], all_labels[te_idx]
    Xtr_f, ytr_f = oversample_files(Xtr_f, ytr_f)

    train_ds = WaveformDataset(Xtr_f, ytr_f)
    train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=8, pin_memory=True, persistent_workers=True)
    print(f"  Segmentos train: {len(train_ds)}")

    weights  = class_weight.compute_class_weight('balanced',
                   classes=np.unique(ytr_f), y=ytr_f)
    cw_tensor = torch.tensor(weights, dtype=torch.float32).to(DEVICE)

    model     = build_model().to(DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=1e-4)
    criterion = nn.BCELoss(reduction='none')
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=3, min_lr=1e-7)

    if fold_idx == 0:
        n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
        print(f"  Parámetros entrenables: {n_params:,}\n")

    best_loss, patience_cnt, PATIENCE, best_state = float('inf'), 0, 5, None
    for epoch in range(EPOCHS):
        loss = train_one_epoch(model, train_dl, optimizer, criterion, cw_tensor)
        scheduler.step(loss)
        if loss < best_loss:
            best_loss = loss; patience_cnt = 0
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            mark = "✓"
        else:
            patience_cnt += 1
            mark = f"({patience_cnt}/{PATIENCE})"
        print(f"  Epoch {epoch+1:02d}/{EPOCHS} — loss: {loss:.4f}  best: {best_loss:.4f}  {mark}")
        if patience_cnt >= PATIENCE:
            print("  ► Early stopping activado."); break

    model.load_state_dict(best_state)

    for i, fp in enumerate(Xte_f):
        avg_prob = predict_file(model, fp)
        if avg_prob is None: continue
        global_y_true.append(int(yte_f[i]))
        global_y_pred.append(1 if avg_prob > 0.33 else 0)
        global_y_prob.append(avg_prob)

    del model, train_ds, train_dl, best_state, cw_tensor
    gc.collect(); torch.cuda.empty_cache()

print("\n" + "="*60)
print(f"RESULTADOS {MODEL_NAME} | {SEGMENT_DURATION}s  ({len(global_y_true)} pacientes)")
print("="*60)

cm = confusion_matrix(global_y_true, global_y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Sin Ansiedad', 'Con Ansiedad'],
            yticklabels=['Sin Ansiedad', 'Con Ansiedad'])
plt.title(f'Confusión {MODEL_NAME} {SEGMENT_DURATION}s ({N_FOLDS}-Fold CV)')
plt.ylabel('Real'); plt.xlabel('Predicción')
plt.tight_layout()
plt.savefig(f'confusion_{MODEL_NAME.replace(" ","_")}_{SEGMENT_DURATION}s.png', dpi=150)
plt.show()

acc      = accuracy_score(global_y_true, global_y_pred)
prec     = precision_score(global_y_true, global_y_pred, zero_division=0)
rec      = recall_score(global_y_true, global_y_pred, zero_division=0)
f1       = f1_score(global_y_true, global_y_pred, zero_division=0)
f1_macro = f1_score(global_y_true, global_y_pred, average='macro', zero_division=0)
print(f"\nAccuracy :  {acc:.4f}")
print(f"Precision:  {prec:.4f}")
print(f"Recall   :  {rec:.4f}")
print(f"F1-Ans   :  {f1:.4f}  (clase positiva)")
print(f"F1-Macro :  {f1_macro:.4f}")
print()
print(classification_report(global_y_true, global_y_pred,
                             target_names=['Sin Ansiedad', 'Con Ansiedad']))

fpr, tpr, _ = roc_curve(global_y_true, global_y_prob)
roc_auc = auc(fpr, tpr)
plt.figure(figsize=(7, 6))
plt.plot(fpr, tpr, lw=2, label=f'AUC = {roc_auc:.2f}')
plt.plot([0,1],[0,1],'navy',lw=2,linestyle='--')
plt.xlabel('Falsos Positivos'); plt.ylabel('Verdaderos Positivos')
plt.title(f'ROC — {MODEL_NAME} {SEGMENT_DURATION}s')
plt.legend(loc='lower right'); plt.tight_layout()
plt.savefig(f'roc_{MODEL_NAME.replace(" ","_")}_{SEGMENT_DURATION}s.png', dpi=150)
plt.show()


# RNN — GRU Raw Waveform

## 10 s | Sin overlap

In [ ]:
import os
import gc
import numpy as np
import librosa
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (confusion_matrix, classification_report,
                             accuracy_score, precision_score, recall_score,
                             f1_score, roc_curve, auc)
from sklearn.utils import class_weight

# ══════════════════════════════════════════════════════════

BASE_PATH_0 = "../Clasificación Final/Ansiedad/wav/0"
BASE_PATH_1 = "../Clasificación Final/Ansiedad/wav/1"

SAMPLE_RATE         = 16000
SEGMENT_DURATION    = 10
SAMPLES_PER_SEGMENT = int(SEGMENT_DURATION * SAMPLE_RATE)  # auto-calculado
# Sin overlap para 10 s
N_FOLDS             = 5
BATCH_SIZE          = 128
EPOCHS              = 25

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Usando dispositivo: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
    torch.backends.cudnn.benchmark = True

def get_file_lists():
    files_0 = [os.path.join(BASE_PATH_0, f)
               for f in sorted(os.listdir(BASE_PATH_0)) if f.endswith('.wav')]
    files_1 = [os.path.join(BASE_PATH_1, f)
               for f in sorted(os.listdir(BASE_PATH_1)) if f.endswith('.wav')]
    files  = np.array(files_0 + files_1)
    labels = np.array([0]*len(files_0) + [1]*len(files_1), dtype=np.int64)
    return files, labels

all_files, all_labels = get_file_lists()
print(f"\nTotal archivos : {len(all_files)}")
print(f" - Sin Ansiedad (0): {np.sum(all_labels == 0)}")
print(f" - Con Ansiedad (1): {np.sum(all_labels == 1)}")
print(f"Segment: {SEGMENT_DURATION}s | Samples/seg: {SAMPLES_PER_SEGMENT}")

class WaveformDataset(Dataset):
    """Lazy loading — sin overlap para 10 s."""
    def __init__(self, file_list, label_list):
        self.samples = []
        for fp, lbl in zip(file_list, label_list):
            try:
                total = librosa.get_duration(path=fp)
                start = 0.0
                while start < total:
                    if total - start < SEGMENT_DURATION / 2:
                        break
                    self.samples.append((fp, start, int(lbl)))
                    start += SEGMENT_DURATION
            except Exception as e:
                print(f"[AVISO] {fp}: {e}")

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        fp, offset, label = self.samples[idx]
        audio, _ = librosa.load(fp, sr=SAMPLE_RATE, offset=offset,
                                duration=SEGMENT_DURATION)
        if len(audio) < SAMPLES_PER_SEGMENT:
            audio = np.pad(audio, (0, SAMPLES_PER_SEGMENT - len(audio)))
        audio = audio[:SAMPLES_PER_SEGMENT].astype(np.float32)
        audio /= (np.max(np.abs(audio)) + 1e-8)
        return torch.from_numpy(audio).unsqueeze(0), torch.tensor(label, dtype=torch.long)

def oversample_files(files, labels):
    f1 = files[labels == 1]; l1 = labels[labels == 1]
    diff = np.sum(labels == 0) - np.sum(labels == 1)
    if diff > 0:
        idx    = np.random.choice(len(f1), size=diff, replace=True)
        files  = np.concatenate([files,  f1[idx]])
        labels = np.concatenate([labels, l1[idx]])
    return files, labels

MODEL_NAME = "RNN-GRU"

class RNN_GRU(nn.Module):
    """CNN Subsampler → GRU Stack (CuDNN-compatible).
    CNN reduce 220k→~430 timesteps antes de la GRU para mayor velocidad."""

    def __init__(self, input_length):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv1d(1, 64, 160, stride=80, padding=80),
            nn.ReLU(inplace=True), nn.BatchNorm1d(64),
            nn.MaxPool1d(2), nn.Dropout(0.2),
            nn.Conv1d(64, 128, 9, padding=4),
            nn.ReLU(inplace=True), nn.BatchNorm1d(128), nn.Dropout(0.2),
            nn.Conv1d(128, 128, 5, padding=2),
            nn.ReLU(inplace=True), nn.BatchNorm1d(128), nn.Dropout(0.2),
        )
        self.gru1 = nn.GRU(128, 128, batch_first=True)
        self.drop1 = nn.Dropout(0.3)
        self.gru2 = nn.GRU(128, 64, batch_first=True)
        self.drop2 = nn.Dropout(0.3)
        self.gru3 = nn.GRU(64, 32, batch_first=True)
        self.drop3 = nn.Dropout(0.3)
        self.classifier = nn.Sequential(
            nn.Linear(32, 64), nn.ReLU(inplace=True), nn.Dropout(0.4),
            nn.Linear(64, 32), nn.ReLU(inplace=True), nn.Dropout(0.3),
            nn.Linear(32, 1), nn.Sigmoid(),
        )

    def forward(self, x):
        x = self.cnn(x).permute(0, 2, 1)   # (B, T, 128)
        x, _ = self.gru1(x); x = self.drop1(x)
        x, _ = self.gru2(x); x = self.drop2(x)
        x, _ = self.gru3(x); x = self.drop3(x)
        x = x[:, -1, :]                     # último timestep
        return self.classifier(x).squeeze(1)

def build_model():
    return RNN_GRU(SAMPLES_PER_SEGMENT)

def train_one_epoch(model, loader, optimizer, criterion, cw_tensor):
    model.train()
    total_loss = 0.0
    for xb, yb in loader:
        xb = xb.to(DEVICE, non_blocking=True)
        yb = yb.to(DEVICE, non_blocking=True).float()
        optimizer.zero_grad(set_to_none=True)
        preds = model(xb)
        sample_w = cw_tensor[yb.long()]
        loss = (criterion(preds, yb) * sample_w).mean()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

@torch.no_grad()
def predict_file(model, file_path):
    model.eval()
    segs = []
    try:
        audio, _ = librosa.load(file_path, sr=SAMPLE_RATE)
        hop = SAMPLES_PER_SEGMENT
        for start in range(0, len(audio), hop):
            seg = audio[start:start + SAMPLES_PER_SEGMENT]
            if len(seg) < SAMPLES_PER_SEGMENT // 2: continue
            if len(seg) < SAMPLES_PER_SEGMENT:
                seg = np.pad(seg, (0, SAMPLES_PER_SEGMENT - len(seg)))
            seg = seg.astype(np.float32)
            seg /= (np.max(np.abs(seg)) + 1e-8)
            segs.append(seg)
    except Exception as e:
        print(f"  Error inferencia {file_path}: {e}"); return None
    if not segs: return None
    probs = []
    for b in range(0, len(segs), BATCH_SIZE):
        batch = np.stack(segs[b:b + BATCH_SIZE])
        bt = torch.from_numpy(batch).unsqueeze(1).to(DEVICE)
        out = model(bt).cpu().numpy()
        probs.extend(out.tolist())
    return float(np.mean(probs))

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
global_y_true, global_y_pred, global_y_prob = [], [], []

print(f"\nINICIANDO CROSS-VALIDATION {MODEL_NAME} | {SEGMENT_DURATION}s | {N_FOLDS} FOLDS\n")

for fold_idx, (tr_idx, te_idx) in enumerate(skf.split(all_files, all_labels)):
    print(f"\n{'─'*55}\n  FOLD {fold_idx+1}/{N_FOLDS}\n{'─'*55}")

    Xtr_f, Xte_f = all_files[tr_idx], all_files[te_idx]
    ytr_f, yte_f = all_labels[tr_idx], all_labels[te_idx]
    Xtr_f, ytr_f = oversample_files(Xtr_f, ytr_f)

    train_ds = WaveformDataset(Xtr_f, ytr_f)
    train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=8, pin_memory=True, persistent_workers=True)
    print(f"  Segmentos train: {len(train_ds)}")

    weights  = class_weight.compute_class_weight('balanced',
                   classes=np.unique(ytr_f), y=ytr_f)
    cw_tensor = torch.tensor(weights, dtype=torch.float32).to(DEVICE)

    model     = build_model().to(DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=1e-4)
    criterion = nn.BCELoss(reduction='none')
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=3, min_lr=1e-7)

    if fold_idx == 0:
        n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
        print(f"  Parámetros entrenables: {n_params:,}\n")

    best_loss, patience_cnt, PATIENCE, best_state = float('inf'), 0, 5, None
    for epoch in range(EPOCHS):
        loss = train_one_epoch(model, train_dl, optimizer, criterion, cw_tensor)
        scheduler.step(loss)
        if loss < best_loss:
            best_loss = loss; patience_cnt = 0
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            mark = "✓"
        else:
            patience_cnt += 1
            mark = f"({patience_cnt}/{PATIENCE})"
        print(f"  Epoch {epoch+1:02d}/{EPOCHS} — loss: {loss:.4f}  best: {best_loss:.4f}  {mark}")
        if patience_cnt >= PATIENCE:
            print("  ► Early stopping activado."); break

    model.load_state_dict(best_state)

    for i, fp in enumerate(Xte_f):
        avg_prob = predict_file(model, fp)
        if avg_prob is None: continue
        global_y_true.append(int(yte_f[i]))
        global_y_pred.append(1 if avg_prob > 0.33 else 0)
        global_y_prob.append(avg_prob)

    del model, train_ds, train_dl, best_state, cw_tensor
    gc.collect(); torch.cuda.empty_cache()

print("\n" + "="*60)
print(f"RESULTADOS {MODEL_NAME} | {SEGMENT_DURATION}s  ({len(global_y_true)} pacientes)")
print("="*60)

cm = confusion_matrix(global_y_true, global_y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='YlOrBr',
            xticklabels=['Sin Ansiedad', 'Con Ansiedad'],
            yticklabels=['Sin Ansiedad', 'Con Ansiedad'])
plt.title(f'Confusión {MODEL_NAME} {SEGMENT_DURATION}s ({N_FOLDS}-Fold CV)')
plt.ylabel('Real'); plt.xlabel('Predicción')
plt.tight_layout()
plt.savefig(f'confusion_{MODEL_NAME.replace(" ","_")}_{SEGMENT_DURATION}s.png', dpi=150)
plt.show()

acc      = accuracy_score(global_y_true, global_y_pred)
prec     = precision_score(global_y_true, global_y_pred, zero_division=0)
rec      = recall_score(global_y_true, global_y_pred, zero_division=0)
f1       = f1_score(global_y_true, global_y_pred, zero_division=0)
f1_macro = f1_score(global_y_true, global_y_pred, average='macro', zero_division=0)
print(f"\nAccuracy :  {acc:.4f}")
print(f"Precision:  {prec:.4f}")
print(f"Recall   :  {rec:.4f}")
print(f"F1-Ans   :  {f1:.4f}  (clase positiva)")
print(f"F1-Macro :  {f1_macro:.4f}")
print()
print(classification_report(global_y_true, global_y_pred,
                             target_names=['Sin Ansiedad', 'Con Ansiedad']))

fpr, tpr, _ = roc_curve(global_y_true, global_y_prob)
roc_auc = auc(fpr, tpr)
plt.figure(figsize=(7, 6))
plt.plot(fpr, tpr, lw=2, label=f'AUC = {roc_auc:.2f}')
plt.plot([0,1],[0,1],'navy',lw=2,linestyle='--')
plt.xlabel('Falsos Positivos'); plt.ylabel('Verdaderos Positivos')
plt.title(f'ROC — {MODEL_NAME} {SEGMENT_DURATION}s')
plt.legend(loc='lower right'); plt.tight_layout()
plt.savefig(f'roc_{MODEL_NAME.replace(" ","_")}_{SEGMENT_DURATION}s.png', dpi=150)
plt.show()


## 2 s | Overlap 50%

In [ ]:
import os
import gc
import numpy as np
import librosa
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (confusion_matrix, classification_report,
                             accuracy_score, precision_score, recall_score,
                             f1_score, roc_curve, auc)
from sklearn.utils import class_weight

# ══════════════════════════════════════════════════════════

BASE_PATH_0 = "../Clasificación Final/Ansiedad/wav/0"
BASE_PATH_1 = "../Clasificación Final/Ansiedad/wav/1"

SAMPLE_RATE         = 16000
SEGMENT_DURATION    = 2
SAMPLES_PER_SEGMENT = int(SEGMENT_DURATION * SAMPLE_RATE)  # auto-calculado
HOP_SIZE            = SAMPLES_PER_SEGMENT // 2  # 50% overlap → 22050
N_FOLDS             = 5
BATCH_SIZE          = 128
EPOCHS              = 25

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Usando dispositivo: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
    torch.backends.cudnn.benchmark = True

def get_file_lists():
    files_0 = [os.path.join(BASE_PATH_0, f)
               for f in sorted(os.listdir(BASE_PATH_0)) if f.endswith('.wav')]
    files_1 = [os.path.join(BASE_PATH_1, f)
               for f in sorted(os.listdir(BASE_PATH_1)) if f.endswith('.wav')]
    files  = np.array(files_0 + files_1)
    labels = np.array([0]*len(files_0) + [1]*len(files_1), dtype=np.int64)
    return files, labels

all_files, all_labels = get_file_lists()
print(f"\nTotal archivos : {len(all_files)}")
print(f" - Sin Ansiedad (0): {np.sum(all_labels == 0)}")
print(f" - Con Ansiedad (1): {np.sum(all_labels == 1)}")
print(f"Segment: {SEGMENT_DURATION}s | Samples/seg: {SAMPLES_PER_SEGMENT}")

class WaveformDataset(Dataset):
    """Lazy loading — overlap 50%%."""
    def __init__(self, file_list, label_list):
        self.samples = []
        for fp, lbl in zip(file_list, label_list):
            try:
                total = librosa.get_duration(path=fp)
                n_segs = max(1, int((total * SAMPLE_RATE - SAMPLES_PER_SEGMENT) / HOP_SIZE) + 1)
                for i in range(n_segs):
                    offset = i * HOP_SIZE / SAMPLE_RATE
                    self.samples.append((fp, offset, int(lbl)))
            except Exception as e:
                print(f"[AVISO] {fp}: {e}")

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        fp, offset, label = self.samples[idx]
        audio, _ = librosa.load(fp, sr=SAMPLE_RATE, offset=offset,
                                duration=SEGMENT_DURATION)
        if len(audio) < SAMPLES_PER_SEGMENT:
            audio = np.pad(audio, (0, SAMPLES_PER_SEGMENT - len(audio)))
        audio = audio[:SAMPLES_PER_SEGMENT].astype(np.float32)
        audio /= (np.max(np.abs(audio)) + 1e-8)
        return torch.from_numpy(audio).unsqueeze(0), torch.tensor(label, dtype=torch.long)

def oversample_files(files, labels):
    f1 = files[labels == 1]; l1 = labels[labels == 1]
    diff = np.sum(labels == 0) - np.sum(labels == 1)
    if diff > 0:
        idx    = np.random.choice(len(f1), size=diff, replace=True)
        files  = np.concatenate([files,  f1[idx]])
        labels = np.concatenate([labels, l1[idx]])
    return files, labels

MODEL_NAME = "RNN-GRU"

class RNN_GRU(nn.Module):
    """CNN Subsampler → GRU Stack (CuDNN-compatible).
    CNN reduce 220k→~430 timesteps antes de la GRU para mayor velocidad."""

    def __init__(self, input_length):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv1d(1, 64, 160, stride=80, padding=80),
            nn.ReLU(inplace=True), nn.BatchNorm1d(64),
            nn.MaxPool1d(2), nn.Dropout(0.2),
            nn.Conv1d(64, 128, 9, padding=4),
            nn.ReLU(inplace=True), nn.BatchNorm1d(128), nn.Dropout(0.2),
            nn.Conv1d(128, 128, 5, padding=2),
            nn.ReLU(inplace=True), nn.BatchNorm1d(128), nn.Dropout(0.2),
        )
        self.gru1 = nn.GRU(128, 128, batch_first=True)
        self.drop1 = nn.Dropout(0.3)
        self.gru2 = nn.GRU(128, 64, batch_first=True)
        self.drop2 = nn.Dropout(0.3)
        self.gru3 = nn.GRU(64, 32, batch_first=True)
        self.drop3 = nn.Dropout(0.3)
        self.classifier = nn.Sequential(
            nn.Linear(32, 64), nn.ReLU(inplace=True), nn.Dropout(0.4),
            nn.Linear(64, 32), nn.ReLU(inplace=True), nn.Dropout(0.3),
            nn.Linear(32, 1), nn.Sigmoid(),
        )

    def forward(self, x):
        x = self.cnn(x).permute(0, 2, 1)   # (B, T, 128)
        x, _ = self.gru1(x); x = self.drop1(x)
        x, _ = self.gru2(x); x = self.drop2(x)
        x, _ = self.gru3(x); x = self.drop3(x)
        x = x[:, -1, :]                     # último timestep
        return self.classifier(x).squeeze(1)

def build_model():
    return RNN_GRU(SAMPLES_PER_SEGMENT)

def train_one_epoch(model, loader, optimizer, criterion, cw_tensor):
    model.train()
    total_loss = 0.0
    for xb, yb in loader:
        xb = xb.to(DEVICE, non_blocking=True)
        yb = yb.to(DEVICE, non_blocking=True).float()
        optimizer.zero_grad(set_to_none=True)
        preds = model(xb)
        sample_w = cw_tensor[yb.long()]
        loss = (criterion(preds, yb) * sample_w).mean()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

@torch.no_grad()
def predict_file(model, file_path):
    model.eval()
    segs = []
    try:
        audio, _ = librosa.load(file_path, sr=SAMPLE_RATE)
        hop = HOP_SIZE
        for start in range(0, len(audio), hop):
            seg = audio[start:start + SAMPLES_PER_SEGMENT]
            if len(seg) < SAMPLES_PER_SEGMENT // 2: continue
            if len(seg) < SAMPLES_PER_SEGMENT:
                seg = np.pad(seg, (0, SAMPLES_PER_SEGMENT - len(seg)))
            seg = seg.astype(np.float32)
            seg /= (np.max(np.abs(seg)) + 1e-8)
            segs.append(seg)
    except Exception as e:
        print(f"  Error inferencia {file_path}: {e}"); return None
    if not segs: return None
    probs = []
    for b in range(0, len(segs), BATCH_SIZE):
        batch = np.stack(segs[b:b + BATCH_SIZE])
        bt = torch.from_numpy(batch).unsqueeze(1).to(DEVICE)
        out = model(bt).cpu().numpy()
        probs.extend(out.tolist())
    return float(np.mean(probs))

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
global_y_true, global_y_pred, global_y_prob = [], [], []

print(f"\nINICIANDO CROSS-VALIDATION {MODEL_NAME} | {SEGMENT_DURATION}s | {N_FOLDS} FOLDS\n")

for fold_idx, (tr_idx, te_idx) in enumerate(skf.split(all_files, all_labels)):
    print(f"\n{'─'*55}\n  FOLD {fold_idx+1}/{N_FOLDS}\n{'─'*55}")

    Xtr_f, Xte_f = all_files[tr_idx], all_files[te_idx]
    ytr_f, yte_f = all_labels[tr_idx], all_labels[te_idx]
    Xtr_f, ytr_f = oversample_files(Xtr_f, ytr_f)

    train_ds = WaveformDataset(Xtr_f, ytr_f)
    train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=8, pin_memory=True, persistent_workers=True)
    print(f"  Segmentos train: {len(train_ds)}")

    weights  = class_weight.compute_class_weight('balanced',
                   classes=np.unique(ytr_f), y=ytr_f)
    cw_tensor = torch.tensor(weights, dtype=torch.float32).to(DEVICE)

    model     = build_model().to(DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=1e-4)
    criterion = nn.BCELoss(reduction='none')
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=3, min_lr=1e-7)

    if fold_idx == 0:
        n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
        print(f"  Parámetros entrenables: {n_params:,}\n")

    best_loss, patience_cnt, PATIENCE, best_state = float('inf'), 0, 5, None
    for epoch in range(EPOCHS):
        loss = train_one_epoch(model, train_dl, optimizer, criterion, cw_tensor)
        scheduler.step(loss)
        if loss < best_loss:
            best_loss = loss; patience_cnt = 0
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            mark = "✓"
        else:
            patience_cnt += 1
            mark = f"({patience_cnt}/{PATIENCE})"
        print(f"  Epoch {epoch+1:02d}/{EPOCHS} — loss: {loss:.4f}  best: {best_loss:.4f}  {mark}")
        if patience_cnt >= PATIENCE:
            print("  ► Early stopping activado."); break

    model.load_state_dict(best_state)

    for i, fp in enumerate(Xte_f):
        avg_prob = predict_file(model, fp)
        if avg_prob is None: continue
        global_y_true.append(int(yte_f[i]))
        global_y_pred.append(1 if avg_prob > 0.33 else 0)
        global_y_prob.append(avg_prob)

    del model, train_ds, train_dl, best_state, cw_tensor
    gc.collect(); torch.cuda.empty_cache()

print("\n" + "="*60)
print(f"RESULTADOS {MODEL_NAME} | {SEGMENT_DURATION}s  ({len(global_y_true)} pacientes)")
print("="*60)

cm = confusion_matrix(global_y_true, global_y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='YlOrBr',
            xticklabels=['Sin Ansiedad', 'Con Ansiedad'],
            yticklabels=['Sin Ansiedad', 'Con Ansiedad'])
plt.title(f'Confusión {MODEL_NAME} {SEGMENT_DURATION}s ({N_FOLDS}-Fold CV)')
plt.ylabel('Real'); plt.xlabel('Predicción')
plt.tight_layout()
plt.savefig(f'confusion_{MODEL_NAME.replace(" ","_")}_{SEGMENT_DURATION}s.png', dpi=150)
plt.show()

acc      = accuracy_score(global_y_true, global_y_pred)
prec     = precision_score(global_y_true, global_y_pred, zero_division=0)
rec      = recall_score(global_y_true, global_y_pred, zero_division=0)
f1       = f1_score(global_y_true, global_y_pred, zero_division=0)
f1_macro = f1_score(global_y_true, global_y_pred, average='macro', zero_division=0)
print(f"\nAccuracy :  {acc:.4f}")
print(f"Precision:  {prec:.4f}")
print(f"Recall   :  {rec:.4f}")
print(f"F1-Ans   :  {f1:.4f}  (clase positiva)")
print(f"F1-Macro :  {f1_macro:.4f}")
print()
print(classification_report(global_y_true, global_y_pred,
                             target_names=['Sin Ansiedad', 'Con Ansiedad']))

fpr, tpr, _ = roc_curve(global_y_true, global_y_prob)
roc_auc = auc(fpr, tpr)
plt.figure(figsize=(7, 6))
plt.plot(fpr, tpr, lw=2, label=f'AUC = {roc_auc:.2f}')
plt.plot([0,1],[0,1],'navy',lw=2,linestyle='--')
plt.xlabel('Falsos Positivos'); plt.ylabel('Verdaderos Positivos')
plt.title(f'ROC — {MODEL_NAME} {SEGMENT_DURATION}s')
plt.legend(loc='lower right'); plt.tight_layout()
plt.savefig(f'roc_{MODEL_NAME.replace(" ","_")}_{SEGMENT_DURATION}s.png', dpi=150)
plt.show()


## 3 s | Overlap 50%

In [ ]:
import os
import gc
import numpy as np
import librosa
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (confusion_matrix, classification_report,
                             accuracy_score, precision_score, recall_score,
                             f1_score, roc_curve, auc)
from sklearn.utils import class_weight

# ══════════════════════════════════════════════════════════

BASE_PATH_0 = "../Clasificación Final/Ansiedad/wav/0"
BASE_PATH_1 = "../Clasificación Final/Ansiedad/wav/1"

SAMPLE_RATE         = 16000
SEGMENT_DURATION    = 3
SAMPLES_PER_SEGMENT = int(SEGMENT_DURATION * SAMPLE_RATE)  # auto-calculado
HOP_SIZE            = SAMPLES_PER_SEGMENT // 2  # 50% overlap → 33075
N_FOLDS             = 5
BATCH_SIZE          = 128
EPOCHS              = 25

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Usando dispositivo: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
    torch.backends.cudnn.benchmark = True

def get_file_lists():
    files_0 = [os.path.join(BASE_PATH_0, f)
               for f in sorted(os.listdir(BASE_PATH_0)) if f.endswith('.wav')]
    files_1 = [os.path.join(BASE_PATH_1, f)
               for f in sorted(os.listdir(BASE_PATH_1)) if f.endswith('.wav')]
    files  = np.array(files_0 + files_1)
    labels = np.array([0]*len(files_0) + [1]*len(files_1), dtype=np.int64)
    return files, labels

all_files, all_labels = get_file_lists()
print(f"\nTotal archivos : {len(all_files)}")
print(f" - Sin Ansiedad (0): {np.sum(all_labels == 0)}")
print(f" - Con Ansiedad (1): {np.sum(all_labels == 1)}")
print(f"Segment: {SEGMENT_DURATION}s | Samples/seg: {SAMPLES_PER_SEGMENT}")

class WaveformDataset(Dataset):
    """Lazy loading — overlap 50%%."""
    def __init__(self, file_list, label_list):
        self.samples = []
        for fp, lbl in zip(file_list, label_list):
            try:
                total = librosa.get_duration(path=fp)
                n_segs = max(1, int((total * SAMPLE_RATE - SAMPLES_PER_SEGMENT) / HOP_SIZE) + 1)
                for i in range(n_segs):
                    offset = i * HOP_SIZE / SAMPLE_RATE
                    self.samples.append((fp, offset, int(lbl)))
            except Exception as e:
                print(f"[AVISO] {fp}: {e}")

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        fp, offset, label = self.samples[idx]
        audio, _ = librosa.load(fp, sr=SAMPLE_RATE, offset=offset,
                                duration=SEGMENT_DURATION)
        if len(audio) < SAMPLES_PER_SEGMENT:
            audio = np.pad(audio, (0, SAMPLES_PER_SEGMENT - len(audio)))
        audio = audio[:SAMPLES_PER_SEGMENT].astype(np.float32)
        audio /= (np.max(np.abs(audio)) + 1e-8)
        return torch.from_numpy(audio).unsqueeze(0), torch.tensor(label, dtype=torch.long)

def oversample_files(files, labels):
    f1 = files[labels == 1]; l1 = labels[labels == 1]
    diff = np.sum(labels == 0) - np.sum(labels == 1)
    if diff > 0:
        idx    = np.random.choice(len(f1), size=diff, replace=True)
        files  = np.concatenate([files,  f1[idx]])
        labels = np.concatenate([labels, l1[idx]])
    return files, labels

MODEL_NAME = "RNN-GRU"

class RNN_GRU(nn.Module):
    """CNN Subsampler → GRU Stack (CuDNN-compatible).
    CNN reduce 220k→~430 timesteps antes de la GRU para mayor velocidad."""

    def __init__(self, input_length):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv1d(1, 64, 160, stride=80, padding=80),
            nn.ReLU(inplace=True), nn.BatchNorm1d(64),
            nn.MaxPool1d(2), nn.Dropout(0.2),
            nn.Conv1d(64, 128, 9, padding=4),
            nn.ReLU(inplace=True), nn.BatchNorm1d(128), nn.Dropout(0.2),
            nn.Conv1d(128, 128, 5, padding=2),
            nn.ReLU(inplace=True), nn.BatchNorm1d(128), nn.Dropout(0.2),
        )
        self.gru1 = nn.GRU(128, 128, batch_first=True)
        self.drop1 = nn.Dropout(0.3)
        self.gru2 = nn.GRU(128, 64, batch_first=True)
        self.drop2 = nn.Dropout(0.3)
        self.gru3 = nn.GRU(64, 32, batch_first=True)
        self.drop3 = nn.Dropout(0.3)
        self.classifier = nn.Sequential(
            nn.Linear(32, 64), nn.ReLU(inplace=True), nn.Dropout(0.4),
            nn.Linear(64, 32), nn.ReLU(inplace=True), nn.Dropout(0.3),
            nn.Linear(32, 1), nn.Sigmoid(),
        )

    def forward(self, x):
        x = self.cnn(x).permute(0, 2, 1)   # (B, T, 128)
        x, _ = self.gru1(x); x = self.drop1(x)
        x, _ = self.gru2(x); x = self.drop2(x)
        x, _ = self.gru3(x); x = self.drop3(x)
        x = x[:, -1, :]                     # último timestep
        return self.classifier(x).squeeze(1)

def build_model():
    return RNN_GRU(SAMPLES_PER_SEGMENT)

def train_one_epoch(model, loader, optimizer, criterion, cw_tensor):
    model.train()
    total_loss = 0.0
    for xb, yb in loader:
        xb = xb.to(DEVICE, non_blocking=True)
        yb = yb.to(DEVICE, non_blocking=True).float()
        optimizer.zero_grad(set_to_none=True)
        preds = model(xb)
        sample_w = cw_tensor[yb.long()]
        loss = (criterion(preds, yb) * sample_w).mean()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

@torch.no_grad()
def predict_file(model, file_path):
    model.eval()
    segs = []
    try:
        audio, _ = librosa.load(file_path, sr=SAMPLE_RATE)
        hop = HOP_SIZE
        for start in range(0, len(audio), hop):
            seg = audio[start:start + SAMPLES_PER_SEGMENT]
            if len(seg) < SAMPLES_PER_SEGMENT // 2: continue
            if len(seg) < SAMPLES_PER_SEGMENT:
                seg = np.pad(seg, (0, SAMPLES_PER_SEGMENT - len(seg)))
            seg = seg.astype(np.float32)
            seg /= (np.max(np.abs(seg)) + 1e-8)
            segs.append(seg)
    except Exception as e:
        print(f"  Error inferencia {file_path}: {e}"); return None
    if not segs: return None
    probs = []
    for b in range(0, len(segs), BATCH_SIZE):
        batch = np.stack(segs[b:b + BATCH_SIZE])
        bt = torch.from_numpy(batch).unsqueeze(1).to(DEVICE)
        out = model(bt).cpu().numpy()
        probs.extend(out.tolist())
    return float(np.mean(probs))

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
global_y_true, global_y_pred, global_y_prob = [], [], []

print(f"\nINICIANDO CROSS-VALIDATION {MODEL_NAME} | {SEGMENT_DURATION}s | {N_FOLDS} FOLDS\n")

for fold_idx, (tr_idx, te_idx) in enumerate(skf.split(all_files, all_labels)):
    print(f"\n{'─'*55}\n  FOLD {fold_idx+1}/{N_FOLDS}\n{'─'*55}")

    Xtr_f, Xte_f = all_files[tr_idx], all_files[te_idx]
    ytr_f, yte_f = all_labels[tr_idx], all_labels[te_idx]
    Xtr_f, ytr_f = oversample_files(Xtr_f, ytr_f)

    train_ds = WaveformDataset(Xtr_f, ytr_f)
    train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=8, pin_memory=True, persistent_workers=True)
    print(f"  Segmentos train: {len(train_ds)}")

    weights  = class_weight.compute_class_weight('balanced',
                   classes=np.unique(ytr_f), y=ytr_f)
    cw_tensor = torch.tensor(weights, dtype=torch.float32).to(DEVICE)

    model     = build_model().to(DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=1e-4)
    criterion = nn.BCELoss(reduction='none')
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=3, min_lr=1e-7)

    if fold_idx == 0:
        n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
        print(f"  Parámetros entrenables: {n_params:,}\n")

    best_loss, patience_cnt, PATIENCE, best_state = float('inf'), 0, 5, None
    for epoch in range(EPOCHS):
        loss = train_one_epoch(model, train_dl, optimizer, criterion, cw_tensor)
        scheduler.step(loss)
        if loss < best_loss:
            best_loss = loss; patience_cnt = 0
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            mark = "✓"
        else:
            patience_cnt += 1
            mark = f"({patience_cnt}/{PATIENCE})"
        print(f"  Epoch {epoch+1:02d}/{EPOCHS} — loss: {loss:.4f}  best: {best_loss:.4f}  {mark}")
        if patience_cnt >= PATIENCE:
            print("  ► Early stopping activado."); break

    model.load_state_dict(best_state)

    for i, fp in enumerate(Xte_f):
        avg_prob = predict_file(model, fp)
        if avg_prob is None: continue
        global_y_true.append(int(yte_f[i]))
        global_y_pred.append(1 if avg_prob > 0.33 else 0)
        global_y_prob.append(avg_prob)

    del model, train_ds, train_dl, best_state, cw_tensor
    gc.collect(); torch.cuda.empty_cache()

print("\n" + "="*60)
print(f"RESULTADOS {MODEL_NAME} | {SEGMENT_DURATION}s  ({len(global_y_true)} pacientes)")
print("="*60)

cm = confusion_matrix(global_y_true, global_y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='YlOrBr',
            xticklabels=['Sin Ansiedad', 'Con Ansiedad'],
            yticklabels=['Sin Ansiedad', 'Con Ansiedad'])
plt.title(f'Confusión {MODEL_NAME} {SEGMENT_DURATION}s ({N_FOLDS}-Fold CV)')
plt.ylabel('Real'); plt.xlabel('Predicción')
plt.tight_layout()
plt.savefig(f'confusion_{MODEL_NAME.replace(" ","_")}_{SEGMENT_DURATION}s.png', dpi=150)
plt.show()

acc      = accuracy_score(global_y_true, global_y_pred)
prec     = precision_score(global_y_true, global_y_pred, zero_division=0)
rec      = recall_score(global_y_true, global_y_pred, zero_division=0)
f1       = f1_score(global_y_true, global_y_pred, zero_division=0)
f1_macro = f1_score(global_y_true, global_y_pred, average='macro', zero_division=0)
print(f"\nAccuracy :  {acc:.4f}")
print(f"Precision:  {prec:.4f}")
print(f"Recall   :  {rec:.4f}")
print(f"F1-Ans   :  {f1:.4f}  (clase positiva)")
print(f"F1-Macro :  {f1_macro:.4f}")
print()
print(classification_report(global_y_true, global_y_pred,
                             target_names=['Sin Ansiedad', 'Con Ansiedad']))

fpr, tpr, _ = roc_curve(global_y_true, global_y_prob)
roc_auc = auc(fpr, tpr)
plt.figure(figsize=(7, 6))
plt.plot(fpr, tpr, lw=2, label=f'AUC = {roc_auc:.2f}')
plt.plot([0,1],[0,1],'navy',lw=2,linestyle='--')
plt.xlabel('Falsos Positivos'); plt.ylabel('Verdaderos Positivos')
plt.title(f'ROC — {MODEL_NAME} {SEGMENT_DURATION}s')
plt.legend(loc='lower right'); plt.tight_layout()
plt.savefig(f'roc_{MODEL_NAME.replace(" ","_")}_{SEGMENT_DURATION}s.png', dpi=150)
plt.show()


## 4 s | Overlap 50%

In [ ]:
import os
import gc
import numpy as np
import librosa
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (confusion_matrix, classification_report,
                             accuracy_score, precision_score, recall_score,
                             f1_score, roc_curve, auc)
from sklearn.utils import class_weight

# ══════════════════════════════════════════════════════════

BASE_PATH_0 = "../Clasificación Final/Ansiedad/wav/0"
BASE_PATH_1 = "../Clasificación Final/Ansiedad/wav/1"

SAMPLE_RATE         = 16000
SEGMENT_DURATION    = 4
SAMPLES_PER_SEGMENT = int(SEGMENT_DURATION * SAMPLE_RATE)  # auto-calculado
HOP_SIZE            = SAMPLES_PER_SEGMENT // 2  # 50% overlap → 44100
N_FOLDS             = 5
BATCH_SIZE          = 128
EPOCHS              = 25

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Usando dispositivo: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
    torch.backends.cudnn.benchmark = True

def get_file_lists():
    files_0 = [os.path.join(BASE_PATH_0, f)
               for f in sorted(os.listdir(BASE_PATH_0)) if f.endswith('.wav')]
    files_1 = [os.path.join(BASE_PATH_1, f)
               for f in sorted(os.listdir(BASE_PATH_1)) if f.endswith('.wav')]
    files  = np.array(files_0 + files_1)
    labels = np.array([0]*len(files_0) + [1]*len(files_1), dtype=np.int64)
    return files, labels

all_files, all_labels = get_file_lists()
print(f"\nTotal archivos : {len(all_files)}")
print(f" - Sin Ansiedad (0): {np.sum(all_labels == 0)}")
print(f" - Con Ansiedad (1): {np.sum(all_labels == 1)}")
print(f"Segment: {SEGMENT_DURATION}s | Samples/seg: {SAMPLES_PER_SEGMENT}")

class WaveformDataset(Dataset):
    """Lazy loading — overlap 50%%."""
    def __init__(self, file_list, label_list):
        self.samples = []
        for fp, lbl in zip(file_list, label_list):
            try:
                total = librosa.get_duration(path=fp)
                n_segs = max(1, int((total * SAMPLE_RATE - SAMPLES_PER_SEGMENT) / HOP_SIZE) + 1)
                for i in range(n_segs):
                    offset = i * HOP_SIZE / SAMPLE_RATE
                    self.samples.append((fp, offset, int(lbl)))
            except Exception as e:
                print(f"[AVISO] {fp}: {e}")

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        fp, offset, label = self.samples[idx]
        audio, _ = librosa.load(fp, sr=SAMPLE_RATE, offset=offset,
                                duration=SEGMENT_DURATION)
        if len(audio) < SAMPLES_PER_SEGMENT:
            audio = np.pad(audio, (0, SAMPLES_PER_SEGMENT - len(audio)))
        audio = audio[:SAMPLES_PER_SEGMENT].astype(np.float32)
        audio /= (np.max(np.abs(audio)) + 1e-8)
        return torch.from_numpy(audio).unsqueeze(0), torch.tensor(label, dtype=torch.long)

def oversample_files(files, labels):
    f1 = files[labels == 1]; l1 = labels[labels == 1]
    diff = np.sum(labels == 0) - np.sum(labels == 1)
    if diff > 0:
        idx    = np.random.choice(len(f1), size=diff, replace=True)
        files  = np.concatenate([files,  f1[idx]])
        labels = np.concatenate([labels, l1[idx]])
    return files, labels

MODEL_NAME = "RNN-GRU"

class RNN_GRU(nn.Module):
    """CNN Subsampler → GRU Stack (CuDNN-compatible).
    CNN reduce 220k→~430 timesteps antes de la GRU para mayor velocidad."""

    def __init__(self, input_length):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv1d(1, 64, 160, stride=80, padding=80),
            nn.ReLU(inplace=True), nn.BatchNorm1d(64),
            nn.MaxPool1d(2), nn.Dropout(0.2),
            nn.Conv1d(64, 128, 9, padding=4),
            nn.ReLU(inplace=True), nn.BatchNorm1d(128), nn.Dropout(0.2),
            nn.Conv1d(128, 128, 5, padding=2),
            nn.ReLU(inplace=True), nn.BatchNorm1d(128), nn.Dropout(0.2),
        )
        self.gru1 = nn.GRU(128, 128, batch_first=True)
        self.drop1 = nn.Dropout(0.3)
        self.gru2 = nn.GRU(128, 64, batch_first=True)
        self.drop2 = nn.Dropout(0.3)
        self.gru3 = nn.GRU(64, 32, batch_first=True)
        self.drop3 = nn.Dropout(0.3)
        self.classifier = nn.Sequential(
            nn.Linear(32, 64), nn.ReLU(inplace=True), nn.Dropout(0.4),
            nn.Linear(64, 32), nn.ReLU(inplace=True), nn.Dropout(0.3),
            nn.Linear(32, 1), nn.Sigmoid(),
        )

    def forward(self, x):
        x = self.cnn(x).permute(0, 2, 1)   # (B, T, 128)
        x, _ = self.gru1(x); x = self.drop1(x)
        x, _ = self.gru2(x); x = self.drop2(x)
        x, _ = self.gru3(x); x = self.drop3(x)
        x = x[:, -1, :]                     # último timestep
        return self.classifier(x).squeeze(1)

def build_model():
    return RNN_GRU(SAMPLES_PER_SEGMENT)

def train_one_epoch(model, loader, optimizer, criterion, cw_tensor):
    model.train()
    total_loss = 0.0
    for xb, yb in loader:
        xb = xb.to(DEVICE, non_blocking=True)
        yb = yb.to(DEVICE, non_blocking=True).float()
        optimizer.zero_grad(set_to_none=True)
        preds = model(xb)
        sample_w = cw_tensor[yb.long()]
        loss = (criterion(preds, yb) * sample_w).mean()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

@torch.no_grad()
def predict_file(model, file_path):
    model.eval()
    segs = []
    try:
        audio, _ = librosa.load(file_path, sr=SAMPLE_RATE)
        hop = HOP_SIZE
        for start in range(0, len(audio), hop):
            seg = audio[start:start + SAMPLES_PER_SEGMENT]
            if len(seg) < SAMPLES_PER_SEGMENT // 2: continue
            if len(seg) < SAMPLES_PER_SEGMENT:
                seg = np.pad(seg, (0, SAMPLES_PER_SEGMENT - len(seg)))
            seg = seg.astype(np.float32)
            seg /= (np.max(np.abs(seg)) + 1e-8)
            segs.append(seg)
    except Exception as e:
        print(f"  Error inferencia {file_path}: {e}"); return None
    if not segs: return None
    probs = []
    for b in range(0, len(segs), BATCH_SIZE):
        batch = np.stack(segs[b:b + BATCH_SIZE])
        bt = torch.from_numpy(batch).unsqueeze(1).to(DEVICE)
        out = model(bt).cpu().numpy()
        probs.extend(out.tolist())
    return float(np.mean(probs))

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
global_y_true, global_y_pred, global_y_prob = [], [], []

print(f"\nINICIANDO CROSS-VALIDATION {MODEL_NAME} | {SEGMENT_DURATION}s | {N_FOLDS} FOLDS\n")

for fold_idx, (tr_idx, te_idx) in enumerate(skf.split(all_files, all_labels)):
    print(f"\n{'─'*55}\n  FOLD {fold_idx+1}/{N_FOLDS}\n{'─'*55}")

    Xtr_f, Xte_f = all_files[tr_idx], all_files[te_idx]
    ytr_f, yte_f = all_labels[tr_idx], all_labels[te_idx]
    Xtr_f, ytr_f = oversample_files(Xtr_f, ytr_f)

    train_ds = WaveformDataset(Xtr_f, ytr_f)
    train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=8, pin_memory=True, persistent_workers=True)
    print(f"  Segmentos train: {len(train_ds)}")

    weights  = class_weight.compute_class_weight('balanced',
                   classes=np.unique(ytr_f), y=ytr_f)
    cw_tensor = torch.tensor(weights, dtype=torch.float32).to(DEVICE)

    model     = build_model().to(DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=1e-4)
    criterion = nn.BCELoss(reduction='none')
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=3, min_lr=1e-7)

    if fold_idx == 0:
        n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
        print(f"  Parámetros entrenables: {n_params:,}\n")

    best_loss, patience_cnt, PATIENCE, best_state = float('inf'), 0, 5, None
    for epoch in range(EPOCHS):
        loss = train_one_epoch(model, train_dl, optimizer, criterion, cw_tensor)
        scheduler.step(loss)
        if loss < best_loss:
            best_loss = loss; patience_cnt = 0
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            mark = "✓"
        else:
            patience_cnt += 1
            mark = f"({patience_cnt}/{PATIENCE})"
        print(f"  Epoch {epoch+1:02d}/{EPOCHS} — loss: {loss:.4f}  best: {best_loss:.4f}  {mark}")
        if patience_cnt >= PATIENCE:
            print("  ► Early stopping activado."); break

    model.load_state_dict(best_state)

    for i, fp in enumerate(Xte_f):
        avg_prob = predict_file(model, fp)
        if avg_prob is None: continue
        global_y_true.append(int(yte_f[i]))
        global_y_pred.append(1 if avg_prob > 0.33 else 0)
        global_y_prob.append(avg_prob)

    del model, train_ds, train_dl, best_state, cw_tensor
    gc.collect(); torch.cuda.empty_cache()

print("\n" + "="*60)
print(f"RESULTADOS {MODEL_NAME} | {SEGMENT_DURATION}s  ({len(global_y_true)} pacientes)")
print("="*60)

cm = confusion_matrix(global_y_true, global_y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='YlOrBr',
            xticklabels=['Sin Ansiedad', 'Con Ansiedad'],
            yticklabels=['Sin Ansiedad', 'Con Ansiedad'])
plt.title(f'Confusión {MODEL_NAME} {SEGMENT_DURATION}s ({N_FOLDS}-Fold CV)')
plt.ylabel('Real'); plt.xlabel('Predicción')
plt.tight_layout()
plt.savefig(f'confusion_{MODEL_NAME.replace(" ","_")}_{SEGMENT_DURATION}s.png', dpi=150)
plt.show()

acc      = accuracy_score(global_y_true, global_y_pred)
prec     = precision_score(global_y_true, global_y_pred, zero_division=0)
rec      = recall_score(global_y_true, global_y_pred, zero_division=0)
f1       = f1_score(global_y_true, global_y_pred, zero_division=0)
f1_macro = f1_score(global_y_true, global_y_pred, average='macro', zero_division=0)
print(f"\nAccuracy :  {acc:.4f}")
print(f"Precision:  {prec:.4f}")
print(f"Recall   :  {rec:.4f}")
print(f"F1-Ans   :  {f1:.4f}  (clase positiva)")
print(f"F1-Macro :  {f1_macro:.4f}")
print()
print(classification_report(global_y_true, global_y_pred,
                             target_names=['Sin Ansiedad', 'Con Ansiedad']))

fpr, tpr, _ = roc_curve(global_y_true, global_y_prob)
roc_auc = auc(fpr, tpr)
plt.figure(figsize=(7, 6))
plt.plot(fpr, tpr, lw=2, label=f'AUC = {roc_auc:.2f}')
plt.plot([0,1],[0,1],'navy',lw=2,linestyle='--')
plt.xlabel('Falsos Positivos'); plt.ylabel('Verdaderos Positivos')
plt.title(f'ROC — {MODEL_NAME} {SEGMENT_DURATION}s')
plt.legend(loc='lower right'); plt.tight_layout()
plt.savefig(f'roc_{MODEL_NAME.replace(" ","_")}_{SEGMENT_DURATION}s.png', dpi=150)
plt.show()


# BiLSTM — Raw Waveform

## 10 s | Sin overlap

In [ ]:
import os
import gc
import numpy as np
import librosa
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (confusion_matrix, classification_report,
                             accuracy_score, precision_score, recall_score,
                             f1_score, roc_curve, auc)
from sklearn.utils import class_weight

# ══════════════════════════════════════════════════════════

BASE_PATH_0 = "../Clasificación Final/Ansiedad/wav/0"
BASE_PATH_1 = "../Clasificación Final/Ansiedad/wav/1"

SAMPLE_RATE         = 16000
SEGMENT_DURATION    = 10
SAMPLES_PER_SEGMENT = int(SEGMENT_DURATION * SAMPLE_RATE)  # auto-calculado
# Sin overlap para 10 s
N_FOLDS             = 5
BATCH_SIZE          = 128
EPOCHS              = 25

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Usando dispositivo: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
    torch.backends.cudnn.benchmark = True

def get_file_lists():
    files_0 = [os.path.join(BASE_PATH_0, f)
               for f in sorted(os.listdir(BASE_PATH_0)) if f.endswith('.wav')]
    files_1 = [os.path.join(BASE_PATH_1, f)
               for f in sorted(os.listdir(BASE_PATH_1)) if f.endswith('.wav')]
    files  = np.array(files_0 + files_1)
    labels = np.array([0]*len(files_0) + [1]*len(files_1), dtype=np.int64)
    return files, labels

all_files, all_labels = get_file_lists()
print(f"\nTotal archivos : {len(all_files)}")
print(f" - Sin Ansiedad (0): {np.sum(all_labels == 0)}")
print(f" - Con Ansiedad (1): {np.sum(all_labels == 1)}")
print(f"Segment: {SEGMENT_DURATION}s | Samples/seg: {SAMPLES_PER_SEGMENT}")

class WaveformDataset(Dataset):
    """Lazy loading — sin overlap para 10 s."""
    def __init__(self, file_list, label_list):
        self.samples = []
        for fp, lbl in zip(file_list, label_list):
            try:
                total = librosa.get_duration(path=fp)
                start = 0.0
                while start < total:
                    if total - start < SEGMENT_DURATION / 2:
                        break
                    self.samples.append((fp, start, int(lbl)))
                    start += SEGMENT_DURATION
            except Exception as e:
                print(f"[AVISO] {fp}: {e}")

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        fp, offset, label = self.samples[idx]
        audio, _ = librosa.load(fp, sr=SAMPLE_RATE, offset=offset,
                                duration=SEGMENT_DURATION)
        if len(audio) < SAMPLES_PER_SEGMENT:
            audio = np.pad(audio, (0, SAMPLES_PER_SEGMENT - len(audio)))
        audio = audio[:SAMPLES_PER_SEGMENT].astype(np.float32)
        audio /= (np.max(np.abs(audio)) + 1e-8)
        return torch.from_numpy(audio).unsqueeze(0), torch.tensor(label, dtype=torch.long)

def oversample_files(files, labels):
    f1 = files[labels == 1]; l1 = labels[labels == 1]
    diff = np.sum(labels == 0) - np.sum(labels == 1)
    if diff > 0:
        idx    = np.random.choice(len(f1), size=diff, replace=True)
        files  = np.concatenate([files,  f1[idx]])
        labels = np.concatenate([labels, l1[idx]])
    return files, labels

MODEL_NAME = "BiLSTM"

class BiLSTM(nn.Module):
    """CNN Subsampler → BiLSTM x2 (CuDNN-compatible, sin recurrent_dropout)."""

    def __init__(self, input_length):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv1d(1, 64, 160, stride=80, padding=80),
            nn.ReLU(inplace=True), nn.BatchNorm1d(64),
            nn.MaxPool1d(2), nn.Dropout(0.2),
            nn.Conv1d(64, 128, 9, padding=4),
            nn.ReLU(inplace=True), nn.BatchNorm1d(128), nn.Dropout(0.2),
        )
        self.lstm1 = nn.LSTM(128, 128, batch_first=True, bidirectional=True)
        self.drop1 = nn.Dropout(0.35)
        self.lstm2 = nn.LSTM(256,  64, batch_first=True, bidirectional=True)
        self.drop2 = nn.Dropout(0.35)
        self.classifier = nn.Sequential(
            nn.Linear(128, 64), nn.ReLU(inplace=True), nn.Dropout(0.4),
            nn.Linear(64, 1), nn.Sigmoid(),
        )

    def forward(self, x):
        x = self.cnn(x).permute(0, 2, 1)    # (B, T, 128)
        x, _ = self.lstm1(x); x = self.drop1(x)
        x, _ = self.lstm2(x); x = self.drop2(x)
        x = x[:, -1, :]
        return self.classifier(x).squeeze(1)

def build_model():
    return BiLSTM(SAMPLES_PER_SEGMENT)

def train_one_epoch(model, loader, optimizer, criterion, cw_tensor):
    model.train()
    total_loss = 0.0
    for xb, yb in loader:
        xb = xb.to(DEVICE, non_blocking=True)
        yb = yb.to(DEVICE, non_blocking=True).float()
        optimizer.zero_grad(set_to_none=True)
        preds = model(xb)
        sample_w = cw_tensor[yb.long()]
        loss = (criterion(preds, yb) * sample_w).mean()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

@torch.no_grad()
def predict_file(model, file_path):
    model.eval()
    segs = []
    try:
        audio, _ = librosa.load(file_path, sr=SAMPLE_RATE)
        hop = SAMPLES_PER_SEGMENT
        for start in range(0, len(audio), hop):
            seg = audio[start:start + SAMPLES_PER_SEGMENT]
            if len(seg) < SAMPLES_PER_SEGMENT // 2: continue
            if len(seg) < SAMPLES_PER_SEGMENT:
                seg = np.pad(seg, (0, SAMPLES_PER_SEGMENT - len(seg)))
            seg = seg.astype(np.float32)
            seg /= (np.max(np.abs(seg)) + 1e-8)
            segs.append(seg)
    except Exception as e:
        print(f"  Error inferencia {file_path}: {e}"); return None
    if not segs: return None
    probs = []
    for b in range(0, len(segs), BATCH_SIZE):
        batch = np.stack(segs[b:b + BATCH_SIZE])
        bt = torch.from_numpy(batch).unsqueeze(1).to(DEVICE)
        out = model(bt).cpu().numpy()
        probs.extend(out.tolist())
    return float(np.mean(probs))

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
global_y_true, global_y_pred, global_y_prob = [], [], []

print(f"\nINICIANDO CROSS-VALIDATION {MODEL_NAME} | {SEGMENT_DURATION}s | {N_FOLDS} FOLDS\n")

for fold_idx, (tr_idx, te_idx) in enumerate(skf.split(all_files, all_labels)):
    print(f"\n{'─'*55}\n  FOLD {fold_idx+1}/{N_FOLDS}\n{'─'*55}")

    Xtr_f, Xte_f = all_files[tr_idx], all_files[te_idx]
    ytr_f, yte_f = all_labels[tr_idx], all_labels[te_idx]
    Xtr_f, ytr_f = oversample_files(Xtr_f, ytr_f)

    train_ds = WaveformDataset(Xtr_f, ytr_f)
    train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=8, pin_memory=True, persistent_workers=True)
    print(f"  Segmentos train: {len(train_ds)}")

    weights  = class_weight.compute_class_weight('balanced',
                   classes=np.unique(ytr_f), y=ytr_f)
    cw_tensor = torch.tensor(weights, dtype=torch.float32).to(DEVICE)

    model     = build_model().to(DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=1e-4)
    criterion = nn.BCELoss(reduction='none')
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=3, min_lr=1e-7)

    if fold_idx == 0:
        n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
        print(f"  Parámetros entrenables: {n_params:,}\n")

    best_loss, patience_cnt, PATIENCE, best_state = float('inf'), 0, 5, None
    for epoch in range(EPOCHS):
        loss = train_one_epoch(model, train_dl, optimizer, criterion, cw_tensor)
        scheduler.step(loss)
        if loss < best_loss:
            best_loss = loss; patience_cnt = 0
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            mark = "✓"
        else:
            patience_cnt += 1
            mark = f"({patience_cnt}/{PATIENCE})"
        print(f"  Epoch {epoch+1:02d}/{EPOCHS} — loss: {loss:.4f}  best: {best_loss:.4f}  {mark}")
        if patience_cnt >= PATIENCE:
            print("  ► Early stopping activado."); break

    model.load_state_dict(best_state)

    for i, fp in enumerate(Xte_f):
        avg_prob = predict_file(model, fp)
        if avg_prob is None: continue
        global_y_true.append(int(yte_f[i]))
        global_y_pred.append(1 if avg_prob > 0.33 else 0)
        global_y_prob.append(avg_prob)

    del model, train_ds, train_dl, best_state, cw_tensor
    gc.collect(); torch.cuda.empty_cache()

print("\n" + "="*60)
print(f"RESULTADOS {MODEL_NAME} | {SEGMENT_DURATION}s  ({len(global_y_true)} pacientes)")
print("="*60)

cm = confusion_matrix(global_y_true, global_y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Purples',
            xticklabels=['Sin Ansiedad', 'Con Ansiedad'],
            yticklabels=['Sin Ansiedad', 'Con Ansiedad'])
plt.title(f'Confusión {MODEL_NAME} {SEGMENT_DURATION}s ({N_FOLDS}-Fold CV)')
plt.ylabel('Real'); plt.xlabel('Predicción')
plt.tight_layout()
plt.savefig(f'confusion_{MODEL_NAME.replace(" ","_")}_{SEGMENT_DURATION}s.png', dpi=150)
plt.show()

acc      = accuracy_score(global_y_true, global_y_pred)
prec     = precision_score(global_y_true, global_y_pred, zero_division=0)
rec      = recall_score(global_y_true, global_y_pred, zero_division=0)
f1       = f1_score(global_y_true, global_y_pred, zero_division=0)
f1_macro = f1_score(global_y_true, global_y_pred, average='macro', zero_division=0)
print(f"\nAccuracy :  {acc:.4f}")
print(f"Precision:  {prec:.4f}")
print(f"Recall   :  {rec:.4f}")
print(f"F1-Ans   :  {f1:.4f}  (clase positiva)")
print(f"F1-Macro :  {f1_macro:.4f}")
print()
print(classification_report(global_y_true, global_y_pred,
                             target_names=['Sin Ansiedad', 'Con Ansiedad']))

fpr, tpr, _ = roc_curve(global_y_true, global_y_prob)
roc_auc = auc(fpr, tpr)
plt.figure(figsize=(7, 6))
plt.plot(fpr, tpr, lw=2, label=f'AUC = {roc_auc:.2f}')
plt.plot([0,1],[0,1],'navy',lw=2,linestyle='--')
plt.xlabel('Falsos Positivos'); plt.ylabel('Verdaderos Positivos')
plt.title(f'ROC — {MODEL_NAME} {SEGMENT_DURATION}s')
plt.legend(loc='lower right'); plt.tight_layout()
plt.savefig(f'roc_{MODEL_NAME.replace(" ","_")}_{SEGMENT_DURATION}s.png', dpi=150)
plt.show()


## 2 s | Overlap 50%

In [ ]:
import os
import gc
import numpy as np
import librosa
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (confusion_matrix, classification_report,
                             accuracy_score, precision_score, recall_score,
                             f1_score, roc_curve, auc)
from sklearn.utils import class_weight

# ══════════════════════════════════════════════════════════

BASE_PATH_0 = "../Clasificación Final/Ansiedad/wav/0"
BASE_PATH_1 = "../Clasificación Final/Ansiedad/wav/1"

SAMPLE_RATE         = 16000
SEGMENT_DURATION    = 2
SAMPLES_PER_SEGMENT = int(SEGMENT_DURATION * SAMPLE_RATE)  # auto-calculado
HOP_SIZE            = SAMPLES_PER_SEGMENT // 2  # 50% overlap → 22050
N_FOLDS             = 5
BATCH_SIZE          = 128
EPOCHS              = 25

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Usando dispositivo: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
    torch.backends.cudnn.benchmark = True

def get_file_lists():
    files_0 = [os.path.join(BASE_PATH_0, f)
               for f in sorted(os.listdir(BASE_PATH_0)) if f.endswith('.wav')]
    files_1 = [os.path.join(BASE_PATH_1, f)
               for f in sorted(os.listdir(BASE_PATH_1)) if f.endswith('.wav')]
    files  = np.array(files_0 + files_1)
    labels = np.array([0]*len(files_0) + [1]*len(files_1), dtype=np.int64)
    return files, labels

all_files, all_labels = get_file_lists()
print(f"\nTotal archivos : {len(all_files)}")
print(f" - Sin Ansiedad (0): {np.sum(all_labels == 0)}")
print(f" - Con Ansiedad (1): {np.sum(all_labels == 1)}")
print(f"Segment: {SEGMENT_DURATION}s | Samples/seg: {SAMPLES_PER_SEGMENT}")

class WaveformDataset(Dataset):
    """Lazy loading — overlap 50%%."""
    def __init__(self, file_list, label_list):
        self.samples = []
        for fp, lbl in zip(file_list, label_list):
            try:
                total = librosa.get_duration(path=fp)
                n_segs = max(1, int((total * SAMPLE_RATE - SAMPLES_PER_SEGMENT) / HOP_SIZE) + 1)
                for i in range(n_segs):
                    offset = i * HOP_SIZE / SAMPLE_RATE
                    self.samples.append((fp, offset, int(lbl)))
            except Exception as e:
                print(f"[AVISO] {fp}: {e}")

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        fp, offset, label = self.samples[idx]
        audio, _ = librosa.load(fp, sr=SAMPLE_RATE, offset=offset,
                                duration=SEGMENT_DURATION)
        if len(audio) < SAMPLES_PER_SEGMENT:
            audio = np.pad(audio, (0, SAMPLES_PER_SEGMENT - len(audio)))
        audio = audio[:SAMPLES_PER_SEGMENT].astype(np.float32)
        audio /= (np.max(np.abs(audio)) + 1e-8)
        return torch.from_numpy(audio).unsqueeze(0), torch.tensor(label, dtype=torch.long)

def oversample_files(files, labels):
    f1 = files[labels == 1]; l1 = labels[labels == 1]
    diff = np.sum(labels == 0) - np.sum(labels == 1)
    if diff > 0:
        idx    = np.random.choice(len(f1), size=diff, replace=True)
        files  = np.concatenate([files,  f1[idx]])
        labels = np.concatenate([labels, l1[idx]])
    return files, labels

MODEL_NAME = "BiLSTM"

class BiLSTM(nn.Module):
    """CNN Subsampler → BiLSTM x2 (CuDNN-compatible, sin recurrent_dropout)."""

    def __init__(self, input_length):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv1d(1, 64, 160, stride=80, padding=80),
            nn.ReLU(inplace=True), nn.BatchNorm1d(64),
            nn.MaxPool1d(2), nn.Dropout(0.2),
            nn.Conv1d(64, 128, 9, padding=4),
            nn.ReLU(inplace=True), nn.BatchNorm1d(128), nn.Dropout(0.2),
        )
        self.lstm1 = nn.LSTM(128, 128, batch_first=True, bidirectional=True)
        self.drop1 = nn.Dropout(0.35)
        self.lstm2 = nn.LSTM(256,  64, batch_first=True, bidirectional=True)
        self.drop2 = nn.Dropout(0.35)
        self.classifier = nn.Sequential(
            nn.Linear(128, 64), nn.ReLU(inplace=True), nn.Dropout(0.4),
            nn.Linear(64, 1), nn.Sigmoid(),
        )

    def forward(self, x):
        x = self.cnn(x).permute(0, 2, 1)    # (B, T, 128)
        x, _ = self.lstm1(x); x = self.drop1(x)
        x, _ = self.lstm2(x); x = self.drop2(x)
        x = x[:, -1, :]
        return self.classifier(x).squeeze(1)

def build_model():
    return BiLSTM(SAMPLES_PER_SEGMENT)

def train_one_epoch(model, loader, optimizer, criterion, cw_tensor):
    model.train()
    total_loss = 0.0
    for xb, yb in loader:
        xb = xb.to(DEVICE, non_blocking=True)
        yb = yb.to(DEVICE, non_blocking=True).float()
        optimizer.zero_grad(set_to_none=True)
        preds = model(xb)
        sample_w = cw_tensor[yb.long()]
        loss = (criterion(preds, yb) * sample_w).mean()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

@torch.no_grad()
def predict_file(model, file_path):
    model.eval()
    segs = []
    try:
        audio, _ = librosa.load(file_path, sr=SAMPLE_RATE)
        hop = HOP_SIZE
        for start in range(0, len(audio), hop):
            seg = audio[start:start + SAMPLES_PER_SEGMENT]
            if len(seg) < SAMPLES_PER_SEGMENT // 2: continue
            if len(seg) < SAMPLES_PER_SEGMENT:
                seg = np.pad(seg, (0, SAMPLES_PER_SEGMENT - len(seg)))
            seg = seg.astype(np.float32)
            seg /= (np.max(np.abs(seg)) + 1e-8)
            segs.append(seg)
    except Exception as e:
        print(f"  Error inferencia {file_path}: {e}"); return None
    if not segs: return None
    probs = []
    for b in range(0, len(segs), BATCH_SIZE):
        batch = np.stack(segs[b:b + BATCH_SIZE])
        bt = torch.from_numpy(batch).unsqueeze(1).to(DEVICE)
        out = model(bt).cpu().numpy()
        probs.extend(out.tolist())
    return float(np.mean(probs))

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
global_y_true, global_y_pred, global_y_prob = [], [], []

print(f"\nINICIANDO CROSS-VALIDATION {MODEL_NAME} | {SEGMENT_DURATION}s | {N_FOLDS} FOLDS\n")

for fold_idx, (tr_idx, te_idx) in enumerate(skf.split(all_files, all_labels)):
    print(f"\n{'─'*55}\n  FOLD {fold_idx+1}/{N_FOLDS}\n{'─'*55}")

    Xtr_f, Xte_f = all_files[tr_idx], all_files[te_idx]
    ytr_f, yte_f = all_labels[tr_idx], all_labels[te_idx]
    Xtr_f, ytr_f = oversample_files(Xtr_f, ytr_f)

    train_ds = WaveformDataset(Xtr_f, ytr_f)
    train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=8, pin_memory=True, persistent_workers=True)
    print(f"  Segmentos train: {len(train_ds)}")

    weights  = class_weight.compute_class_weight('balanced',
                   classes=np.unique(ytr_f), y=ytr_f)
    cw_tensor = torch.tensor(weights, dtype=torch.float32).to(DEVICE)

    model     = build_model().to(DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=1e-4)
    criterion = nn.BCELoss(reduction='none')
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=3, min_lr=1e-7)

    if fold_idx == 0:
        n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
        print(f"  Parámetros entrenables: {n_params:,}\n")

    best_loss, patience_cnt, PATIENCE, best_state = float('inf'), 0, 5, None
    for epoch in range(EPOCHS):
        loss = train_one_epoch(model, train_dl, optimizer, criterion, cw_tensor)
        scheduler.step(loss)
        if loss < best_loss:
            best_loss = loss; patience_cnt = 0
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            mark = "✓"
        else:
            patience_cnt += 1
            mark = f"({patience_cnt}/{PATIENCE})"
        print(f"  Epoch {epoch+1:02d}/{EPOCHS} — loss: {loss:.4f}  best: {best_loss:.4f}  {mark}")
        if patience_cnt >= PATIENCE:
            print("  ► Early stopping activado."); break

    model.load_state_dict(best_state)

    for i, fp in enumerate(Xte_f):
        avg_prob = predict_file(model, fp)
        if avg_prob is None: continue
        global_y_true.append(int(yte_f[i]))
        global_y_pred.append(1 if avg_prob > 0.33 else 0)
        global_y_prob.append(avg_prob)

    del model, train_ds, train_dl, best_state, cw_tensor
    gc.collect(); torch.cuda.empty_cache()

print("\n" + "="*60)
print(f"RESULTADOS {MODEL_NAME} | {SEGMENT_DURATION}s  ({len(global_y_true)} pacientes)")
print("="*60)

cm = confusion_matrix(global_y_true, global_y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Purples',
            xticklabels=['Sin Ansiedad', 'Con Ansiedad'],
            yticklabels=['Sin Ansiedad', 'Con Ansiedad'])
plt.title(f'Confusión {MODEL_NAME} {SEGMENT_DURATION}s ({N_FOLDS}-Fold CV)')
plt.ylabel('Real'); plt.xlabel('Predicción')
plt.tight_layout()
plt.savefig(f'confusion_{MODEL_NAME.replace(" ","_")}_{SEGMENT_DURATION}s.png', dpi=150)
plt.show()

acc      = accuracy_score(global_y_true, global_y_pred)
prec     = precision_score(global_y_true, global_y_pred, zero_division=0)
rec      = recall_score(global_y_true, global_y_pred, zero_division=0)
f1       = f1_score(global_y_true, global_y_pred, zero_division=0)
f1_macro = f1_score(global_y_true, global_y_pred, average='macro', zero_division=0)
print(f"\nAccuracy :  {acc:.4f}")
print(f"Precision:  {prec:.4f}")
print(f"Recall   :  {rec:.4f}")
print(f"F1-Ans   :  {f1:.4f}  (clase positiva)")
print(f"F1-Macro :  {f1_macro:.4f}")
print()
print(classification_report(global_y_true, global_y_pred,
                             target_names=['Sin Ansiedad', 'Con Ansiedad']))

fpr, tpr, _ = roc_curve(global_y_true, global_y_prob)
roc_auc = auc(fpr, tpr)
plt.figure(figsize=(7, 6))
plt.plot(fpr, tpr, lw=2, label=f'AUC = {roc_auc:.2f}')
plt.plot([0,1],[0,1],'navy',lw=2,linestyle='--')
plt.xlabel('Falsos Positivos'); plt.ylabel('Verdaderos Positivos')
plt.title(f'ROC — {MODEL_NAME} {SEGMENT_DURATION}s')
plt.legend(loc='lower right'); plt.tight_layout()
plt.savefig(f'roc_{MODEL_NAME.replace(" ","_")}_{SEGMENT_DURATION}s.png', dpi=150)
plt.show()


## 3 s | Overlap 50%

In [ ]:
import os
import gc
import numpy as np
import librosa
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (confusion_matrix, classification_report,
                             accuracy_score, precision_score, recall_score,
                             f1_score, roc_curve, auc)
from sklearn.utils import class_weight

# ══════════════════════════════════════════════════════════

BASE_PATH_0 = "../Clasificación Final/Ansiedad/wav/0"
BASE_PATH_1 = "../Clasificación Final/Ansiedad/wav/1"

SAMPLE_RATE         = 16000
SEGMENT_DURATION    = 3
SAMPLES_PER_SEGMENT = int(SEGMENT_DURATION * SAMPLE_RATE)  # auto-calculado
HOP_SIZE            = SAMPLES_PER_SEGMENT // 2  # 50% overlap → 33075
N_FOLDS             = 5
BATCH_SIZE          = 128
EPOCHS              = 25

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Usando dispositivo: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
    torch.backends.cudnn.benchmark = True

def get_file_lists():
    files_0 = [os.path.join(BASE_PATH_0, f)
               for f in sorted(os.listdir(BASE_PATH_0)) if f.endswith('.wav')]
    files_1 = [os.path.join(BASE_PATH_1, f)
               for f in sorted(os.listdir(BASE_PATH_1)) if f.endswith('.wav')]
    files  = np.array(files_0 + files_1)
    labels = np.array([0]*len(files_0) + [1]*len(files_1), dtype=np.int64)
    return files, labels

all_files, all_labels = get_file_lists()
print(f"\nTotal archivos : {len(all_files)}")
print(f" - Sin Ansiedad (0): {np.sum(all_labels == 0)}")
print(f" - Con Ansiedad (1): {np.sum(all_labels == 1)}")
print(f"Segment: {SEGMENT_DURATION}s | Samples/seg: {SAMPLES_PER_SEGMENT}")

class WaveformDataset(Dataset):
    """Lazy loading — overlap 50%%."""
    def __init__(self, file_list, label_list):
        self.samples = []
        for fp, lbl in zip(file_list, label_list):
            try:
                total = librosa.get_duration(path=fp)
                n_segs = max(1, int((total * SAMPLE_RATE - SAMPLES_PER_SEGMENT) / HOP_SIZE) + 1)
                for i in range(n_segs):
                    offset = i * HOP_SIZE / SAMPLE_RATE
                    self.samples.append((fp, offset, int(lbl)))
            except Exception as e:
                print(f"[AVISO] {fp}: {e}")

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        fp, offset, label = self.samples[idx]
        audio, _ = librosa.load(fp, sr=SAMPLE_RATE, offset=offset,
                                duration=SEGMENT_DURATION)
        if len(audio) < SAMPLES_PER_SEGMENT:
            audio = np.pad(audio, (0, SAMPLES_PER_SEGMENT - len(audio)))
        audio = audio[:SAMPLES_PER_SEGMENT].astype(np.float32)
        audio /= (np.max(np.abs(audio)) + 1e-8)
        return torch.from_numpy(audio).unsqueeze(0), torch.tensor(label, dtype=torch.long)

def oversample_files(files, labels):
    f1 = files[labels == 1]; l1 = labels[labels == 1]
    diff = np.sum(labels == 0) - np.sum(labels == 1)
    if diff > 0:
        idx    = np.random.choice(len(f1), size=diff, replace=True)
        files  = np.concatenate([files,  f1[idx]])
        labels = np.concatenate([labels, l1[idx]])
    return files, labels

MODEL_NAME = "BiLSTM"

class BiLSTM(nn.Module):
    """CNN Subsampler → BiLSTM x2 (CuDNN-compatible, sin recurrent_dropout)."""

    def __init__(self, input_length):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv1d(1, 64, 160, stride=80, padding=80),
            nn.ReLU(inplace=True), nn.BatchNorm1d(64),
            nn.MaxPool1d(2), nn.Dropout(0.2),
            nn.Conv1d(64, 128, 9, padding=4),
            nn.ReLU(inplace=True), nn.BatchNorm1d(128), nn.Dropout(0.2),
        )
        self.lstm1 = nn.LSTM(128, 128, batch_first=True, bidirectional=True)
        self.drop1 = nn.Dropout(0.35)
        self.lstm2 = nn.LSTM(256,  64, batch_first=True, bidirectional=True)
        self.drop2 = nn.Dropout(0.35)
        self.classifier = nn.Sequential(
            nn.Linear(128, 64), nn.ReLU(inplace=True), nn.Dropout(0.4),
            nn.Linear(64, 1), nn.Sigmoid(),
        )

    def forward(self, x):
        x = self.cnn(x).permute(0, 2, 1)    # (B, T, 128)
        x, _ = self.lstm1(x); x = self.drop1(x)
        x, _ = self.lstm2(x); x = self.drop2(x)
        x = x[:, -1, :]
        return self.classifier(x).squeeze(1)

def build_model():
    return BiLSTM(SAMPLES_PER_SEGMENT)

def train_one_epoch(model, loader, optimizer, criterion, cw_tensor):
    model.train()
    total_loss = 0.0
    for xb, yb in loader:
        xb = xb.to(DEVICE, non_blocking=True)
        yb = yb.to(DEVICE, non_blocking=True).float()
        optimizer.zero_grad(set_to_none=True)
        preds = model(xb)
        sample_w = cw_tensor[yb.long()]
        loss = (criterion(preds, yb) * sample_w).mean()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

@torch.no_grad()
def predict_file(model, file_path):
    model.eval()
    segs = []
    try:
        audio, _ = librosa.load(file_path, sr=SAMPLE_RATE)
        hop = HOP_SIZE
        for start in range(0, len(audio), hop):
            seg = audio[start:start + SAMPLES_PER_SEGMENT]
            if len(seg) < SAMPLES_PER_SEGMENT // 2: continue
            if len(seg) < SAMPLES_PER_SEGMENT:
                seg = np.pad(seg, (0, SAMPLES_PER_SEGMENT - len(seg)))
            seg = seg.astype(np.float32)
            seg /= (np.max(np.abs(seg)) + 1e-8)
            segs.append(seg)
    except Exception as e:
        print(f"  Error inferencia {file_path}: {e}"); return None
    if not segs: return None
    probs = []
    for b in range(0, len(segs), BATCH_SIZE):
        batch = np.stack(segs[b:b + BATCH_SIZE])
        bt = torch.from_numpy(batch).unsqueeze(1).to(DEVICE)
        out = model(bt).cpu().numpy()
        probs.extend(out.tolist())
    return float(np.mean(probs))

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
global_y_true, global_y_pred, global_y_prob = [], [], []

print(f"\nINICIANDO CROSS-VALIDATION {MODEL_NAME} | {SEGMENT_DURATION}s | {N_FOLDS} FOLDS\n")

for fold_idx, (tr_idx, te_idx) in enumerate(skf.split(all_files, all_labels)):
    print(f"\n{'─'*55}\n  FOLD {fold_idx+1}/{N_FOLDS}\n{'─'*55}")

    Xtr_f, Xte_f = all_files[tr_idx], all_files[te_idx]
    ytr_f, yte_f = all_labels[tr_idx], all_labels[te_idx]
    Xtr_f, ytr_f = oversample_files(Xtr_f, ytr_f)

    train_ds = WaveformDataset(Xtr_f, ytr_f)
    train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=8, pin_memory=True, persistent_workers=True)
    print(f"  Segmentos train: {len(train_ds)}")

    weights  = class_weight.compute_class_weight('balanced',
                   classes=np.unique(ytr_f), y=ytr_f)
    cw_tensor = torch.tensor(weights, dtype=torch.float32).to(DEVICE)

    model     = build_model().to(DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=1e-4)
    criterion = nn.BCELoss(reduction='none')
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=3, min_lr=1e-7)

    if fold_idx == 0:
        n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
        print(f"  Parámetros entrenables: {n_params:,}\n")

    best_loss, patience_cnt, PATIENCE, best_state = float('inf'), 0, 5, None
    for epoch in range(EPOCHS):
        loss = train_one_epoch(model, train_dl, optimizer, criterion, cw_tensor)
        scheduler.step(loss)
        if loss < best_loss:
            best_loss = loss; patience_cnt = 0
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            mark = "✓"
        else:
            patience_cnt += 1
            mark = f"({patience_cnt}/{PATIENCE})"
        print(f"  Epoch {epoch+1:02d}/{EPOCHS} — loss: {loss:.4f}  best: {best_loss:.4f}  {mark}")
        if patience_cnt >= PATIENCE:
            print("  ► Early stopping activado."); break

    model.load_state_dict(best_state)

    for i, fp in enumerate(Xte_f):
        avg_prob = predict_file(model, fp)
        if avg_prob is None: continue
        global_y_true.append(int(yte_f[i]))
        global_y_pred.append(1 if avg_prob > 0.33 else 0)
        global_y_prob.append(avg_prob)

    del model, train_ds, train_dl, best_state, cw_tensor
    gc.collect(); torch.cuda.empty_cache()

print("\n" + "="*60)
print(f"RESULTADOS {MODEL_NAME} | {SEGMENT_DURATION}s  ({len(global_y_true)} pacientes)")
print("="*60)

cm = confusion_matrix(global_y_true, global_y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Purples',
            xticklabels=['Sin Ansiedad', 'Con Ansiedad'],
            yticklabels=['Sin Ansiedad', 'Con Ansiedad'])
plt.title(f'Confusión {MODEL_NAME} {SEGMENT_DURATION}s ({N_FOLDS}-Fold CV)')
plt.ylabel('Real'); plt.xlabel('Predicción')
plt.tight_layout()
plt.savefig(f'confusion_{MODEL_NAME.replace(" ","_")}_{SEGMENT_DURATION}s.png', dpi=150)
plt.show()

acc      = accuracy_score(global_y_true, global_y_pred)
prec     = precision_score(global_y_true, global_y_pred, zero_division=0)
rec      = recall_score(global_y_true, global_y_pred, zero_division=0)
f1       = f1_score(global_y_true, global_y_pred, zero_division=0)
f1_macro = f1_score(global_y_true, global_y_pred, average='macro', zero_division=0)
print(f"\nAccuracy :  {acc:.4f}")
print(f"Precision:  {prec:.4f}")
print(f"Recall   :  {rec:.4f}")
print(f"F1-Ans   :  {f1:.4f}  (clase positiva)")
print(f"F1-Macro :  {f1_macro:.4f}")
print()
print(classification_report(global_y_true, global_y_pred,
                             target_names=['Sin Ansiedad', 'Con Ansiedad']))

fpr, tpr, _ = roc_curve(global_y_true, global_y_prob)
roc_auc = auc(fpr, tpr)
plt.figure(figsize=(7, 6))
plt.plot(fpr, tpr, lw=2, label=f'AUC = {roc_auc:.2f}')
plt.plot([0,1],[0,1],'navy',lw=2,linestyle='--')
plt.xlabel('Falsos Positivos'); plt.ylabel('Verdaderos Positivos')
plt.title(f'ROC — {MODEL_NAME} {SEGMENT_DURATION}s')
plt.legend(loc='lower right'); plt.tight_layout()
plt.savefig(f'roc_{MODEL_NAME.replace(" ","_")}_{SEGMENT_DURATION}s.png', dpi=150)
plt.show()


## 4 s | Overlap 50%

In [ ]:
import os
import gc
import numpy as np
import librosa
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (confusion_matrix, classification_report,
                             accuracy_score, precision_score, recall_score,
                             f1_score, roc_curve, auc)
from sklearn.utils import class_weight

# ══════════════════════════════════════════════════════════

BASE_PATH_0 = "../Clasificación Final/Ansiedad/wav/0"
BASE_PATH_1 = "../Clasificación Final/Ansiedad/wav/1"

SAMPLE_RATE         = 16000
SEGMENT_DURATION    = 4
SAMPLES_PER_SEGMENT = int(SEGMENT_DURATION * SAMPLE_RATE)  # auto-calculado
HOP_SIZE            = SAMPLES_PER_SEGMENT // 2  # 50% overlap → 44100
N_FOLDS             = 5
BATCH_SIZE          = 128
EPOCHS              = 25

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Usando dispositivo: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
    torch.backends.cudnn.benchmark = True

def get_file_lists():
    files_0 = [os.path.join(BASE_PATH_0, f)
               for f in sorted(os.listdir(BASE_PATH_0)) if f.endswith('.wav')]
    files_1 = [os.path.join(BASE_PATH_1, f)
               for f in sorted(os.listdir(BASE_PATH_1)) if f.endswith('.wav')]
    files  = np.array(files_0 + files_1)
    labels = np.array([0]*len(files_0) + [1]*len(files_1), dtype=np.int64)
    return files, labels

all_files, all_labels = get_file_lists()
print(f"\nTotal archivos : {len(all_files)}")
print(f" - Sin Ansiedad (0): {np.sum(all_labels == 0)}")
print(f" - Con Ansiedad (1): {np.sum(all_labels == 1)}")
print(f"Segment: {SEGMENT_DURATION}s | Samples/seg: {SAMPLES_PER_SEGMENT}")

class WaveformDataset(Dataset):
    """Lazy loading — overlap 50%%."""
    def __init__(self, file_list, label_list):
        self.samples = []
        for fp, lbl in zip(file_list, label_list):
            try:
                total = librosa.get_duration(path=fp)
                n_segs = max(1, int((total * SAMPLE_RATE - SAMPLES_PER_SEGMENT) / HOP_SIZE) + 1)
                for i in range(n_segs):
                    offset = i * HOP_SIZE / SAMPLE_RATE
                    self.samples.append((fp, offset, int(lbl)))
            except Exception as e:
                print(f"[AVISO] {fp}: {e}")

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        fp, offset, label = self.samples[idx]
        audio, _ = librosa.load(fp, sr=SAMPLE_RATE, offset=offset,
                                duration=SEGMENT_DURATION)
        if len(audio) < SAMPLES_PER_SEGMENT:
            audio = np.pad(audio, (0, SAMPLES_PER_SEGMENT - len(audio)))
        audio = audio[:SAMPLES_PER_SEGMENT].astype(np.float32)
        audio /= (np.max(np.abs(audio)) + 1e-8)
        return torch.from_numpy(audio).unsqueeze(0), torch.tensor(label, dtype=torch.long)

def oversample_files(files, labels):
    f1 = files[labels == 1]; l1 = labels[labels == 1]
    diff = np.sum(labels == 0) - np.sum(labels == 1)
    if diff > 0:
        idx    = np.random.choice(len(f1), size=diff, replace=True)
        files  = np.concatenate([files,  f1[idx]])
        labels = np.concatenate([labels, l1[idx]])
    return files, labels

MODEL_NAME = "BiLSTM"

class BiLSTM(nn.Module):
    """CNN Subsampler → BiLSTM x2 (CuDNN-compatible, sin recurrent_dropout)."""

    def __init__(self, input_length):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv1d(1, 64, 160, stride=80, padding=80),
            nn.ReLU(inplace=True), nn.BatchNorm1d(64),
            nn.MaxPool1d(2), nn.Dropout(0.2),
            nn.Conv1d(64, 128, 9, padding=4),
            nn.ReLU(inplace=True), nn.BatchNorm1d(128), nn.Dropout(0.2),
        )
        self.lstm1 = nn.LSTM(128, 128, batch_first=True, bidirectional=True)
        self.drop1 = nn.Dropout(0.35)
        self.lstm2 = nn.LSTM(256,  64, batch_first=True, bidirectional=True)
        self.drop2 = nn.Dropout(0.35)
        self.classifier = nn.Sequential(
            nn.Linear(128, 64), nn.ReLU(inplace=True), nn.Dropout(0.4),
            nn.Linear(64, 1), nn.Sigmoid(),
        )

    def forward(self, x):
        x = self.cnn(x).permute(0, 2, 1)    # (B, T, 128)
        x, _ = self.lstm1(x); x = self.drop1(x)
        x, _ = self.lstm2(x); x = self.drop2(x)
        x = x[:, -1, :]
        return self.classifier(x).squeeze(1)

def build_model():
    return BiLSTM(SAMPLES_PER_SEGMENT)

def train_one_epoch(model, loader, optimizer, criterion, cw_tensor):
    model.train()
    total_loss = 0.0
    for xb, yb in loader:
        xb = xb.to(DEVICE, non_blocking=True)
        yb = yb.to(DEVICE, non_blocking=True).float()
        optimizer.zero_grad(set_to_none=True)
        preds = model(xb)
        sample_w = cw_tensor[yb.long()]
        loss = (criterion(preds, yb) * sample_w).mean()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

@torch.no_grad()
def predict_file(model, file_path):
    model.eval()
    segs = []
    try:
        audio, _ = librosa.load(file_path, sr=SAMPLE_RATE)
        hop = HOP_SIZE
        for start in range(0, len(audio), hop):
            seg = audio[start:start + SAMPLES_PER_SEGMENT]
            if len(seg) < SAMPLES_PER_SEGMENT // 2: continue
            if len(seg) < SAMPLES_PER_SEGMENT:
                seg = np.pad(seg, (0, SAMPLES_PER_SEGMENT - len(seg)))
            seg = seg.astype(np.float32)
            seg /= (np.max(np.abs(seg)) + 1e-8)
            segs.append(seg)
    except Exception as e:
        print(f"  Error inferencia {file_path}: {e}"); return None
    if not segs: return None
    probs = []
    for b in range(0, len(segs), BATCH_SIZE):
        batch = np.stack(segs[b:b + BATCH_SIZE])
        bt = torch.from_numpy(batch).unsqueeze(1).to(DEVICE)
        out = model(bt).cpu().numpy()
        probs.extend(out.tolist())
    return float(np.mean(probs))

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
global_y_true, global_y_pred, global_y_prob = [], [], []

print(f"\nINICIANDO CROSS-VALIDATION {MODEL_NAME} | {SEGMENT_DURATION}s | {N_FOLDS} FOLDS\n")

for fold_idx, (tr_idx, te_idx) in enumerate(skf.split(all_files, all_labels)):
    print(f"\n{'─'*55}\n  FOLD {fold_idx+1}/{N_FOLDS}\n{'─'*55}")

    Xtr_f, Xte_f = all_files[tr_idx], all_files[te_idx]
    ytr_f, yte_f = all_labels[tr_idx], all_labels[te_idx]
    Xtr_f, ytr_f = oversample_files(Xtr_f, ytr_f)

    train_ds = WaveformDataset(Xtr_f, ytr_f)
    train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=8, pin_memory=True, persistent_workers=True)
    print(f"  Segmentos train: {len(train_ds)}")

    weights  = class_weight.compute_class_weight('balanced',
                   classes=np.unique(ytr_f), y=ytr_f)
    cw_tensor = torch.tensor(weights, dtype=torch.float32).to(DEVICE)

    model     = build_model().to(DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=1e-4)
    criterion = nn.BCELoss(reduction='none')
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=3, min_lr=1e-7)

    if fold_idx == 0:
        n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
        print(f"  Parámetros entrenables: {n_params:,}\n")

    best_loss, patience_cnt, PATIENCE, best_state = float('inf'), 0, 5, None
    for epoch in range(EPOCHS):
        loss = train_one_epoch(model, train_dl, optimizer, criterion, cw_tensor)
        scheduler.step(loss)
        if loss < best_loss:
            best_loss = loss; patience_cnt = 0
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            mark = "✓"
        else:
            patience_cnt += 1
            mark = f"({patience_cnt}/{PATIENCE})"
        print(f"  Epoch {epoch+1:02d}/{EPOCHS} — loss: {loss:.4f}  best: {best_loss:.4f}  {mark}")
        if patience_cnt >= PATIENCE:
            print("  ► Early stopping activado."); break

    model.load_state_dict(best_state)

    for i, fp in enumerate(Xte_f):
        avg_prob = predict_file(model, fp)
        if avg_prob is None: continue
        global_y_true.append(int(yte_f[i]))
        global_y_pred.append(1 if avg_prob > 0.33 else 0)
        global_y_prob.append(avg_prob)

    del model, train_ds, train_dl, best_state, cw_tensor
    gc.collect(); torch.cuda.empty_cache()

print("\n" + "="*60)
print(f"RESULTADOS {MODEL_NAME} | {SEGMENT_DURATION}s  ({len(global_y_true)} pacientes)")
print("="*60)

cm = confusion_matrix(global_y_true, global_y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Purples',
            xticklabels=['Sin Ansiedad', 'Con Ansiedad'],
            yticklabels=['Sin Ansiedad', 'Con Ansiedad'])
plt.title(f'Confusión {MODEL_NAME} {SEGMENT_DURATION}s ({N_FOLDS}-Fold CV)')
plt.ylabel('Real'); plt.xlabel('Predicción')
plt.tight_layout()
plt.savefig(f'confusion_{MODEL_NAME.replace(" ","_")}_{SEGMENT_DURATION}s.png', dpi=150)
plt.show()

acc      = accuracy_score(global_y_true, global_y_pred)
prec     = precision_score(global_y_true, global_y_pred, zero_division=0)
rec      = recall_score(global_y_true, global_y_pred, zero_division=0)
f1       = f1_score(global_y_true, global_y_pred, zero_division=0)
f1_macro = f1_score(global_y_true, global_y_pred, average='macro', zero_division=0)
print(f"\nAccuracy :  {acc:.4f}")
print(f"Precision:  {prec:.4f}")
print(f"Recall   :  {rec:.4f}")
print(f"F1-Ans   :  {f1:.4f}  (clase positiva)")
print(f"F1-Macro :  {f1_macro:.4f}")
print()
print(classification_report(global_y_true, global_y_pred,
                             target_names=['Sin Ansiedad', 'Con Ansiedad']))

fpr, tpr, _ = roc_curve(global_y_true, global_y_prob)
roc_auc = auc(fpr, tpr)
plt.figure(figsize=(7, 6))
plt.plot(fpr, tpr, lw=2, label=f'AUC = {roc_auc:.2f}')
plt.plot([0,1],[0,1],'navy',lw=2,linestyle='--')
plt.xlabel('Falsos Positivos'); plt.ylabel('Verdaderos Positivos')
plt.title(f'ROC — {MODEL_NAME} {SEGMENT_DURATION}s')
plt.legend(loc='lower right'); plt.tight_layout()
plt.savefig(f'roc_{MODEL_NAME.replace(" ","_")}_{SEGMENT_DURATION}s.png', dpi=150)
plt.show()


# CRNN — Convolutional + Recurrent Raw Waveform

## 10 s | Sin overlap

In [ ]:
import os
import gc
import numpy as np
import librosa
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (confusion_matrix, classification_report,
                             accuracy_score, precision_score, recall_score,
                             f1_score, roc_curve, auc)
from sklearn.utils import class_weight

# ══════════════════════════════════════════════════════════

BASE_PATH_0 = "../Clasificación Final/Ansiedad/wav/0"
BASE_PATH_1 = "../Clasificación Final/Ansiedad/wav/1"

SAMPLE_RATE         = 16000
SEGMENT_DURATION    = 10
SAMPLES_PER_SEGMENT = int(SEGMENT_DURATION * SAMPLE_RATE)  # auto-calculado
# Sin overlap para 10 s
N_FOLDS             = 5
BATCH_SIZE          = 128
EPOCHS              = 25

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Usando dispositivo: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
    torch.backends.cudnn.benchmark = True

def get_file_lists():
    files_0 = [os.path.join(BASE_PATH_0, f)
               for f in sorted(os.listdir(BASE_PATH_0)) if f.endswith('.wav')]
    files_1 = [os.path.join(BASE_PATH_1, f)
               for f in sorted(os.listdir(BASE_PATH_1)) if f.endswith('.wav')]
    files  = np.array(files_0 + files_1)
    labels = np.array([0]*len(files_0) + [1]*len(files_1), dtype=np.int64)
    return files, labels

all_files, all_labels = get_file_lists()
print(f"\nTotal archivos : {len(all_files)}")
print(f" - Sin Ansiedad (0): {np.sum(all_labels == 0)}")
print(f" - Con Ansiedad (1): {np.sum(all_labels == 1)}")
print(f"Segment: {SEGMENT_DURATION}s | Samples/seg: {SAMPLES_PER_SEGMENT}")

class WaveformDataset(Dataset):
    """Lazy loading — sin overlap para 10 s."""
    def __init__(self, file_list, label_list):
        self.samples = []
        for fp, lbl in zip(file_list, label_list):
            try:
                total = librosa.get_duration(path=fp)
                start = 0.0
                while start < total:
                    if total - start < SEGMENT_DURATION / 2:
                        break
                    self.samples.append((fp, start, int(lbl)))
                    start += SEGMENT_DURATION
            except Exception as e:
                print(f"[AVISO] {fp}: {e}")

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        fp, offset, label = self.samples[idx]
        audio, _ = librosa.load(fp, sr=SAMPLE_RATE, offset=offset,
                                duration=SEGMENT_DURATION)
        if len(audio) < SAMPLES_PER_SEGMENT:
            audio = np.pad(audio, (0, SAMPLES_PER_SEGMENT - len(audio)))
        audio = audio[:SAMPLES_PER_SEGMENT].astype(np.float32)
        audio /= (np.max(np.abs(audio)) + 1e-8)
        return torch.from_numpy(audio).unsqueeze(0), torch.tensor(label, dtype=torch.long)

def oversample_files(files, labels):
    f1 = files[labels == 1]; l1 = labels[labels == 1]
    diff = np.sum(labels == 0) - np.sum(labels == 1)
    if diff > 0:
        idx    = np.random.choice(len(f1), size=diff, replace=True)
        files  = np.concatenate([files,  f1[idx]])
        labels = np.concatenate([labels, l1[idx]])
    return files, labels

MODEL_NAME = "CRNN"

class CRNN(nn.Module):
    """Convolutional + Recurrent NN.
    CNN extrae features locales; BiGRU captura dependencias temporales."""

    def __init__(self, input_length):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv1d(1, 64, 80, stride=40, padding=40),
            nn.BatchNorm1d(64), nn.ReLU(inplace=True),
            nn.MaxPool1d(4), nn.Dropout(0.2),

            nn.Conv1d(64,  128, 16, padding=8),
            nn.BatchNorm1d(128), nn.ReLU(inplace=True),
            nn.MaxPool1d(2), nn.Dropout(0.25),

            nn.Conv1d(128, 256, 8, padding=4),
            nn.BatchNorm1d(256), nn.ReLU(inplace=True), nn.Dropout(0.3),

            nn.Conv1d(256, 256, 4, padding=2),
            nn.BatchNorm1d(256), nn.ReLU(inplace=True), nn.Dropout(0.3),
        )
        # BiGRU: output = hidden*2 por bidireccional
        self.bigru1 = nn.GRU(256, 128, batch_first=True, bidirectional=True)
        self.bn_gru  = nn.BatchNorm1d(256)
        self.drop_g1 = nn.Dropout(0.35)
        self.bigru2 = nn.GRU(256, 64, batch_first=True, bidirectional=True)
        self.drop_g2 = nn.Dropout(0.3)
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.classifier = nn.Sequential(
            nn.Linear(128, 64), nn.ReLU(inplace=True), nn.Dropout(0.5),
            nn.Linear(64,  32), nn.ReLU(inplace=True), nn.Dropout(0.4),
            nn.Linear(32,   1), nn.Sigmoid(),
        )

    def forward(self, x):
        x = self.cnn(x)                     # (B, 256, T)
        x = x.permute(0, 2, 1)             # (B, T, 256)
        x, _ = self.bigru1(x)              # (B, T, 256)
        x = self.bn_gru(x.permute(0,2,1)).permute(0,2,1)
        x = self.drop_g1(x)
        x, _ = self.bigru2(x)              # (B, T, 128)
        x = self.drop_g2(x)
        x = self.pool(x.permute(0,2,1)).squeeze(2)  # (B, 128)
        return self.classifier(x).squeeze(1)

def build_model():
    return CRNN(SAMPLES_PER_SEGMENT)

def train_one_epoch(model, loader, optimizer, criterion, cw_tensor):
    model.train()
    total_loss = 0.0
    for xb, yb in loader:
        xb = xb.to(DEVICE, non_blocking=True)
        yb = yb.to(DEVICE, non_blocking=True).float()
        optimizer.zero_grad(set_to_none=True)
        preds = model(xb)
        sample_w = cw_tensor[yb.long()]
        loss = (criterion(preds, yb) * sample_w).mean()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

@torch.no_grad()
def predict_file(model, file_path):
    model.eval()
    segs = []
    try:
        audio, _ = librosa.load(file_path, sr=SAMPLE_RATE)
        hop = SAMPLES_PER_SEGMENT
        for start in range(0, len(audio), hop):
            seg = audio[start:start + SAMPLES_PER_SEGMENT]
            if len(seg) < SAMPLES_PER_SEGMENT // 2: continue
            if len(seg) < SAMPLES_PER_SEGMENT:
                seg = np.pad(seg, (0, SAMPLES_PER_SEGMENT - len(seg)))
            seg = seg.astype(np.float32)
            seg /= (np.max(np.abs(seg)) + 1e-8)
            segs.append(seg)
    except Exception as e:
        print(f"  Error inferencia {file_path}: {e}"); return None
    if not segs: return None
    probs = []
    for b in range(0, len(segs), BATCH_SIZE):
        batch = np.stack(segs[b:b + BATCH_SIZE])
        bt = torch.from_numpy(batch).unsqueeze(1).to(DEVICE)
        out = model(bt).cpu().numpy()
        probs.extend(out.tolist())
    return float(np.mean(probs))

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
global_y_true, global_y_pred, global_y_prob = [], [], []

print(f"\nINICIANDO CROSS-VALIDATION {MODEL_NAME} | {SEGMENT_DURATION}s | {N_FOLDS} FOLDS\n")

for fold_idx, (tr_idx, te_idx) in enumerate(skf.split(all_files, all_labels)):
    print(f"\n{'─'*55}\n  FOLD {fold_idx+1}/{N_FOLDS}\n{'─'*55}")

    Xtr_f, Xte_f = all_files[tr_idx], all_files[te_idx]
    ytr_f, yte_f = all_labels[tr_idx], all_labels[te_idx]
    Xtr_f, ytr_f = oversample_files(Xtr_f, ytr_f)

    train_ds = WaveformDataset(Xtr_f, ytr_f)
    train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=8, pin_memory=True, persistent_workers=True)
    print(f"  Segmentos train: {len(train_ds)}")

    weights  = class_weight.compute_class_weight('balanced',
                   classes=np.unique(ytr_f), y=ytr_f)
    cw_tensor = torch.tensor(weights, dtype=torch.float32).to(DEVICE)

    model     = build_model().to(DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=1e-4)
    criterion = nn.BCELoss(reduction='none')
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=3, min_lr=1e-7)

    if fold_idx == 0:
        n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
        print(f"  Parámetros entrenables: {n_params:,}\n")

    best_loss, patience_cnt, PATIENCE, best_state = float('inf'), 0, 5, None
    for epoch in range(EPOCHS):
        loss = train_one_epoch(model, train_dl, optimizer, criterion, cw_tensor)
        scheduler.step(loss)
        if loss < best_loss:
            best_loss = loss; patience_cnt = 0
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            mark = "✓"
        else:
            patience_cnt += 1
            mark = f"({patience_cnt}/{PATIENCE})"
        print(f"  Epoch {epoch+1:02d}/{EPOCHS} — loss: {loss:.4f}  best: {best_loss:.4f}  {mark}")
        if patience_cnt >= PATIENCE:
            print("  ► Early stopping activado."); break

    model.load_state_dict(best_state)

    for i, fp in enumerate(Xte_f):
        avg_prob = predict_file(model, fp)
        if avg_prob is None: continue
        global_y_true.append(int(yte_f[i]))
        global_y_pred.append(1 if avg_prob > 0.33 else 0)
        global_y_prob.append(avg_prob)

    del model, train_ds, train_dl, best_state, cw_tensor
    gc.collect(); torch.cuda.empty_cache()

print("\n" + "="*60)
print(f"RESULTADOS {MODEL_NAME} | {SEGMENT_DURATION}s  ({len(global_y_true)} pacientes)")
print("="*60)

cm = confusion_matrix(global_y_true, global_y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens',
            xticklabels=['Sin Ansiedad', 'Con Ansiedad'],
            yticklabels=['Sin Ansiedad', 'Con Ansiedad'])
plt.title(f'Confusión {MODEL_NAME} {SEGMENT_DURATION}s ({N_FOLDS}-Fold CV)')
plt.ylabel('Real'); plt.xlabel('Predicción')
plt.tight_layout()
plt.savefig(f'confusion_{MODEL_NAME.replace(" ","_")}_{SEGMENT_DURATION}s.png', dpi=150)
plt.show()

acc      = accuracy_score(global_y_true, global_y_pred)
prec     = precision_score(global_y_true, global_y_pred, zero_division=0)
rec      = recall_score(global_y_true, global_y_pred, zero_division=0)
f1       = f1_score(global_y_true, global_y_pred, zero_division=0)
f1_macro = f1_score(global_y_true, global_y_pred, average='macro', zero_division=0)
print(f"\nAccuracy :  {acc:.4f}")
print(f"Precision:  {prec:.4f}")
print(f"Recall   :  {rec:.4f}")
print(f"F1-Ans   :  {f1:.4f}  (clase positiva)")
print(f"F1-Macro :  {f1_macro:.4f}")
print()
print(classification_report(global_y_true, global_y_pred,
                             target_names=['Sin Ansiedad', 'Con Ansiedad']))

fpr, tpr, _ = roc_curve(global_y_true, global_y_prob)
roc_auc = auc(fpr, tpr)
plt.figure(figsize=(7, 6))
plt.plot(fpr, tpr, lw=2, label=f'AUC = {roc_auc:.2f}')
plt.plot([0,1],[0,1],'navy',lw=2,linestyle='--')
plt.xlabel('Falsos Positivos'); plt.ylabel('Verdaderos Positivos')
plt.title(f'ROC — {MODEL_NAME} {SEGMENT_DURATION}s')
plt.legend(loc='lower right'); plt.tight_layout()
plt.savefig(f'roc_{MODEL_NAME.replace(" ","_")}_{SEGMENT_DURATION}s.png', dpi=150)
plt.show()


## 2 s | Overlap 50%

In [ ]:
import os
import gc
import numpy as np
import librosa
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (confusion_matrix, classification_report,
                             accuracy_score, precision_score, recall_score,
                             f1_score, roc_curve, auc)
from sklearn.utils import class_weight

# ══════════════════════════════════════════════════════════

BASE_PATH_0 = "../Clasificación Final/Ansiedad/wav/0"
BASE_PATH_1 = "../Clasificación Final/Ansiedad/wav/1"

SAMPLE_RATE         = 16000
SEGMENT_DURATION    = 2
SAMPLES_PER_SEGMENT = int(SEGMENT_DURATION * SAMPLE_RATE)  # auto-calculado
HOP_SIZE            = SAMPLES_PER_SEGMENT // 2  # 50% overlap → 22050
N_FOLDS             = 5
BATCH_SIZE          = 128
EPOCHS              = 25

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Usando dispositivo: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
    torch.backends.cudnn.benchmark = True

def get_file_lists():
    files_0 = [os.path.join(BASE_PATH_0, f)
               for f in sorted(os.listdir(BASE_PATH_0)) if f.endswith('.wav')]
    files_1 = [os.path.join(BASE_PATH_1, f)
               for f in sorted(os.listdir(BASE_PATH_1)) if f.endswith('.wav')]
    files  = np.array(files_0 + files_1)
    labels = np.array([0]*len(files_0) + [1]*len(files_1), dtype=np.int64)
    return files, labels

all_files, all_labels = get_file_lists()
print(f"\nTotal archivos : {len(all_files)}")
print(f" - Sin Ansiedad (0): {np.sum(all_labels == 0)}")
print(f" - Con Ansiedad (1): {np.sum(all_labels == 1)}")
print(f"Segment: {SEGMENT_DURATION}s | Samples/seg: {SAMPLES_PER_SEGMENT}")

class WaveformDataset(Dataset):
    """Lazy loading — overlap 50%%."""
    def __init__(self, file_list, label_list):
        self.samples = []
        for fp, lbl in zip(file_list, label_list):
            try:
                total = librosa.get_duration(path=fp)
                n_segs = max(1, int((total * SAMPLE_RATE - SAMPLES_PER_SEGMENT) / HOP_SIZE) + 1)
                for i in range(n_segs):
                    offset = i * HOP_SIZE / SAMPLE_RATE
                    self.samples.append((fp, offset, int(lbl)))
            except Exception as e:
                print(f"[AVISO] {fp}: {e}")

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        fp, offset, label = self.samples[idx]
        audio, _ = librosa.load(fp, sr=SAMPLE_RATE, offset=offset,
                                duration=SEGMENT_DURATION)
        if len(audio) < SAMPLES_PER_SEGMENT:
            audio = np.pad(audio, (0, SAMPLES_PER_SEGMENT - len(audio)))
        audio = audio[:SAMPLES_PER_SEGMENT].astype(np.float32)
        audio /= (np.max(np.abs(audio)) + 1e-8)
        return torch.from_numpy(audio).unsqueeze(0), torch.tensor(label, dtype=torch.long)

def oversample_files(files, labels):
    f1 = files[labels == 1]; l1 = labels[labels == 1]
    diff = np.sum(labels == 0) - np.sum(labels == 1)
    if diff > 0:
        idx    = np.random.choice(len(f1), size=diff, replace=True)
        files  = np.concatenate([files,  f1[idx]])
        labels = np.concatenate([labels, l1[idx]])
    return files, labels

MODEL_NAME = "CRNN"

class CRNN(nn.Module):
    """Convolutional + Recurrent NN.
    CNN extrae features locales; BiGRU captura dependencias temporales."""

    def __init__(self, input_length):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv1d(1, 64, 80, stride=40, padding=40),
            nn.BatchNorm1d(64), nn.ReLU(inplace=True),
            nn.MaxPool1d(4), nn.Dropout(0.2),

            nn.Conv1d(64,  128, 16, padding=8),
            nn.BatchNorm1d(128), nn.ReLU(inplace=True),
            nn.MaxPool1d(2), nn.Dropout(0.25),

            nn.Conv1d(128, 256, 8, padding=4),
            nn.BatchNorm1d(256), nn.ReLU(inplace=True), nn.Dropout(0.3),

            nn.Conv1d(256, 256, 4, padding=2),
            nn.BatchNorm1d(256), nn.ReLU(inplace=True), nn.Dropout(0.3),
        )
        # BiGRU: output = hidden*2 por bidireccional
        self.bigru1 = nn.GRU(256, 128, batch_first=True, bidirectional=True)
        self.bn_gru  = nn.BatchNorm1d(256)
        self.drop_g1 = nn.Dropout(0.35)
        self.bigru2 = nn.GRU(256, 64, batch_first=True, bidirectional=True)
        self.drop_g2 = nn.Dropout(0.3)
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.classifier = nn.Sequential(
            nn.Linear(128, 64), nn.ReLU(inplace=True), nn.Dropout(0.5),
            nn.Linear(64,  32), nn.ReLU(inplace=True), nn.Dropout(0.4),
            nn.Linear(32,   1), nn.Sigmoid(),
        )

    def forward(self, x):
        x = self.cnn(x)                     # (B, 256, T)
        x = x.permute(0, 2, 1)             # (B, T, 256)
        x, _ = self.bigru1(x)              # (B, T, 256)
        x = self.bn_gru(x.permute(0,2,1)).permute(0,2,1)
        x = self.drop_g1(x)
        x, _ = self.bigru2(x)              # (B, T, 128)
        x = self.drop_g2(x)
        x = self.pool(x.permute(0,2,1)).squeeze(2)  # (B, 128)
        return self.classifier(x).squeeze(1)

def build_model():
    return CRNN(SAMPLES_PER_SEGMENT)

def train_one_epoch(model, loader, optimizer, criterion, cw_tensor):
    model.train()
    total_loss = 0.0
    for xb, yb in loader:
        xb = xb.to(DEVICE, non_blocking=True)
        yb = yb.to(DEVICE, non_blocking=True).float()
        optimizer.zero_grad(set_to_none=True)
        preds = model(xb)
        sample_w = cw_tensor[yb.long()]
        loss = (criterion(preds, yb) * sample_w).mean()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

@torch.no_grad()
def predict_file(model, file_path):
    model.eval()
    segs = []
    try:
        audio, _ = librosa.load(file_path, sr=SAMPLE_RATE)
        hop = HOP_SIZE
        for start in range(0, len(audio), hop):
            seg = audio[start:start + SAMPLES_PER_SEGMENT]
            if len(seg) < SAMPLES_PER_SEGMENT // 2: continue
            if len(seg) < SAMPLES_PER_SEGMENT:
                seg = np.pad(seg, (0, SAMPLES_PER_SEGMENT - len(seg)))
            seg = seg.astype(np.float32)
            seg /= (np.max(np.abs(seg)) + 1e-8)
            segs.append(seg)
    except Exception as e:
        print(f"  Error inferencia {file_path}: {e}"); return None
    if not segs: return None
    probs = []
    for b in range(0, len(segs), BATCH_SIZE):
        batch = np.stack(segs[b:b + BATCH_SIZE])
        bt = torch.from_numpy(batch).unsqueeze(1).to(DEVICE)
        out = model(bt).cpu().numpy()
        probs.extend(out.tolist())
    return float(np.mean(probs))

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
global_y_true, global_y_pred, global_y_prob = [], [], []

print(f"\nINICIANDO CROSS-VALIDATION {MODEL_NAME} | {SEGMENT_DURATION}s | {N_FOLDS} FOLDS\n")

for fold_idx, (tr_idx, te_idx) in enumerate(skf.split(all_files, all_labels)):
    print(f"\n{'─'*55}\n  FOLD {fold_idx+1}/{N_FOLDS}\n{'─'*55}")

    Xtr_f, Xte_f = all_files[tr_idx], all_files[te_idx]
    ytr_f, yte_f = all_labels[tr_idx], all_labels[te_idx]
    Xtr_f, ytr_f = oversample_files(Xtr_f, ytr_f)

    train_ds = WaveformDataset(Xtr_f, ytr_f)
    train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=8, pin_memory=True, persistent_workers=True)
    print(f"  Segmentos train: {len(train_ds)}")

    weights  = class_weight.compute_class_weight('balanced',
                   classes=np.unique(ytr_f), y=ytr_f)
    cw_tensor = torch.tensor(weights, dtype=torch.float32).to(DEVICE)

    model     = build_model().to(DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=1e-4)
    criterion = nn.BCELoss(reduction='none')
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=3, min_lr=1e-7)

    if fold_idx == 0:
        n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
        print(f"  Parámetros entrenables: {n_params:,}\n")

    best_loss, patience_cnt, PATIENCE, best_state = float('inf'), 0, 5, None
    for epoch in range(EPOCHS):
        loss = train_one_epoch(model, train_dl, optimizer, criterion, cw_tensor)
        scheduler.step(loss)
        if loss < best_loss:
            best_loss = loss; patience_cnt = 0
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            mark = "✓"
        else:
            patience_cnt += 1
            mark = f"({patience_cnt}/{PATIENCE})"
        print(f"  Epoch {epoch+1:02d}/{EPOCHS} — loss: {loss:.4f}  best: {best_loss:.4f}  {mark}")
        if patience_cnt >= PATIENCE:
            print("  ► Early stopping activado."); break

    model.load_state_dict(best_state)

    for i, fp in enumerate(Xte_f):
        avg_prob = predict_file(model, fp)
        if avg_prob is None: continue
        global_y_true.append(int(yte_f[i]))
        global_y_pred.append(1 if avg_prob > 0.33 else 0)
        global_y_prob.append(avg_prob)

    del model, train_ds, train_dl, best_state, cw_tensor
    gc.collect(); torch.cuda.empty_cache()

print("\n" + "="*60)
print(f"RESULTADOS {MODEL_NAME} | {SEGMENT_DURATION}s  ({len(global_y_true)} pacientes)")
print("="*60)

cm = confusion_matrix(global_y_true, global_y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens',
            xticklabels=['Sin Ansiedad', 'Con Ansiedad'],
            yticklabels=['Sin Ansiedad', 'Con Ansiedad'])
plt.title(f'Confusión {MODEL_NAME} {SEGMENT_DURATION}s ({N_FOLDS}-Fold CV)')
plt.ylabel('Real'); plt.xlabel('Predicción')
plt.tight_layout()
plt.savefig(f'confusion_{MODEL_NAME.replace(" ","_")}_{SEGMENT_DURATION}s.png', dpi=150)
plt.show()

acc      = accuracy_score(global_y_true, global_y_pred)
prec     = precision_score(global_y_true, global_y_pred, zero_division=0)
rec      = recall_score(global_y_true, global_y_pred, zero_division=0)
f1       = f1_score(global_y_true, global_y_pred, zero_division=0)
f1_macro = f1_score(global_y_true, global_y_pred, average='macro', zero_division=0)
print(f"\nAccuracy :  {acc:.4f}")
print(f"Precision:  {prec:.4f}")
print(f"Recall   :  {rec:.4f}")
print(f"F1-Ans   :  {f1:.4f}  (clase positiva)")
print(f"F1-Macro :  {f1_macro:.4f}")
print()
print(classification_report(global_y_true, global_y_pred,
                             target_names=['Sin Ansiedad', 'Con Ansiedad']))

fpr, tpr, _ = roc_curve(global_y_true, global_y_prob)
roc_auc = auc(fpr, tpr)
plt.figure(figsize=(7, 6))
plt.plot(fpr, tpr, lw=2, label=f'AUC = {roc_auc:.2f}')
plt.plot([0,1],[0,1],'navy',lw=2,linestyle='--')
plt.xlabel('Falsos Positivos'); plt.ylabel('Verdaderos Positivos')
plt.title(f'ROC — {MODEL_NAME} {SEGMENT_DURATION}s')
plt.legend(loc='lower right'); plt.tight_layout()
plt.savefig(f'roc_{MODEL_NAME.replace(" ","_")}_{SEGMENT_DURATION}s.png', dpi=150)
plt.show()


## 3 s | Overlap 50%

In [ ]:
import os
import gc
import numpy as np
import librosa
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (confusion_matrix, classification_report,
                             accuracy_score, precision_score, recall_score,
                             f1_score, roc_curve, auc)
from sklearn.utils import class_weight

# ══════════════════════════════════════════════════════════

BASE_PATH_0 = "../Clasificación Final/Ansiedad/wav/0"
BASE_PATH_1 = "../Clasificación Final/Ansiedad/wav/1"

SAMPLE_RATE         = 16000
SEGMENT_DURATION    = 3
SAMPLES_PER_SEGMENT = int(SEGMENT_DURATION * SAMPLE_RATE)  # auto-calculado
HOP_SIZE            = SAMPLES_PER_SEGMENT // 2  # 50% overlap → 33075
N_FOLDS             = 5
BATCH_SIZE          = 128
EPOCHS              = 25

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Usando dispositivo: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
    torch.backends.cudnn.benchmark = True

def get_file_lists():
    files_0 = [os.path.join(BASE_PATH_0, f)
               for f in sorted(os.listdir(BASE_PATH_0)) if f.endswith('.wav')]
    files_1 = [os.path.join(BASE_PATH_1, f)
               for f in sorted(os.listdir(BASE_PATH_1)) if f.endswith('.wav')]
    files  = np.array(files_0 + files_1)
    labels = np.array([0]*len(files_0) + [1]*len(files_1), dtype=np.int64)
    return files, labels

all_files, all_labels = get_file_lists()
print(f"\nTotal archivos : {len(all_files)}")
print(f" - Sin Ansiedad (0): {np.sum(all_labels == 0)}")
print(f" - Con Ansiedad (1): {np.sum(all_labels == 1)}")
print(f"Segment: {SEGMENT_DURATION}s | Samples/seg: {SAMPLES_PER_SEGMENT}")

class WaveformDataset(Dataset):
    """Lazy loading — overlap 50%%."""
    def __init__(self, file_list, label_list):
        self.samples = []
        for fp, lbl in zip(file_list, label_list):
            try:
                total = librosa.get_duration(path=fp)
                n_segs = max(1, int((total * SAMPLE_RATE - SAMPLES_PER_SEGMENT) / HOP_SIZE) + 1)
                for i in range(n_segs):
                    offset = i * HOP_SIZE / SAMPLE_RATE
                    self.samples.append((fp, offset, int(lbl)))
            except Exception as e:
                print(f"[AVISO] {fp}: {e}")

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        fp, offset, label = self.samples[idx]
        audio, _ = librosa.load(fp, sr=SAMPLE_RATE, offset=offset,
                                duration=SEGMENT_DURATION)
        if len(audio) < SAMPLES_PER_SEGMENT:
            audio = np.pad(audio, (0, SAMPLES_PER_SEGMENT - len(audio)))
        audio = audio[:SAMPLES_PER_SEGMENT].astype(np.float32)
        audio /= (np.max(np.abs(audio)) + 1e-8)
        return torch.from_numpy(audio).unsqueeze(0), torch.tensor(label, dtype=torch.long)

def oversample_files(files, labels):
    f1 = files[labels == 1]; l1 = labels[labels == 1]
    diff = np.sum(labels == 0) - np.sum(labels == 1)
    if diff > 0:
        idx    = np.random.choice(len(f1), size=diff, replace=True)
        files  = np.concatenate([files,  f1[idx]])
        labels = np.concatenate([labels, l1[idx]])
    return files, labels

MODEL_NAME = "CRNN"

class CRNN(nn.Module):
    """Convolutional + Recurrent NN.
    CNN extrae features locales; BiGRU captura dependencias temporales."""

    def __init__(self, input_length):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv1d(1, 64, 80, stride=40, padding=40),
            nn.BatchNorm1d(64), nn.ReLU(inplace=True),
            nn.MaxPool1d(4), nn.Dropout(0.2),

            nn.Conv1d(64,  128, 16, padding=8),
            nn.BatchNorm1d(128), nn.ReLU(inplace=True),
            nn.MaxPool1d(2), nn.Dropout(0.25),

            nn.Conv1d(128, 256, 8, padding=4),
            nn.BatchNorm1d(256), nn.ReLU(inplace=True), nn.Dropout(0.3),

            nn.Conv1d(256, 256, 4, padding=2),
            nn.BatchNorm1d(256), nn.ReLU(inplace=True), nn.Dropout(0.3),
        )
        # BiGRU: output = hidden*2 por bidireccional
        self.bigru1 = nn.GRU(256, 128, batch_first=True, bidirectional=True)
        self.bn_gru  = nn.BatchNorm1d(256)
        self.drop_g1 = nn.Dropout(0.35)
        self.bigru2 = nn.GRU(256, 64, batch_first=True, bidirectional=True)
        self.drop_g2 = nn.Dropout(0.3)
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.classifier = nn.Sequential(
            nn.Linear(128, 64), nn.ReLU(inplace=True), nn.Dropout(0.5),
            nn.Linear(64,  32), nn.ReLU(inplace=True), nn.Dropout(0.4),
            nn.Linear(32,   1), nn.Sigmoid(),
        )

    def forward(self, x):
        x = self.cnn(x)                     # (B, 256, T)
        x = x.permute(0, 2, 1)             # (B, T, 256)
        x, _ = self.bigru1(x)              # (B, T, 256)
        x = self.bn_gru(x.permute(0,2,1)).permute(0,2,1)
        x = self.drop_g1(x)
        x, _ = self.bigru2(x)              # (B, T, 128)
        x = self.drop_g2(x)
        x = self.pool(x.permute(0,2,1)).squeeze(2)  # (B, 128)
        return self.classifier(x).squeeze(1)

def build_model():
    return CRNN(SAMPLES_PER_SEGMENT)

def train_one_epoch(model, loader, optimizer, criterion, cw_tensor):
    model.train()
    total_loss = 0.0
    for xb, yb in loader:
        xb = xb.to(DEVICE, non_blocking=True)
        yb = yb.to(DEVICE, non_blocking=True).float()
        optimizer.zero_grad(set_to_none=True)
        preds = model(xb)
        sample_w = cw_tensor[yb.long()]
        loss = (criterion(preds, yb) * sample_w).mean()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

@torch.no_grad()
def predict_file(model, file_path):
    model.eval()
    segs = []
    try:
        audio, _ = librosa.load(file_path, sr=SAMPLE_RATE)
        hop = HOP_SIZE
        for start in range(0, len(audio), hop):
            seg = audio[start:start + SAMPLES_PER_SEGMENT]
            if len(seg) < SAMPLES_PER_SEGMENT // 2: continue
            if len(seg) < SAMPLES_PER_SEGMENT:
                seg = np.pad(seg, (0, SAMPLES_PER_SEGMENT - len(seg)))
            seg = seg.astype(np.float32)
            seg /= (np.max(np.abs(seg)) + 1e-8)
            segs.append(seg)
    except Exception as e:
        print(f"  Error inferencia {file_path}: {e}"); return None
    if not segs: return None
    probs = []
    for b in range(0, len(segs), BATCH_SIZE):
        batch = np.stack(segs[b:b + BATCH_SIZE])
        bt = torch.from_numpy(batch).unsqueeze(1).to(DEVICE)
        out = model(bt).cpu().numpy()
        probs.extend(out.tolist())
    return float(np.mean(probs))

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
global_y_true, global_y_pred, global_y_prob = [], [], []

print(f"\nINICIANDO CROSS-VALIDATION {MODEL_NAME} | {SEGMENT_DURATION}s | {N_FOLDS} FOLDS\n")

for fold_idx, (tr_idx, te_idx) in enumerate(skf.split(all_files, all_labels)):
    print(f"\n{'─'*55}\n  FOLD {fold_idx+1}/{N_FOLDS}\n{'─'*55}")

    Xtr_f, Xte_f = all_files[tr_idx], all_files[te_idx]
    ytr_f, yte_f = all_labels[tr_idx], all_labels[te_idx]
    Xtr_f, ytr_f = oversample_files(Xtr_f, ytr_f)

    train_ds = WaveformDataset(Xtr_f, ytr_f)
    train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=8, pin_memory=True, persistent_workers=True)
    print(f"  Segmentos train: {len(train_ds)}")

    weights  = class_weight.compute_class_weight('balanced',
                   classes=np.unique(ytr_f), y=ytr_f)
    cw_tensor = torch.tensor(weights, dtype=torch.float32).to(DEVICE)

    model     = build_model().to(DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=1e-4)
    criterion = nn.BCELoss(reduction='none')
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=3, min_lr=1e-7)

    if fold_idx == 0:
        n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
        print(f"  Parámetros entrenables: {n_params:,}\n")

    best_loss, patience_cnt, PATIENCE, best_state = float('inf'), 0, 5, None
    for epoch in range(EPOCHS):
        loss = train_one_epoch(model, train_dl, optimizer, criterion, cw_tensor)
        scheduler.step(loss)
        if loss < best_loss:
            best_loss = loss; patience_cnt = 0
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            mark = "✓"
        else:
            patience_cnt += 1
            mark = f"({patience_cnt}/{PATIENCE})"
        print(f"  Epoch {epoch+1:02d}/{EPOCHS} — loss: {loss:.4f}  best: {best_loss:.4f}  {mark}")
        if patience_cnt >= PATIENCE:
            print("  ► Early stopping activado."); break

    model.load_state_dict(best_state)

    for i, fp in enumerate(Xte_f):
        avg_prob = predict_file(model, fp)
        if avg_prob is None: continue
        global_y_true.append(int(yte_f[i]))
        global_y_pred.append(1 if avg_prob > 0.33 else 0)
        global_y_prob.append(avg_prob)

    del model, train_ds, train_dl, best_state, cw_tensor
    gc.collect(); torch.cuda.empty_cache()

print("\n" + "="*60)
print(f"RESULTADOS {MODEL_NAME} | {SEGMENT_DURATION}s  ({len(global_y_true)} pacientes)")
print("="*60)

cm = confusion_matrix(global_y_true, global_y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens',
            xticklabels=['Sin Ansiedad', 'Con Ansiedad'],
            yticklabels=['Sin Ansiedad', 'Con Ansiedad'])
plt.title(f'Confusión {MODEL_NAME} {SEGMENT_DURATION}s ({N_FOLDS}-Fold CV)')
plt.ylabel('Real'); plt.xlabel('Predicción')
plt.tight_layout()
plt.savefig(f'confusion_{MODEL_NAME.replace(" ","_")}_{SEGMENT_DURATION}s.png', dpi=150)
plt.show()

acc      = accuracy_score(global_y_true, global_y_pred)
prec     = precision_score(global_y_true, global_y_pred, zero_division=0)
rec      = recall_score(global_y_true, global_y_pred, zero_division=0)
f1       = f1_score(global_y_true, global_y_pred, zero_division=0)
f1_macro = f1_score(global_y_true, global_y_pred, average='macro', zero_division=0)
print(f"\nAccuracy :  {acc:.4f}")
print(f"Precision:  {prec:.4f}")
print(f"Recall   :  {rec:.4f}")
print(f"F1-Ans   :  {f1:.4f}  (clase positiva)")
print(f"F1-Macro :  {f1_macro:.4f}")
print()
print(classification_report(global_y_true, global_y_pred,
                             target_names=['Sin Ansiedad', 'Con Ansiedad']))

fpr, tpr, _ = roc_curve(global_y_true, global_y_prob)
roc_auc = auc(fpr, tpr)
plt.figure(figsize=(7, 6))
plt.plot(fpr, tpr, lw=2, label=f'AUC = {roc_auc:.2f}')
plt.plot([0,1],[0,1],'navy',lw=2,linestyle='--')
plt.xlabel('Falsos Positivos'); plt.ylabel('Verdaderos Positivos')
plt.title(f'ROC — {MODEL_NAME} {SEGMENT_DURATION}s')
plt.legend(loc='lower right'); plt.tight_layout()
plt.savefig(f'roc_{MODEL_NAME.replace(" ","_")}_{SEGMENT_DURATION}s.png', dpi=150)
plt.show()


## 4 s | Overlap 50%

In [ ]:
import os
import gc
import numpy as np
import librosa
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (confusion_matrix, classification_report,
                             accuracy_score, precision_score, recall_score,
                             f1_score, roc_curve, auc)
from sklearn.utils import class_weight

# ══════════════════════════════════════════════════════════

BASE_PATH_0 = "../Clasificación Final/Ansiedad/wav/0"
BASE_PATH_1 = "../Clasificación Final/Ansiedad/wav/1"

SAMPLE_RATE         = 16000
SEGMENT_DURATION    = 4
SAMPLES_PER_SEGMENT = int(SEGMENT_DURATION * SAMPLE_RATE)  # auto-calculado
HOP_SIZE            = SAMPLES_PER_SEGMENT // 2  # 50% overlap → 44100
N_FOLDS             = 5
BATCH_SIZE          = 128
EPOCHS              = 25

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Usando dispositivo: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
    torch.backends.cudnn.benchmark = True

def get_file_lists():
    files_0 = [os.path.join(BASE_PATH_0, f)
               for f in sorted(os.listdir(BASE_PATH_0)) if f.endswith('.wav')]
    files_1 = [os.path.join(BASE_PATH_1, f)
               for f in sorted(os.listdir(BASE_PATH_1)) if f.endswith('.wav')]
    files  = np.array(files_0 + files_1)
    labels = np.array([0]*len(files_0) + [1]*len(files_1), dtype=np.int64)
    return files, labels

all_files, all_labels = get_file_lists()
print(f"\nTotal archivos : {len(all_files)}")
print(f" - Sin Ansiedad (0): {np.sum(all_labels == 0)}")
print(f" - Con Ansiedad (1): {np.sum(all_labels == 1)}")
print(f"Segment: {SEGMENT_DURATION}s | Samples/seg: {SAMPLES_PER_SEGMENT}")

class WaveformDataset(Dataset):
    """Lazy loading — overlap 50%%."""
    def __init__(self, file_list, label_list):
        self.samples = []
        for fp, lbl in zip(file_list, label_list):
            try:
                total = librosa.get_duration(path=fp)
                n_segs = max(1, int((total * SAMPLE_RATE - SAMPLES_PER_SEGMENT) / HOP_SIZE) + 1)
                for i in range(n_segs):
                    offset = i * HOP_SIZE / SAMPLE_RATE
                    self.samples.append((fp, offset, int(lbl)))
            except Exception as e:
                print(f"[AVISO] {fp}: {e}")

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        fp, offset, label = self.samples[idx]
        audio, _ = librosa.load(fp, sr=SAMPLE_RATE, offset=offset,
                                duration=SEGMENT_DURATION)
        if len(audio) < SAMPLES_PER_SEGMENT:
            audio = np.pad(audio, (0, SAMPLES_PER_SEGMENT - len(audio)))
        audio = audio[:SAMPLES_PER_SEGMENT].astype(np.float32)
        audio /= (np.max(np.abs(audio)) + 1e-8)
        return torch.from_numpy(audio).unsqueeze(0), torch.tensor(label, dtype=torch.long)

def oversample_files(files, labels):
    f1 = files[labels == 1]; l1 = labels[labels == 1]
    diff = np.sum(labels == 0) - np.sum(labels == 1)
    if diff > 0:
        idx    = np.random.choice(len(f1), size=diff, replace=True)
        files  = np.concatenate([files,  f1[idx]])
        labels = np.concatenate([labels, l1[idx]])
    return files, labels

MODEL_NAME = "CRNN"

class CRNN(nn.Module):
    """Convolutional + Recurrent NN.
    CNN extrae features locales; BiGRU captura dependencias temporales."""

    def __init__(self, input_length):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv1d(1, 64, 80, stride=40, padding=40),
            nn.BatchNorm1d(64), nn.ReLU(inplace=True),
            nn.MaxPool1d(4), nn.Dropout(0.2),

            nn.Conv1d(64,  128, 16, padding=8),
            nn.BatchNorm1d(128), nn.ReLU(inplace=True),
            nn.MaxPool1d(2), nn.Dropout(0.25),

            nn.Conv1d(128, 256, 8, padding=4),
            nn.BatchNorm1d(256), nn.ReLU(inplace=True), nn.Dropout(0.3),

            nn.Conv1d(256, 256, 4, padding=2),
            nn.BatchNorm1d(256), nn.ReLU(inplace=True), nn.Dropout(0.3),
        )
        # BiGRU: output = hidden*2 por bidireccional
        self.bigru1 = nn.GRU(256, 128, batch_first=True, bidirectional=True)
        self.bn_gru  = nn.BatchNorm1d(256)
        self.drop_g1 = nn.Dropout(0.35)
        self.bigru2 = nn.GRU(256, 64, batch_first=True, bidirectional=True)
        self.drop_g2 = nn.Dropout(0.3)
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.classifier = nn.Sequential(
            nn.Linear(128, 64), nn.ReLU(inplace=True), nn.Dropout(0.5),
            nn.Linear(64,  32), nn.ReLU(inplace=True), nn.Dropout(0.4),
            nn.Linear(32,   1), nn.Sigmoid(),
        )

    def forward(self, x):
        x = self.cnn(x)                     # (B, 256, T)
        x = x.permute(0, 2, 1)             # (B, T, 256)
        x, _ = self.bigru1(x)              # (B, T, 256)
        x = self.bn_gru(x.permute(0,2,1)).permute(0,2,1)
        x = self.drop_g1(x)
        x, _ = self.bigru2(x)              # (B, T, 128)
        x = self.drop_g2(x)
        x = self.pool(x.permute(0,2,1)).squeeze(2)  # (B, 128)
        return self.classifier(x).squeeze(1)

def build_model():
    return CRNN(SAMPLES_PER_SEGMENT)

def train_one_epoch(model, loader, optimizer, criterion, cw_tensor):
    model.train()
    total_loss = 0.0
    for xb, yb in loader:
        xb = xb.to(DEVICE, non_blocking=True)
        yb = yb.to(DEVICE, non_blocking=True).float()
        optimizer.zero_grad(set_to_none=True)
        preds = model(xb)
        sample_w = cw_tensor[yb.long()]
        loss = (criterion(preds, yb) * sample_w).mean()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

@torch.no_grad()
def predict_file(model, file_path):
    model.eval()
    segs = []
    try:
        audio, _ = librosa.load(file_path, sr=SAMPLE_RATE)
        hop = HOP_SIZE
        for start in range(0, len(audio), hop):
            seg = audio[start:start + SAMPLES_PER_SEGMENT]
            if len(seg) < SAMPLES_PER_SEGMENT // 2: continue
            if len(seg) < SAMPLES_PER_SEGMENT:
                seg = np.pad(seg, (0, SAMPLES_PER_SEGMENT - len(seg)))
            seg = seg.astype(np.float32)
            seg /= (np.max(np.abs(seg)) + 1e-8)
            segs.append(seg)
    except Exception as e:
        print(f"  Error inferencia {file_path}: {e}"); return None
    if not segs: return None
    probs = []
    for b in range(0, len(segs), BATCH_SIZE):
        batch = np.stack(segs[b:b + BATCH_SIZE])
        bt = torch.from_numpy(batch).unsqueeze(1).to(DEVICE)
        out = model(bt).cpu().numpy()
        probs.extend(out.tolist())
    return float(np.mean(probs))

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
global_y_true, global_y_pred, global_y_prob = [], [], []

print(f"\nINICIANDO CROSS-VALIDATION {MODEL_NAME} | {SEGMENT_DURATION}s | {N_FOLDS} FOLDS\n")

for fold_idx, (tr_idx, te_idx) in enumerate(skf.split(all_files, all_labels)):
    print(f"\n{'─'*55}\n  FOLD {fold_idx+1}/{N_FOLDS}\n{'─'*55}")

    Xtr_f, Xte_f = all_files[tr_idx], all_files[te_idx]
    ytr_f, yte_f = all_labels[tr_idx], all_labels[te_idx]
    Xtr_f, ytr_f = oversample_files(Xtr_f, ytr_f)

    train_ds = WaveformDataset(Xtr_f, ytr_f)
    train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=8, pin_memory=True, persistent_workers=True)
    print(f"  Segmentos train: {len(train_ds)}")

    weights  = class_weight.compute_class_weight('balanced',
                   classes=np.unique(ytr_f), y=ytr_f)
    cw_tensor = torch.tensor(weights, dtype=torch.float32).to(DEVICE)

    model     = build_model().to(DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=1e-4)
    criterion = nn.BCELoss(reduction='none')
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=3, min_lr=1e-7)

    if fold_idx == 0:
        n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
        print(f"  Parámetros entrenables: {n_params:,}\n")

    best_loss, patience_cnt, PATIENCE, best_state = float('inf'), 0, 5, None
    for epoch in range(EPOCHS):
        loss = train_one_epoch(model, train_dl, optimizer, criterion, cw_tensor)
        scheduler.step(loss)
        if loss < best_loss:
            best_loss = loss; patience_cnt = 0
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            mark = "✓"
        else:
            patience_cnt += 1
            mark = f"({patience_cnt}/{PATIENCE})"
        print(f"  Epoch {epoch+1:02d}/{EPOCHS} — loss: {loss:.4f}  best: {best_loss:.4f}  {mark}")
        if patience_cnt >= PATIENCE:
            print("  ► Early stopping activado."); break

    model.load_state_dict(best_state)

    for i, fp in enumerate(Xte_f):
        avg_prob = predict_file(model, fp)
        if avg_prob is None: continue
        global_y_true.append(int(yte_f[i]))
        global_y_pred.append(1 if avg_prob > 0.33 else 0)
        global_y_prob.append(avg_prob)

    del model, train_ds, train_dl, best_state, cw_tensor
    gc.collect(); torch.cuda.empty_cache()

print("\n" + "="*60)
print(f"RESULTADOS {MODEL_NAME} | {SEGMENT_DURATION}s  ({len(global_y_true)} pacientes)")
print("="*60)

cm = confusion_matrix(global_y_true, global_y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens',
            xticklabels=['Sin Ansiedad', 'Con Ansiedad'],
            yticklabels=['Sin Ansiedad', 'Con Ansiedad'])
plt.title(f'Confusión {MODEL_NAME} {SEGMENT_DURATION}s ({N_FOLDS}-Fold CV)')
plt.ylabel('Real'); plt.xlabel('Predicción')
plt.tight_layout()
plt.savefig(f'confusion_{MODEL_NAME.replace(" ","_")}_{SEGMENT_DURATION}s.png', dpi=150)
plt.show()

acc      = accuracy_score(global_y_true, global_y_pred)
prec     = precision_score(global_y_true, global_y_pred, zero_division=0)
rec      = recall_score(global_y_true, global_y_pred, zero_division=0)
f1       = f1_score(global_y_true, global_y_pred, zero_division=0)
f1_macro = f1_score(global_y_true, global_y_pred, average='macro', zero_division=0)
print(f"\nAccuracy :  {acc:.4f}")
print(f"Precision:  {prec:.4f}")
print(f"Recall   :  {rec:.4f}")
print(f"F1-Ans   :  {f1:.4f}  (clase positiva)")
print(f"F1-Macro :  {f1_macro:.4f}")
print()
print(classification_report(global_y_true, global_y_pred,
                             target_names=['Sin Ansiedad', 'Con Ansiedad']))

fpr, tpr, _ = roc_curve(global_y_true, global_y_prob)
roc_auc = auc(fpr, tpr)
plt.figure(figsize=(7, 6))
plt.plot(fpr, tpr, lw=2, label=f'AUC = {roc_auc:.2f}')
plt.plot([0,1],[0,1],'navy',lw=2,linestyle='--')
plt.xlabel('Falsos Positivos'); plt.ylabel('Verdaderos Positivos')
plt.title(f'ROC — {MODEL_NAME} {SEGMENT_DURATION}s')
plt.legend(loc='lower right'); plt.tight_layout()
plt.savefig(f'roc_{MODEL_NAME.replace(" ","_")}_{SEGMENT_DURATION}s.png', dpi=150)
plt.show()


# LSTM — MFCC + Δ + ΔΔ  (SR=16000)

## 10 s | Overlap 50%

In [ ]:
import os
import gc
import numpy as np
import librosa
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit
from sklearn.metrics import (confusion_matrix, classification_report,
                             accuracy_score, precision_score, recall_score,
                             f1_score, roc_curve, auc)
from sklearn.utils import class_weight

# ══════════════════════════════════════════════════════════

BASE_PATH_0 = "../Clasificación Final/Ansiedad/wav/0"
BASE_PATH_1 = "../Clasificación Final/Ansiedad/wav/1"

SAMPLE_RATE         = 16000
SEGMENT_DURATION    = 10
SAMPLES_PER_SEGMENT = int(SEGMENT_DURATION * SAMPLE_RATE)  # 160000
HOP_SIZE            = SAMPLES_PER_SEGMENT // 2  # 50% overlap → 80000
N_MFCC              = 40
HOP_LENGTH          = 512
N_FFT               = 1024
N_FEATURES          = N_MFCC * 3  # MFCC + Δ + ΔΔ = 120
EXPECTED_FRAMES     = int(np.ceil(SAMPLES_PER_SEGMENT / HOP_LENGTH))  # ≈ 313
N_FOLDS             = 5
BATCH_SIZE          = 128
EPOCHS              = 25

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Usando dispositivo: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
    torch.backends.cudnn.benchmark = True

def get_file_lists():
    files_0 = [os.path.join(BASE_PATH_0, f)
               for f in sorted(os.listdir(BASE_PATH_0)) if f.endswith('.wav')]
    files_1 = [os.path.join(BASE_PATH_1, f)
               for f in sorted(os.listdir(BASE_PATH_1)) if f.endswith('.wav')]
    files  = np.array(files_0 + files_1)
    labels = np.array([0]*len(files_0) + [1]*len(files_1), dtype=np.int64)
    return files, labels

all_files, all_labels = get_file_lists()
print(f"\nTotal archivos : {len(all_files)}")
print(f" - Sin Ansiedad (0): {np.sum(all_labels == 0)}")
print(f" - Con Ansiedad (1): {np.sum(all_labels == 1)}")
print(f"Segment: {SEGMENT_DURATION}s | Features: ({EXPECTED_FRAMES}, {N_FEATURES})")

class MFCCDataset(Dataset):
    """Extrae MFCC+Δ+ΔΔ on-the-fly. Lazy loading."""
    def __init__(self, file_list, label_list):
        self.samples = []
        for fp, lbl in zip(file_list, label_list):
            try:
                total = librosa.get_duration(path=fp)
                n_segs = max(1, int((total * SAMPLE_RATE - SAMPLES_PER_SEGMENT) / HOP_SIZE) + 1)
                for i in range(n_segs):
                    offset = i * HOP_SIZE / SAMPLE_RATE
                    self.samples.append((fp, offset, int(lbl)))
            except Exception as e:
                print(f"[AVISO] {fp}: {e}")

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        fp, offset, label = self.samples[idx]
        audio, _ = librosa.load(fp, sr=SAMPLE_RATE, offset=offset,
                                duration=SEGMENT_DURATION)
        if len(audio) < SAMPLES_PER_SEGMENT:
            audio = np.pad(audio, (0, SAMPLES_PER_SEGMENT - len(audio)))
        audio = audio[:SAMPLES_PER_SEGMENT]

        mfcc = librosa.feature.mfcc(y=audio, sr=SAMPLE_RATE, n_mfcc=N_MFCC,
                                    n_fft=N_FFT, hop_length=HOP_LENGTH)
        d1   = librosa.feature.delta(mfcc)
        d2   = librosa.feature.delta(mfcc, order=2)
        feat = np.concatenate([mfcc, d1, d2], axis=0).T.astype(np.float32)  # (T, F)
        feat = (feat - feat.mean(0)) / (feat.std(0) + 1e-8)
        if feat.shape[0] < EXPECTED_FRAMES:
            feat = np.pad(feat, ((0, EXPECTED_FRAMES - feat.shape[0]), (0, 0)))
        else:
            feat = feat[:EXPECTED_FRAMES]
        return torch.from_numpy(feat), torch.tensor(label, dtype=torch.long)

def oversample_files(files, labels):
    f1 = files[labels == 1]; l1 = labels[labels == 1]
    diff = np.sum(labels == 0) - np.sum(labels == 1)
    if diff > 0:
        idx    = np.random.choice(len(f1), size=diff, replace=True)
        files  = np.concatenate([files,  f1[idx]])
        labels = np.concatenate([labels, l1[idx]])
    return files, labels

MODEL_NAME = "LSTM-MFCC"

class LSTM_MFCC(nn.Module):
    """LSTM puro sobre MFCC + Δ + ΔΔ.
    Entrada: (batch, EXPECTED_FRAMES, N_FEATURES)."""

    def __init__(self):
        super().__init__()
        self.lstm1 = nn.LSTM(N_FEATURES, 128, batch_first=True)
        self.drop1 = nn.Dropout(0.3)
        self.lstm2 = nn.LSTM(128, 64, batch_first=True)
        self.drop2 = nn.Dropout(0.3)
        self.lstm3 = nn.LSTM(64, 32, batch_first=True)
        self.drop3 = nn.Dropout(0.3)
        self.classifier = nn.Sequential(
            nn.Linear(32, 64), nn.ReLU(inplace=True), nn.Dropout(0.4),
            nn.Linear(64, 32), nn.ReLU(inplace=True), nn.Dropout(0.3),
            nn.Linear(32,  1), nn.Sigmoid(),
        )

    def forward(self, x):
        x, _ = self.lstm1(x); x = self.drop1(x)
        x, _ = self.lstm2(x); x = self.drop2(x)
        x, _ = self.lstm3(x); x = self.drop3(x)
        x = x[:, -1, :]       # último timestep
        return self.classifier(x).squeeze(1)

def build_model():
    return LSTM_MFCC()

def train_one_epoch(model, loader, optimizer, criterion, cw_tensor):
    model.train()
    total_loss = 0.0
    for xb, yb in loader:
        xb = xb.to(DEVICE, non_blocking=True)
        yb = yb.to(DEVICE, non_blocking=True).float()
        optimizer.zero_grad(set_to_none=True)
        preds = model(xb)
        sample_w = cw_tensor[yb.long()]
        loss = (criterion(preds, yb) * sample_w).mean()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

@torch.no_grad()
def predict_file(model, file_path):
    model.eval()
    segs = []
    try:
        audio, _ = librosa.load(file_path, sr=SAMPLE_RATE)
        for start in range(0, len(audio), HOP_SIZE):
            seg = audio[start:start + SAMPLES_PER_SEGMENT]
            if len(seg) < SAMPLES_PER_SEGMENT // 2: continue
            if len(seg) < SAMPLES_PER_SEGMENT:
                seg = np.pad(seg, (0, SAMPLES_PER_SEGMENT - len(seg)))
            seg = seg[:SAMPLES_PER_SEGMENT]
            mfcc = librosa.feature.mfcc(y=seg.astype(np.float32), sr=SAMPLE_RATE,
                                        n_mfcc=N_MFCC, n_fft=N_FFT, hop_length=HOP_LENGTH)
            d1   = librosa.feature.delta(mfcc)
            d2   = librosa.feature.delta(mfcc, order=2)
            feat = np.concatenate([mfcc, d1, d2], axis=0).T.astype(np.float32)
            feat = (feat - feat.mean(0)) / (feat.std(0) + 1e-8)
            if feat.shape[0] < EXPECTED_FRAMES:
                feat = np.pad(feat, ((0, EXPECTED_FRAMES - feat.shape[0]), (0, 0)))
            else:
                feat = feat[:EXPECTED_FRAMES]
            segs.append(feat)
    except Exception as e:
        print(f"  Error {file_path}: {e}"); return None
    if not segs: return None
    probs = []
    for b in range(0, len(segs), BATCH_SIZE):
        batch = np.stack(segs[b:b + BATCH_SIZE])
        bt = torch.from_numpy(batch).to(DEVICE)
        out = model(bt).cpu().numpy()
        probs.extend(out.tolist())
    return float(np.mean(probs))

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
global_y_true, global_y_pred, global_y_prob = [], [], []

print(f"\nINICIANDO CROSS-VALIDATION {MODEL_NAME} | {SEGMENT_DURATION}s | {N_FOLDS} FOLDS\n")

for fold_idx, (tr_idx, te_idx) in enumerate(skf.split(all_files, all_labels)):
    print(f"\n{'─'*55}\n  FOLD {fold_idx+1}/{N_FOLDS}\n{'─'*55}")

    Xtr_f, Xte_f = all_files[tr_idx], all_files[te_idx]
    ytr_f, yte_f = all_labels[tr_idx], all_labels[te_idx]
    Xtr_f, ytr_f = oversample_files(Xtr_f, ytr_f)

    train_ds = MFCCDataset(Xtr_f, ytr_f)
    train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=8, pin_memory=True, persistent_workers=True)
    print(f"  Segmentos train: {len(train_ds)}")

    weights   = class_weight.compute_class_weight('balanced',
                    classes=np.unique(ytr_f), y=ytr_f)
    cw_tensor = torch.tensor(weights, dtype=torch.float32).to(DEVICE)

    model     = build_model().to(DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=1e-4)
    criterion = nn.BCELoss(reduction='none')
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=3, min_lr=1e-7)

    if fold_idx == 0:
        n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
        print(f"  Parámetros entrenables: {n_params:,}\n")

    best_loss, patience_cnt, PATIENCE, best_state = float('inf'), 0, 5, None
    for epoch in range(EPOCHS):
        loss = train_one_epoch(model, train_dl, optimizer, criterion, cw_tensor)
        scheduler.step(loss)
        if loss < best_loss:
            best_loss = loss; patience_cnt = 0
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            mark = "✓"
        else:
            patience_cnt += 1
            mark = f"({patience_cnt}/{PATIENCE})"
        print(f"  Epoch {epoch+1:02d}/{EPOCHS} — loss: {loss:.4f}  best: {best_loss:.4f}  {mark}")
        if patience_cnt >= PATIENCE:
            print("  ► Early stopping activado."); break

    model.load_state_dict(best_state)

    for i, fp in enumerate(Xte_f):
        avg_prob = predict_file(model, fp)
        if avg_prob is None: continue
        global_y_true.append(int(yte_f[i]))
        global_y_pred.append(1 if avg_prob > 0.33 else 0)
        global_y_prob.append(avg_prob)

    del model, train_ds, train_dl, best_state, cw_tensor
    gc.collect(); torch.cuda.empty_cache()

print("\n" + "="*60)
print(f"RESULTADOS {MODEL_NAME} | {SEGMENT_DURATION}s  ({len(global_y_true)} pacientes)")
print("="*60)

cm = confusion_matrix(global_y_true, global_y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Oranges',
            xticklabels=['Sin Ansiedad', 'Con Ansiedad'],
            yticklabels=['Sin Ansiedad', 'Con Ansiedad'])
plt.title(f'Confusión {MODEL_NAME} {SEGMENT_DURATION}s ({N_FOLDS}-Fold CV)')
plt.ylabel('Real'); plt.xlabel('Predicción')
plt.tight_layout()
plt.savefig(f'confusion_{MODEL_NAME.replace(" ","_")}_{SEGMENT_DURATION}s.png', dpi=150)
plt.show()

acc      = accuracy_score(global_y_true, global_y_pred)
prec     = precision_score(global_y_true, global_y_pred, zero_division=0)
rec      = recall_score(global_y_true, global_y_pred, zero_division=0)
f1       = f1_score(global_y_true, global_y_pred, zero_division=0)
f1_macro = f1_score(global_y_true, global_y_pred, average='macro', zero_division=0)
print(f"\nAccuracy :  {acc:.4f}")
print(f"Precision:  {prec:.4f}")
print(f"Recall   :  {rec:.4f}")
print(f"F1-Ans   :  {f1:.4f}  (clase positiva)")
print(f"F1-Macro :  {f1_macro:.4f}")
print()
print(classification_report(global_y_true, global_y_pred,
                             target_names=['Sin Ansiedad', 'Con Ansiedad']))

fpr, tpr, _ = roc_curve(global_y_true, global_y_prob)
roc_auc = auc(fpr, tpr)
plt.figure(figsize=(7, 6))
plt.plot(fpr, tpr, lw=2, label=f'AUC = {roc_auc:.2f}')
plt.plot([0,1],[0,1],'navy',lw=2,linestyle='--')
plt.xlabel('Falsos Positivos'); plt.ylabel('Verdaderos Positivos')
plt.title(f'ROC — {MODEL_NAME} {SEGMENT_DURATION}s')
plt.legend(loc='lower right'); plt.tight_layout()
plt.savefig(f'roc_{MODEL_NAME.replace(" ","_")}_{SEGMENT_DURATION}s.png', dpi=150)
plt.show()


## 2 s | Overlap 50%

In [ ]:
import os
import gc
import numpy as np
import librosa
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit
from sklearn.metrics import (confusion_matrix, classification_report,
                             accuracy_score, precision_score, recall_score,
                             f1_score, roc_curve, auc)
from sklearn.utils import class_weight

# ══════════════════════════════════════════════════════════

BASE_PATH_0 = "../Clasificación Final/Ansiedad/wav/0"
BASE_PATH_1 = "../Clasificación Final/Ansiedad/wav/1"

SAMPLE_RATE         = 16000
SEGMENT_DURATION    = 2
SAMPLES_PER_SEGMENT = int(SEGMENT_DURATION * SAMPLE_RATE)  # 32000
HOP_SIZE            = SAMPLES_PER_SEGMENT // 2  # 50% overlap → 16000
N_MFCC              = 40
HOP_LENGTH          = 512
N_FFT               = 1024
N_FEATURES          = N_MFCC * 3  # MFCC + Δ + ΔΔ = 120
EXPECTED_FRAMES     = int(np.ceil(SAMPLES_PER_SEGMENT / HOP_LENGTH))  # ≈ 63
N_FOLDS             = 5
BATCH_SIZE          = 128
EPOCHS              = 25

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Usando dispositivo: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
    torch.backends.cudnn.benchmark = True

def get_file_lists():
    files_0 = [os.path.join(BASE_PATH_0, f)
               for f in sorted(os.listdir(BASE_PATH_0)) if f.endswith('.wav')]
    files_1 = [os.path.join(BASE_PATH_1, f)
               for f in sorted(os.listdir(BASE_PATH_1)) if f.endswith('.wav')]
    files  = np.array(files_0 + files_1)
    labels = np.array([0]*len(files_0) + [1]*len(files_1), dtype=np.int64)
    return files, labels

all_files, all_labels = get_file_lists()
print(f"\nTotal archivos : {len(all_files)}")
print(f" - Sin Ansiedad (0): {np.sum(all_labels == 0)}")
print(f" - Con Ansiedad (1): {np.sum(all_labels == 1)}")
print(f"Segment: {SEGMENT_DURATION}s | Features: ({EXPECTED_FRAMES}, {N_FEATURES})")

class MFCCDataset(Dataset):
    """Extrae MFCC+Δ+ΔΔ on-the-fly. Lazy loading."""
    def __init__(self, file_list, label_list):
        self.samples = []
        for fp, lbl in zip(file_list, label_list):
            try:
                total = librosa.get_duration(path=fp)
                n_segs = max(1, int((total * SAMPLE_RATE - SAMPLES_PER_SEGMENT) / HOP_SIZE) + 1)
                for i in range(n_segs):
                    offset = i * HOP_SIZE / SAMPLE_RATE
                    self.samples.append((fp, offset, int(lbl)))
            except Exception as e:
                print(f"[AVISO] {fp}: {e}")

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        fp, offset, label = self.samples[idx]
        audio, _ = librosa.load(fp, sr=SAMPLE_RATE, offset=offset,
                                duration=SEGMENT_DURATION)
        if len(audio) < SAMPLES_PER_SEGMENT:
            audio = np.pad(audio, (0, SAMPLES_PER_SEGMENT - len(audio)))
        audio = audio[:SAMPLES_PER_SEGMENT]

        mfcc = librosa.feature.mfcc(y=audio, sr=SAMPLE_RATE, n_mfcc=N_MFCC,
                                    n_fft=N_FFT, hop_length=HOP_LENGTH)
        d1   = librosa.feature.delta(mfcc)
        d2   = librosa.feature.delta(mfcc, order=2)
        feat = np.concatenate([mfcc, d1, d2], axis=0).T.astype(np.float32)  # (T, F)
        feat = (feat - feat.mean(0)) / (feat.std(0) + 1e-8)
        if feat.shape[0] < EXPECTED_FRAMES:
            feat = np.pad(feat, ((0, EXPECTED_FRAMES - feat.shape[0]), (0, 0)))
        else:
            feat = feat[:EXPECTED_FRAMES]
        return torch.from_numpy(feat), torch.tensor(label, dtype=torch.long)

def oversample_files(files, labels):
    f1 = files[labels == 1]; l1 = labels[labels == 1]
    diff = np.sum(labels == 0) - np.sum(labels == 1)
    if diff > 0:
        idx    = np.random.choice(len(f1), size=diff, replace=True)
        files  = np.concatenate([files,  f1[idx]])
        labels = np.concatenate([labels, l1[idx]])
    return files, labels

MODEL_NAME = "LSTM-MFCC"

class LSTM_MFCC(nn.Module):
    """LSTM puro sobre MFCC + Δ + ΔΔ.
    Entrada: (batch, EXPECTED_FRAMES, N_FEATURES)."""

    def __init__(self):
        super().__init__()
        self.lstm1 = nn.LSTM(N_FEATURES, 128, batch_first=True)
        self.drop1 = nn.Dropout(0.3)
        self.lstm2 = nn.LSTM(128, 64, batch_first=True)
        self.drop2 = nn.Dropout(0.3)
        self.lstm3 = nn.LSTM(64, 32, batch_first=True)
        self.drop3 = nn.Dropout(0.3)
        self.classifier = nn.Sequential(
            nn.Linear(32, 64), nn.ReLU(inplace=True), nn.Dropout(0.4),
            nn.Linear(64, 32), nn.ReLU(inplace=True), nn.Dropout(0.3),
            nn.Linear(32,  1), nn.Sigmoid(),
        )

    def forward(self, x):
        x, _ = self.lstm1(x); x = self.drop1(x)
        x, _ = self.lstm2(x); x = self.drop2(x)
        x, _ = self.lstm3(x); x = self.drop3(x)
        x = x[:, -1, :]       # último timestep
        return self.classifier(x).squeeze(1)

def build_model():
    return LSTM_MFCC()

def train_one_epoch(model, loader, optimizer, criterion, cw_tensor):
    model.train()
    total_loss = 0.0
    for xb, yb in loader:
        xb = xb.to(DEVICE, non_blocking=True)
        yb = yb.to(DEVICE, non_blocking=True).float()
        optimizer.zero_grad(set_to_none=True)
        preds = model(xb)
        sample_w = cw_tensor[yb.long()]
        loss = (criterion(preds, yb) * sample_w).mean()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

@torch.no_grad()
def predict_file(model, file_path):
    model.eval()
    segs = []
    try:
        audio, _ = librosa.load(file_path, sr=SAMPLE_RATE)
        for start in range(0, len(audio), HOP_SIZE):
            seg = audio[start:start + SAMPLES_PER_SEGMENT]
            if len(seg) < SAMPLES_PER_SEGMENT // 2: continue
            if len(seg) < SAMPLES_PER_SEGMENT:
                seg = np.pad(seg, (0, SAMPLES_PER_SEGMENT - len(seg)))
            seg = seg[:SAMPLES_PER_SEGMENT]
            mfcc = librosa.feature.mfcc(y=seg.astype(np.float32), sr=SAMPLE_RATE,
                                        n_mfcc=N_MFCC, n_fft=N_FFT, hop_length=HOP_LENGTH)
            d1   = librosa.feature.delta(mfcc)
            d2   = librosa.feature.delta(mfcc, order=2)
            feat = np.concatenate([mfcc, d1, d2], axis=0).T.astype(np.float32)
            feat = (feat - feat.mean(0)) / (feat.std(0) + 1e-8)
            if feat.shape[0] < EXPECTED_FRAMES:
                feat = np.pad(feat, ((0, EXPECTED_FRAMES - feat.shape[0]), (0, 0)))
            else:
                feat = feat[:EXPECTED_FRAMES]
            segs.append(feat)
    except Exception as e:
        print(f"  Error {file_path}: {e}"); return None
    if not segs: return None
    probs = []
    for b in range(0, len(segs), BATCH_SIZE):
        batch = np.stack(segs[b:b + BATCH_SIZE])
        bt = torch.from_numpy(batch).to(DEVICE)
        out = model(bt).cpu().numpy()
        probs.extend(out.tolist())
    return float(np.mean(probs))

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
global_y_true, global_y_pred, global_y_prob = [], [], []

print(f"\nINICIANDO CROSS-VALIDATION {MODEL_NAME} | {SEGMENT_DURATION}s | {N_FOLDS} FOLDS\n")

for fold_idx, (tr_idx, te_idx) in enumerate(skf.split(all_files, all_labels)):
    print(f"\n{'─'*55}\n  FOLD {fold_idx+1}/{N_FOLDS}\n{'─'*55}")

    Xtr_f, Xte_f = all_files[tr_idx], all_files[te_idx]
    ytr_f, yte_f = all_labels[tr_idx], all_labels[te_idx]
    Xtr_f, ytr_f = oversample_files(Xtr_f, ytr_f)

    train_ds = MFCCDataset(Xtr_f, ytr_f)
    train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=8, pin_memory=True, persistent_workers=True)
    print(f"  Segmentos train: {len(train_ds)}")

    weights   = class_weight.compute_class_weight('balanced',
                    classes=np.unique(ytr_f), y=ytr_f)
    cw_tensor = torch.tensor(weights, dtype=torch.float32).to(DEVICE)

    model     = build_model().to(DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=1e-4)
    criterion = nn.BCELoss(reduction='none')
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=3, min_lr=1e-7)

    if fold_idx == 0:
        n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
        print(f"  Parámetros entrenables: {n_params:,}\n")

    best_loss, patience_cnt, PATIENCE, best_state = float('inf'), 0, 5, None
    for epoch in range(EPOCHS):
        loss = train_one_epoch(model, train_dl, optimizer, criterion, cw_tensor)
        scheduler.step(loss)
        if loss < best_loss:
            best_loss = loss; patience_cnt = 0
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            mark = "✓"
        else:
            patience_cnt += 1
            mark = f"({patience_cnt}/{PATIENCE})"
        print(f"  Epoch {epoch+1:02d}/{EPOCHS} — loss: {loss:.4f}  best: {best_loss:.4f}  {mark}")
        if patience_cnt >= PATIENCE:
            print("  ► Early stopping activado."); break

    model.load_state_dict(best_state)

    for i, fp in enumerate(Xte_f):
        avg_prob = predict_file(model, fp)
        if avg_prob is None: continue
        global_y_true.append(int(yte_f[i]))
        global_y_pred.append(1 if avg_prob > 0.33 else 0)
        global_y_prob.append(avg_prob)

    del model, train_ds, train_dl, best_state, cw_tensor
    gc.collect(); torch.cuda.empty_cache()

print("\n" + "="*60)
print(f"RESULTADOS {MODEL_NAME} | {SEGMENT_DURATION}s  ({len(global_y_true)} pacientes)")
print("="*60)

cm = confusion_matrix(global_y_true, global_y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Oranges',
            xticklabels=['Sin Ansiedad', 'Con Ansiedad'],
            yticklabels=['Sin Ansiedad', 'Con Ansiedad'])
plt.title(f'Confusión {MODEL_NAME} {SEGMENT_DURATION}s ({N_FOLDS}-Fold CV)')
plt.ylabel('Real'); plt.xlabel('Predicción')
plt.tight_layout()
plt.savefig(f'confusion_{MODEL_NAME.replace(" ","_")}_{SEGMENT_DURATION}s.png', dpi=150)
plt.show()

acc      = accuracy_score(global_y_true, global_y_pred)
prec     = precision_score(global_y_true, global_y_pred, zero_division=0)
rec      = recall_score(global_y_true, global_y_pred, zero_division=0)
f1       = f1_score(global_y_true, global_y_pred, zero_division=0)
f1_macro = f1_score(global_y_true, global_y_pred, average='macro', zero_division=0)
print(f"\nAccuracy :  {acc:.4f}")
print(f"Precision:  {prec:.4f}")
print(f"Recall   :  {rec:.4f}")
print(f"F1-Ans   :  {f1:.4f}  (clase positiva)")
print(f"F1-Macro :  {f1_macro:.4f}")
print()
print(classification_report(global_y_true, global_y_pred,
                             target_names=['Sin Ansiedad', 'Con Ansiedad']))

fpr, tpr, _ = roc_curve(global_y_true, global_y_prob)
roc_auc = auc(fpr, tpr)
plt.figure(figsize=(7, 6))
plt.plot(fpr, tpr, lw=2, label=f'AUC = {roc_auc:.2f}')
plt.plot([0,1],[0,1],'navy',lw=2,linestyle='--')
plt.xlabel('Falsos Positivos'); plt.ylabel('Verdaderos Positivos')
plt.title(f'ROC — {MODEL_NAME} {SEGMENT_DURATION}s')
plt.legend(loc='lower right'); plt.tight_layout()
plt.savefig(f'roc_{MODEL_NAME.replace(" ","_")}_{SEGMENT_DURATION}s.png', dpi=150)
plt.show()


## 3 s | Overlap 50%

In [ ]:
import os
import gc
import numpy as np
import librosa
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit
from sklearn.metrics import (confusion_matrix, classification_report,
                             accuracy_score, precision_score, recall_score,
                             f1_score, roc_curve, auc)
from sklearn.utils import class_weight

# ══════════════════════════════════════════════════════════

BASE_PATH_0 = "../Clasificación Final/Ansiedad/wav/0"
BASE_PATH_1 = "../Clasificación Final/Ansiedad/wav/1"

SAMPLE_RATE         = 16000
SEGMENT_DURATION    = 3
SAMPLES_PER_SEGMENT = int(SEGMENT_DURATION * SAMPLE_RATE)  # 48000
HOP_SIZE            = SAMPLES_PER_SEGMENT // 2  # 50% overlap → 24000
N_MFCC              = 40
HOP_LENGTH          = 512
N_FFT               = 1024
N_FEATURES          = N_MFCC * 3  # MFCC + Δ + ΔΔ = 120
EXPECTED_FRAMES     = int(np.ceil(SAMPLES_PER_SEGMENT / HOP_LENGTH))  # ≈ 94
N_FOLDS             = 5
BATCH_SIZE          = 128
EPOCHS              = 25

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Usando dispositivo: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
    torch.backends.cudnn.benchmark = True

def get_file_lists():
    files_0 = [os.path.join(BASE_PATH_0, f)
               for f in sorted(os.listdir(BASE_PATH_0)) if f.endswith('.wav')]
    files_1 = [os.path.join(BASE_PATH_1, f)
               for f in sorted(os.listdir(BASE_PATH_1)) if f.endswith('.wav')]
    files  = np.array(files_0 + files_1)
    labels = np.array([0]*len(files_0) + [1]*len(files_1), dtype=np.int64)
    return files, labels

all_files, all_labels = get_file_lists()
print(f"\nTotal archivos : {len(all_files)}")
print(f" - Sin Ansiedad (0): {np.sum(all_labels == 0)}")
print(f" - Con Ansiedad (1): {np.sum(all_labels == 1)}")
print(f"Segment: {SEGMENT_DURATION}s | Features: ({EXPECTED_FRAMES}, {N_FEATURES})")

class MFCCDataset(Dataset):
    """Extrae MFCC+Δ+ΔΔ on-the-fly. Lazy loading."""
    def __init__(self, file_list, label_list):
        self.samples = []
        for fp, lbl in zip(file_list, label_list):
            try:
                total = librosa.get_duration(path=fp)
                n_segs = max(1, int((total * SAMPLE_RATE - SAMPLES_PER_SEGMENT) / HOP_SIZE) + 1)
                for i in range(n_segs):
                    offset = i * HOP_SIZE / SAMPLE_RATE
                    self.samples.append((fp, offset, int(lbl)))
            except Exception as e:
                print(f"[AVISO] {fp}: {e}")

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        fp, offset, label = self.samples[idx]
        audio, _ = librosa.load(fp, sr=SAMPLE_RATE, offset=offset,
                                duration=SEGMENT_DURATION)
        if len(audio) < SAMPLES_PER_SEGMENT:
            audio = np.pad(audio, (0, SAMPLES_PER_SEGMENT - len(audio)))
        audio = audio[:SAMPLES_PER_SEGMENT]

        mfcc = librosa.feature.mfcc(y=audio, sr=SAMPLE_RATE, n_mfcc=N_MFCC,
                                    n_fft=N_FFT, hop_length=HOP_LENGTH)
        d1   = librosa.feature.delta(mfcc)
        d2   = librosa.feature.delta(mfcc, order=2)
        feat = np.concatenate([mfcc, d1, d2], axis=0).T.astype(np.float32)  # (T, F)
        feat = (feat - feat.mean(0)) / (feat.std(0) + 1e-8)
        if feat.shape[0] < EXPECTED_FRAMES:
            feat = np.pad(feat, ((0, EXPECTED_FRAMES - feat.shape[0]), (0, 0)))
        else:
            feat = feat[:EXPECTED_FRAMES]
        return torch.from_numpy(feat), torch.tensor(label, dtype=torch.long)

def oversample_files(files, labels):
    f1 = files[labels == 1]; l1 = labels[labels == 1]
    diff = np.sum(labels == 0) - np.sum(labels == 1)
    if diff > 0:
        idx    = np.random.choice(len(f1), size=diff, replace=True)
        files  = np.concatenate([files,  f1[idx]])
        labels = np.concatenate([labels, l1[idx]])
    return files, labels

MODEL_NAME = "LSTM-MFCC"

class LSTM_MFCC(nn.Module):
    """LSTM puro sobre MFCC + Δ + ΔΔ.
    Entrada: (batch, EXPECTED_FRAMES, N_FEATURES)."""

    def __init__(self):
        super().__init__()
        self.lstm1 = nn.LSTM(N_FEATURES, 128, batch_first=True)
        self.drop1 = nn.Dropout(0.3)
        self.lstm2 = nn.LSTM(128, 64, batch_first=True)
        self.drop2 = nn.Dropout(0.3)
        self.lstm3 = nn.LSTM(64, 32, batch_first=True)
        self.drop3 = nn.Dropout(0.3)
        self.classifier = nn.Sequential(
            nn.Linear(32, 64), nn.ReLU(inplace=True), nn.Dropout(0.4),
            nn.Linear(64, 32), nn.ReLU(inplace=True), nn.Dropout(0.3),
            nn.Linear(32,  1), nn.Sigmoid(),
        )

    def forward(self, x):
        x, _ = self.lstm1(x); x = self.drop1(x)
        x, _ = self.lstm2(x); x = self.drop2(x)
        x, _ = self.lstm3(x); x = self.drop3(x)
        x = x[:, -1, :]       # último timestep
        return self.classifier(x).squeeze(1)

def build_model():
    return LSTM_MFCC()

def train_one_epoch(model, loader, optimizer, criterion, cw_tensor):
    model.train()
    total_loss = 0.0
    for xb, yb in loader:
        xb = xb.to(DEVICE, non_blocking=True)
        yb = yb.to(DEVICE, non_blocking=True).float()
        optimizer.zero_grad(set_to_none=True)
        preds = model(xb)
        sample_w = cw_tensor[yb.long()]
        loss = (criterion(preds, yb) * sample_w).mean()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

@torch.no_grad()
def predict_file(model, file_path):
    model.eval()
    segs = []
    try:
        audio, _ = librosa.load(file_path, sr=SAMPLE_RATE)
        for start in range(0, len(audio), HOP_SIZE):
            seg = audio[start:start + SAMPLES_PER_SEGMENT]
            if len(seg) < SAMPLES_PER_SEGMENT // 2: continue
            if len(seg) < SAMPLES_PER_SEGMENT:
                seg = np.pad(seg, (0, SAMPLES_PER_SEGMENT - len(seg)))
            seg = seg[:SAMPLES_PER_SEGMENT]
            mfcc = librosa.feature.mfcc(y=seg.astype(np.float32), sr=SAMPLE_RATE,
                                        n_mfcc=N_MFCC, n_fft=N_FFT, hop_length=HOP_LENGTH)
            d1   = librosa.feature.delta(mfcc)
            d2   = librosa.feature.delta(mfcc, order=2)
            feat = np.concatenate([mfcc, d1, d2], axis=0).T.astype(np.float32)
            feat = (feat - feat.mean(0)) / (feat.std(0) + 1e-8)
            if feat.shape[0] < EXPECTED_FRAMES:
                feat = np.pad(feat, ((0, EXPECTED_FRAMES - feat.shape[0]), (0, 0)))
            else:
                feat = feat[:EXPECTED_FRAMES]
            segs.append(feat)
    except Exception as e:
        print(f"  Error {file_path}: {e}"); return None
    if not segs: return None
    probs = []
    for b in range(0, len(segs), BATCH_SIZE):
        batch = np.stack(segs[b:b + BATCH_SIZE])
        bt = torch.from_numpy(batch).to(DEVICE)
        out = model(bt).cpu().numpy()
        probs.extend(out.tolist())
    return float(np.mean(probs))

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
global_y_true, global_y_pred, global_y_prob = [], [], []

print(f"\nINICIANDO CROSS-VALIDATION {MODEL_NAME} | {SEGMENT_DURATION}s | {N_FOLDS} FOLDS\n")

for fold_idx, (tr_idx, te_idx) in enumerate(skf.split(all_files, all_labels)):
    print(f"\n{'─'*55}\n  FOLD {fold_idx+1}/{N_FOLDS}\n{'─'*55}")

    Xtr_f, Xte_f = all_files[tr_idx], all_files[te_idx]
    ytr_f, yte_f = all_labels[tr_idx], all_labels[te_idx]
    Xtr_f, ytr_f = oversample_files(Xtr_f, ytr_f)

    train_ds = MFCCDataset(Xtr_f, ytr_f)
    train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=8, pin_memory=True, persistent_workers=True)
    print(f"  Segmentos train: {len(train_ds)}")

    weights   = class_weight.compute_class_weight('balanced',
                    classes=np.unique(ytr_f), y=ytr_f)
    cw_tensor = torch.tensor(weights, dtype=torch.float32).to(DEVICE)

    model     = build_model().to(DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=1e-4)
    criterion = nn.BCELoss(reduction='none')
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=3, min_lr=1e-7)

    if fold_idx == 0:
        n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
        print(f"  Parámetros entrenables: {n_params:,}\n")

    best_loss, patience_cnt, PATIENCE, best_state = float('inf'), 0, 5, None
    for epoch in range(EPOCHS):
        loss = train_one_epoch(model, train_dl, optimizer, criterion, cw_tensor)
        scheduler.step(loss)
        if loss < best_loss:
            best_loss = loss; patience_cnt = 0
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            mark = "✓"
        else:
            patience_cnt += 1
            mark = f"({patience_cnt}/{PATIENCE})"
        print(f"  Epoch {epoch+1:02d}/{EPOCHS} — loss: {loss:.4f}  best: {best_loss:.4f}  {mark}")
        if patience_cnt >= PATIENCE:
            print("  ► Early stopping activado."); break

    model.load_state_dict(best_state)

    for i, fp in enumerate(Xte_f):
        avg_prob = predict_file(model, fp)
        if avg_prob is None: continue
        global_y_true.append(int(yte_f[i]))
        global_y_pred.append(1 if avg_prob > 0.33 else 0)
        global_y_prob.append(avg_prob)

    del model, train_ds, train_dl, best_state, cw_tensor
    gc.collect(); torch.cuda.empty_cache()

print("\n" + "="*60)
print(f"RESULTADOS {MODEL_NAME} | {SEGMENT_DURATION}s  ({len(global_y_true)} pacientes)")
print("="*60)

cm = confusion_matrix(global_y_true, global_y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Oranges',
            xticklabels=['Sin Ansiedad', 'Con Ansiedad'],
            yticklabels=['Sin Ansiedad', 'Con Ansiedad'])
plt.title(f'Confusión {MODEL_NAME} {SEGMENT_DURATION}s ({N_FOLDS}-Fold CV)')
plt.ylabel('Real'); plt.xlabel('Predicción')
plt.tight_layout()
plt.savefig(f'confusion_{MODEL_NAME.replace(" ","_")}_{SEGMENT_DURATION}s.png', dpi=150)
plt.show()

acc      = accuracy_score(global_y_true, global_y_pred)
prec     = precision_score(global_y_true, global_y_pred, zero_division=0)
rec      = recall_score(global_y_true, global_y_pred, zero_division=0)
f1       = f1_score(global_y_true, global_y_pred, zero_division=0)
f1_macro = f1_score(global_y_true, global_y_pred, average='macro', zero_division=0)
print(f"\nAccuracy :  {acc:.4f}")
print(f"Precision:  {prec:.4f}")
print(f"Recall   :  {rec:.4f}")
print(f"F1-Ans   :  {f1:.4f}  (clase positiva)")
print(f"F1-Macro :  {f1_macro:.4f}")
print()
print(classification_report(global_y_true, global_y_pred,
                             target_names=['Sin Ansiedad', 'Con Ansiedad']))

fpr, tpr, _ = roc_curve(global_y_true, global_y_prob)
roc_auc = auc(fpr, tpr)
plt.figure(figsize=(7, 6))
plt.plot(fpr, tpr, lw=2, label=f'AUC = {roc_auc:.2f}')
plt.plot([0,1],[0,1],'navy',lw=2,linestyle='--')
plt.xlabel('Falsos Positivos'); plt.ylabel('Verdaderos Positivos')
plt.title(f'ROC — {MODEL_NAME} {SEGMENT_DURATION}s')
plt.legend(loc='lower right'); plt.tight_layout()
plt.savefig(f'roc_{MODEL_NAME.replace(" ","_")}_{SEGMENT_DURATION}s.png', dpi=150)
plt.show()


## 4 s | Overlap 50%

In [ ]:
import os
import gc
import numpy as np
import librosa
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit
from sklearn.metrics import (confusion_matrix, classification_report,
                             accuracy_score, precision_score, recall_score,
                             f1_score, roc_curve, auc)
from sklearn.utils import class_weight

# ══════════════════════════════════════════════════════════

BASE_PATH_0 = "../Clasificación Final/Ansiedad/wav/0"
BASE_PATH_1 = "../Clasificación Final/Ansiedad/wav/1"

SAMPLE_RATE         = 16000
SEGMENT_DURATION    = 4
SAMPLES_PER_SEGMENT = int(SEGMENT_DURATION * SAMPLE_RATE)  # 64000
HOP_SIZE            = SAMPLES_PER_SEGMENT // 2  # 50% overlap → 32000
N_MFCC              = 40
HOP_LENGTH          = 512
N_FFT               = 1024
N_FEATURES          = N_MFCC * 3  # MFCC + Δ + ΔΔ = 120
EXPECTED_FRAMES     = int(np.ceil(SAMPLES_PER_SEGMENT / HOP_LENGTH))  # ≈ 126
N_FOLDS             = 5
BATCH_SIZE          = 128
EPOCHS              = 25

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Usando dispositivo: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
    torch.backends.cudnn.benchmark = True

def get_file_lists():
    files_0 = [os.path.join(BASE_PATH_0, f)
               for f in sorted(os.listdir(BASE_PATH_0)) if f.endswith('.wav')]
    files_1 = [os.path.join(BASE_PATH_1, f)
               for f in sorted(os.listdir(BASE_PATH_1)) if f.endswith('.wav')]
    files  = np.array(files_0 + files_1)
    labels = np.array([0]*len(files_0) + [1]*len(files_1), dtype=np.int64)
    return files, labels

all_files, all_labels = get_file_lists()
print(f"\nTotal archivos : {len(all_files)}")
print(f" - Sin Ansiedad (0): {np.sum(all_labels == 0)}")
print(f" - Con Ansiedad (1): {np.sum(all_labels == 1)}")
print(f"Segment: {SEGMENT_DURATION}s | Features: ({EXPECTED_FRAMES}, {N_FEATURES})")

class MFCCDataset(Dataset):
    """Extrae MFCC+Δ+ΔΔ on-the-fly. Lazy loading."""
    def __init__(self, file_list, label_list):
        self.samples = []
        for fp, lbl in zip(file_list, label_list):
            try:
                total = librosa.get_duration(path=fp)
                n_segs = max(1, int((total * SAMPLE_RATE - SAMPLES_PER_SEGMENT) / HOP_SIZE) + 1)
                for i in range(n_segs):
                    offset = i * HOP_SIZE / SAMPLE_RATE
                    self.samples.append((fp, offset, int(lbl)))
            except Exception as e:
                print(f"[AVISO] {fp}: {e}")

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        fp, offset, label = self.samples[idx]
        audio, _ = librosa.load(fp, sr=SAMPLE_RATE, offset=offset,
                                duration=SEGMENT_DURATION)
        if len(audio) < SAMPLES_PER_SEGMENT:
            audio = np.pad(audio, (0, SAMPLES_PER_SEGMENT - len(audio)))
        audio = audio[:SAMPLES_PER_SEGMENT]

        mfcc = librosa.feature.mfcc(y=audio, sr=SAMPLE_RATE, n_mfcc=N_MFCC,
                                    n_fft=N_FFT, hop_length=HOP_LENGTH)
        d1   = librosa.feature.delta(mfcc)
        d2   = librosa.feature.delta(mfcc, order=2)
        feat = np.concatenate([mfcc, d1, d2], axis=0).T.astype(np.float32)  # (T, F)
        feat = (feat - feat.mean(0)) / (feat.std(0) + 1e-8)
        if feat.shape[0] < EXPECTED_FRAMES:
            feat = np.pad(feat, ((0, EXPECTED_FRAMES - feat.shape[0]), (0, 0)))
        else:
            feat = feat[:EXPECTED_FRAMES]
        return torch.from_numpy(feat), torch.tensor(label, dtype=torch.long)

def oversample_files(files, labels):
    f1 = files[labels == 1]; l1 = labels[labels == 1]
    diff = np.sum(labels == 0) - np.sum(labels == 1)
    if diff > 0:
        idx    = np.random.choice(len(f1), size=diff, replace=True)
        files  = np.concatenate([files,  f1[idx]])
        labels = np.concatenate([labels, l1[idx]])
    return files, labels

MODEL_NAME = "LSTM-MFCC"

class LSTM_MFCC(nn.Module):
    """LSTM puro sobre MFCC + Δ + ΔΔ.
    Entrada: (batch, EXPECTED_FRAMES, N_FEATURES)."""

    def __init__(self):
        super().__init__()
        self.lstm1 = nn.LSTM(N_FEATURES, 128, batch_first=True)
        self.drop1 = nn.Dropout(0.3)
        self.lstm2 = nn.LSTM(128, 64, batch_first=True)
        self.drop2 = nn.Dropout(0.3)
        self.lstm3 = nn.LSTM(64, 32, batch_first=True)
        self.drop3 = nn.Dropout(0.3)
        self.classifier = nn.Sequential(
            nn.Linear(32, 64), nn.ReLU(inplace=True), nn.Dropout(0.4),
            nn.Linear(64, 32), nn.ReLU(inplace=True), nn.Dropout(0.3),
            nn.Linear(32,  1), nn.Sigmoid(),
        )

    def forward(self, x):
        x, _ = self.lstm1(x); x = self.drop1(x)
        x, _ = self.lstm2(x); x = self.drop2(x)
        x, _ = self.lstm3(x); x = self.drop3(x)
        x = x[:, -1, :]       # último timestep
        return self.classifier(x).squeeze(1)

def build_model():
    return LSTM_MFCC()

def train_one_epoch(model, loader, optimizer, criterion, cw_tensor):
    model.train()
    total_loss = 0.0
    for xb, yb in loader:
        xb = xb.to(DEVICE, non_blocking=True)
        yb = yb.to(DEVICE, non_blocking=True).float()
        optimizer.zero_grad(set_to_none=True)
        preds = model(xb)
        sample_w = cw_tensor[yb.long()]
        loss = (criterion(preds, yb) * sample_w).mean()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

@torch.no_grad()
def predict_file(model, file_path):
    model.eval()
    segs = []
    try:
        audio, _ = librosa.load(file_path, sr=SAMPLE_RATE)
        for start in range(0, len(audio), HOP_SIZE):
            seg = audio[start:start + SAMPLES_PER_SEGMENT]
            if len(seg) < SAMPLES_PER_SEGMENT // 2: continue
            if len(seg) < SAMPLES_PER_SEGMENT:
                seg = np.pad(seg, (0, SAMPLES_PER_SEGMENT - len(seg)))
            seg = seg[:SAMPLES_PER_SEGMENT]
            mfcc = librosa.feature.mfcc(y=seg.astype(np.float32), sr=SAMPLE_RATE,
                                        n_mfcc=N_MFCC, n_fft=N_FFT, hop_length=HOP_LENGTH)
            d1   = librosa.feature.delta(mfcc)
            d2   = librosa.feature.delta(mfcc, order=2)
            feat = np.concatenate([mfcc, d1, d2], axis=0).T.astype(np.float32)
            feat = (feat - feat.mean(0)) / (feat.std(0) + 1e-8)
            if feat.shape[0] < EXPECTED_FRAMES:
                feat = np.pad(feat, ((0, EXPECTED_FRAMES - feat.shape[0]), (0, 0)))
            else:
                feat = feat[:EXPECTED_FRAMES]
            segs.append(feat)
    except Exception as e:
        print(f"  Error {file_path}: {e}"); return None
    if not segs: return None
    probs = []
    for b in range(0, len(segs), BATCH_SIZE):
        batch = np.stack(segs[b:b + BATCH_SIZE])
        bt = torch.from_numpy(batch).to(DEVICE)
        out = model(bt).cpu().numpy()
        probs.extend(out.tolist())
    return float(np.mean(probs))

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
global_y_true, global_y_pred, global_y_prob = [], [], []

print(f"\nINICIANDO CROSS-VALIDATION {MODEL_NAME} | {SEGMENT_DURATION}s | {N_FOLDS} FOLDS\n")

for fold_idx, (tr_idx, te_idx) in enumerate(skf.split(all_files, all_labels)):
    print(f"\n{'─'*55}\n  FOLD {fold_idx+1}/{N_FOLDS}\n{'─'*55}")

    Xtr_f, Xte_f = all_files[tr_idx], all_files[te_idx]
    ytr_f, yte_f = all_labels[tr_idx], all_labels[te_idx]
    Xtr_f, ytr_f = oversample_files(Xtr_f, ytr_f)

    train_ds = MFCCDataset(Xtr_f, ytr_f)
    train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=8, pin_memory=True, persistent_workers=True)
    print(f"  Segmentos train: {len(train_ds)}")

    weights   = class_weight.compute_class_weight('balanced',
                    classes=np.unique(ytr_f), y=ytr_f)
    cw_tensor = torch.tensor(weights, dtype=torch.float32).to(DEVICE)

    model     = build_model().to(DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=1e-4)
    criterion = nn.BCELoss(reduction='none')
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=3, min_lr=1e-7)

    if fold_idx == 0:
        n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
        print(f"  Parámetros entrenables: {n_params:,}\n")

    best_loss, patience_cnt, PATIENCE, best_state = float('inf'), 0, 5, None
    for epoch in range(EPOCHS):
        loss = train_one_epoch(model, train_dl, optimizer, criterion, cw_tensor)
        scheduler.step(loss)
        if loss < best_loss:
            best_loss = loss; patience_cnt = 0
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            mark = "✓"
        else:
            patience_cnt += 1
            mark = f"({patience_cnt}/{PATIENCE})"
        print(f"  Epoch {epoch+1:02d}/{EPOCHS} — loss: {loss:.4f}  best: {best_loss:.4f}  {mark}")
        if patience_cnt >= PATIENCE:
            print("  ► Early stopping activado."); break

    model.load_state_dict(best_state)

    for i, fp in enumerate(Xte_f):
        avg_prob = predict_file(model, fp)
        if avg_prob is None: continue
        global_y_true.append(int(yte_f[i]))
        global_y_pred.append(1 if avg_prob > 0.33 else 0)
        global_y_prob.append(avg_prob)

    del model, train_ds, train_dl, best_state, cw_tensor
    gc.collect(); torch.cuda.empty_cache()

print("\n" + "="*60)
print(f"RESULTADOS {MODEL_NAME} | {SEGMENT_DURATION}s  ({len(global_y_true)} pacientes)")
print("="*60)

cm = confusion_matrix(global_y_true, global_y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Oranges',
            xticklabels=['Sin Ansiedad', 'Con Ansiedad'],
            yticklabels=['Sin Ansiedad', 'Con Ansiedad'])
plt.title(f'Confusión {MODEL_NAME} {SEGMENT_DURATION}s ({N_FOLDS}-Fold CV)')
plt.ylabel('Real'); plt.xlabel('Predicción')
plt.tight_layout()
plt.savefig(f'confusion_{MODEL_NAME.replace(" ","_")}_{SEGMENT_DURATION}s.png', dpi=150)
plt.show()

acc      = accuracy_score(global_y_true, global_y_pred)
prec     = precision_score(global_y_true, global_y_pred, zero_division=0)
rec      = recall_score(global_y_true, global_y_pred, zero_division=0)
f1       = f1_score(global_y_true, global_y_pred, zero_division=0)
f1_macro = f1_score(global_y_true, global_y_pred, average='macro', zero_division=0)
print(f"\nAccuracy :  {acc:.4f}")
print(f"Precision:  {prec:.4f}")
print(f"Recall   :  {rec:.4f}")
print(f"F1-Ans   :  {f1:.4f}  (clase positiva)")
print(f"F1-Macro :  {f1_macro:.4f}")
print()
print(classification_report(global_y_true, global_y_pred,
                             target_names=['Sin Ansiedad', 'Con Ansiedad']))

fpr, tpr, _ = roc_curve(global_y_true, global_y_prob)
roc_auc = auc(fpr, tpr)
plt.figure(figsize=(7, 6))
plt.plot(fpr, tpr, lw=2, label=f'AUC = {roc_auc:.2f}')
plt.plot([0,1],[0,1],'navy',lw=2,linestyle='--')
plt.xlabel('Falsos Positivos'); plt.ylabel('Verdaderos Positivos')
plt.title(f'ROC — {MODEL_NAME} {SEGMENT_DURATION}s')
plt.legend(loc='lower right'); plt.tight_layout()
plt.savefig(f'roc_{MODEL_NAME.replace(" ","_")}_{SEGMENT_DURATION}s.png', dpi=150)
plt.show()


# Transformer — Raw Waveform  (SR=16000)

## 10 s | Overlap 50%

In [ ]:
import os
import gc
import numpy as np
import librosa
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit
from sklearn.metrics import (confusion_matrix, classification_report,
                             accuracy_score, precision_score, recall_score,
                             f1_score, roc_curve, auc)
from sklearn.utils import class_weight

# ══════════════════════════════════════════════════════════

BASE_PATH_0 = "../Clasificación Final/Ansiedad/wav/0"
BASE_PATH_1 = "../Clasificación Final/Ansiedad/wav/1"

SAMPLE_RATE         = 16000
SEGMENT_DURATION    = 10
SAMPLES_PER_SEGMENT = int(SEGMENT_DURATION * SAMPLE_RATE)  # 160000
HOP_SIZE            = SAMPLES_PER_SEGMENT // 2  # 50% overlap → 80000
N_MFCC              = 40
HOP_LENGTH          = 512
N_FFT               = 1024
N_FEATURES          = N_MFCC * 3  # MFCC + Δ + ΔΔ = 120
EXPECTED_FRAMES     = int(np.ceil(SAMPLES_PER_SEGMENT / HOP_LENGTH))  # ≈ 313
N_FOLDS             = 5
BATCH_SIZE          = 128
EPOCHS              = 25

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Usando dispositivo: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
    torch.backends.cudnn.benchmark = True

def get_file_lists():
    files_0 = [os.path.join(BASE_PATH_0, f)
               for f in sorted(os.listdir(BASE_PATH_0)) if f.endswith('.wav')]
    files_1 = [os.path.join(BASE_PATH_1, f)
               for f in sorted(os.listdir(BASE_PATH_1)) if f.endswith('.wav')]
    files  = np.array(files_0 + files_1)
    labels = np.array([0]*len(files_0) + [1]*len(files_1), dtype=np.int64)
    return files, labels

all_files, all_labels = get_file_lists()
print(f"\nTotal archivos : {len(all_files)}")
print(f" - Sin Ansiedad (0): {np.sum(all_labels == 0)}")
print(f" - Con Ansiedad (1): {np.sum(all_labels == 1)}")
print(f"Segment: {SEGMENT_DURATION}s | Features: ({EXPECTED_FRAMES}, {N_FEATURES})")

class MFCCDataset(Dataset):
    """Extrae MFCC+Δ+ΔΔ on-the-fly. Lazy loading."""
    def __init__(self, file_list, label_list):
        self.samples = []
        for fp, lbl in zip(file_list, label_list):
            try:
                total = librosa.get_duration(path=fp)
                n_segs = max(1, int((total * SAMPLE_RATE - SAMPLES_PER_SEGMENT) / HOP_SIZE) + 1)
                for i in range(n_segs):
                    offset = i * HOP_SIZE / SAMPLE_RATE
                    self.samples.append((fp, offset, int(lbl)))
            except Exception as e:
                print(f"[AVISO] {fp}: {e}")

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        fp, offset, label = self.samples[idx]
        audio, _ = librosa.load(fp, sr=SAMPLE_RATE, offset=offset,
                                duration=SEGMENT_DURATION)
        if len(audio) < SAMPLES_PER_SEGMENT:
            audio = np.pad(audio, (0, SAMPLES_PER_SEGMENT - len(audio)))
        audio = audio[:SAMPLES_PER_SEGMENT]

        mfcc = librosa.feature.mfcc(y=audio, sr=SAMPLE_RATE, n_mfcc=N_MFCC,
                                    n_fft=N_FFT, hop_length=HOP_LENGTH)
        d1   = librosa.feature.delta(mfcc)
        d2   = librosa.feature.delta(mfcc, order=2)
        feat = np.concatenate([mfcc, d1, d2], axis=0).T.astype(np.float32)  # (T, F)
        feat = (feat - feat.mean(0)) / (feat.std(0) + 1e-8)
        if feat.shape[0] < EXPECTED_FRAMES:
            feat = np.pad(feat, ((0, EXPECTED_FRAMES - feat.shape[0]), (0, 0)))
        else:
            feat = feat[:EXPECTED_FRAMES]
        return torch.from_numpy(feat), torch.tensor(label, dtype=torch.long)

def oversample_files(files, labels):
    f1 = files[labels == 1]; l1 = labels[labels == 1]
    diff = np.sum(labels == 0) - np.sum(labels == 1)
    if diff > 0:
        idx    = np.random.choice(len(f1), size=diff, replace=True)
        files  = np.concatenate([files,  f1[idx]])
        labels = np.concatenate([labels, l1[idx]])
    return files, labels

MODEL_NAME = "Transformer-Waveform"

# ── Bloques auxiliares ────────────────────────────────────────────────────────
class TCNBlock(nn.Module):
    def __init__(self, channels, kernel_size, dilation, dropout=0.2):
        super().__init__()
        pad = (kernel_size - 1) * dilation
        self.net = nn.Sequential(
            nn.ConstantPad1d((pad, 0), 0),
            nn.Conv1d(channels, channels, kernel_size, dilation=dilation),
            nn.BatchNorm1d(channels), nn.ReLU(inplace=True), nn.Dropout(dropout),
            nn.ConstantPad1d((pad, 0), 0),
            nn.Conv1d(channels, channels, kernel_size, dilation=dilation),
            nn.BatchNorm1d(channels), nn.ReLU(inplace=True), nn.Dropout(dropout),
        )

    def forward(self, x): return x + self.net(x)

class TransformerBlock(nn.Module):
    def __init__(self, d_model, num_heads, ff_dim, dropout=0.1):
        super().__init__()
        self.attn = nn.MultiheadAttention(d_model, num_heads, dropout=dropout,
                                          batch_first=True)
        self.ffn  = nn.Sequential(
            nn.Linear(d_model, ff_dim), nn.ReLU(inplace=True),
            nn.Linear(ff_dim, d_model)
        )
        self.ln1 = nn.LayerNorm(d_model); self.ln2 = nn.LayerNorm(d_model)
        self.dp1 = nn.Dropout(dropout);   self.dp2 = nn.Dropout(dropout)

    def forward(self, x):
        attn, _ = self.attn(x, x, x)
        x = self.ln1(x + self.dp1(attn))
        return self.ln2(x + self.dp2(self.ffn(x)))

def positional_encoding(seq_len, d_model, device):
    pos  = torch.arange(seq_len, device=device).unsqueeze(1).float()
    dims = torch.arange(d_model,  device=device).unsqueeze(0).float()
    angles = pos / (10000 ** (2 * (dims // 2) / d_model))
    angles[:, 0::2] = torch.sin(angles[:, 0::2])
    angles[:, 1::2] = torch.cos(angles[:, 1::2])
    return angles.unsqueeze(0)  # (1, seq_len, d_model)

D_MODEL = 64; NUM_HEADS = 4; FF_DIM = 128; NUM_BLOCKS = 2; DROPOUT_TR = 0.1

class TransformerWaveform(nn.Module):
    """TCN sobre raw waveform → Transformer → clasificación."""

    def __init__(self, input_length):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv1d(1, 32, 80, stride=4, padding=40),
            nn.BatchNorm1d(32), nn.ReLU(inplace=True),
            nn.MaxPool1d(4), nn.Dropout(0.15),
        )
        self.tcn = nn.Sequential(
            *[TCNBlock(32, 3, 2**d, 0.2) for d in range(4)]
        )
        self.proj = nn.Sequential(
            nn.MaxPool1d(4), nn.MaxPool1d(4), nn.MaxPool1d(2)
        )
        self.linear_proj = nn.LazyLinear(D_MODEL)
        self.tr_blocks = nn.Sequential(
            *[TransformerBlock(D_MODEL, NUM_HEADS, FF_DIM, DROPOUT_TR)
              for _ in range(NUM_BLOCKS)]
        )
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.head = nn.Sequential(
            nn.Linear(D_MODEL, 64), nn.ReLU(inplace=True), nn.Dropout(0.5),
            nn.Linear(64, 1), nn.Sigmoid()
        )

    def forward(self, x):                          # x: (B, 1, L)
        x = self.stem(x)                           # (B, 32, ~L/16)
        x = self.tcn(x)                            # (B, 32, ~L/16)
        x = self.proj(x)                           # (B, 32, small)
        x = x.permute(0, 2, 1)                    # (B, T, 32)
        x = self.linear_proj(x)                   # (B, T, D_MODEL)
        x = x + positional_encoding(x.size(1), D_MODEL, x.device)
        x = self.tr_blocks(x)                     # (B, T, D_MODEL)
        x = self.pool(x.permute(0,2,1)).squeeze(2) # (B, D_MODEL)
        return self.head(x).squeeze(1)

def build_model():
    return TransformerWaveform(SAMPLES_PER_SEGMENT)

def train_one_epoch(model, loader, optimizer, criterion, cw_tensor):
    model.train()
    total_loss = 0.0
    for xb, yb in loader:
        xb = xb.to(DEVICE, non_blocking=True)
        yb = yb.to(DEVICE, non_blocking=True).float()
        optimizer.zero_grad(set_to_none=True)
        preds = model(xb)
        sample_w = cw_tensor[yb.long()]
        loss = (criterion(preds, yb) * sample_w).mean()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

@torch.no_grad()
def predict_file(model, file_path):
    model.eval()
    segs = []
    try:
        audio, _ = librosa.load(file_path, sr=SAMPLE_RATE)
        for start in range(0, len(audio), HOP_SIZE):
            seg = audio[start:start + SAMPLES_PER_SEGMENT]
            if len(seg) < SAMPLES_PER_SEGMENT // 2: continue
            if len(seg) < SAMPLES_PER_SEGMENT:
                seg = np.pad(seg, (0, SAMPLES_PER_SEGMENT - len(seg)))
            seg = seg[:SAMPLES_PER_SEGMENT]
            mfcc = librosa.feature.mfcc(y=seg.astype(np.float32), sr=SAMPLE_RATE,
                                        n_mfcc=N_MFCC, n_fft=N_FFT, hop_length=HOP_LENGTH)
            d1   = librosa.feature.delta(mfcc)
            d2   = librosa.feature.delta(mfcc, order=2)
            feat = np.concatenate([mfcc, d1, d2], axis=0).T.astype(np.float32)
            feat = (feat - feat.mean(0)) / (feat.std(0) + 1e-8)
            if feat.shape[0] < EXPECTED_FRAMES:
                feat = np.pad(feat, ((0, EXPECTED_FRAMES - feat.shape[0]), (0, 0)))
            else:
                feat = feat[:EXPECTED_FRAMES]
            segs.append(feat)
    except Exception as e:
        print(f"  Error {file_path}: {e}"); return None
    if not segs: return None
    probs = []
    for b in range(0, len(segs), BATCH_SIZE):
        batch = np.stack(segs[b:b + BATCH_SIZE])
        bt = torch.from_numpy(batch).to(DEVICE)
        out = model(bt).cpu().numpy()
        probs.extend(out.tolist())
    return float(np.mean(probs))

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
global_y_true, global_y_pred, global_y_prob = [], [], []

print(f"\nINICIANDO CROSS-VALIDATION {MODEL_NAME} | {SEGMENT_DURATION}s | {N_FOLDS} FOLDS\n")

for fold_idx, (tr_idx, te_idx) in enumerate(skf.split(all_files, all_labels)):
    print(f"\n{'─'*55}\n  FOLD {fold_idx+1}/{N_FOLDS}\n{'─'*55}")

    Xtr_f, Xte_f = all_files[tr_idx], all_files[te_idx]
    ytr_f, yte_f = all_labels[tr_idx], all_labels[te_idx]
    Xtr_f, ytr_f = oversample_files(Xtr_f, ytr_f)

    train_ds = MFCCDataset(Xtr_f, ytr_f)
    train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=8, pin_memory=True, persistent_workers=True)
    print(f"  Segmentos train: {len(train_ds)}")

    weights   = class_weight.compute_class_weight('balanced',
                    classes=np.unique(ytr_f), y=ytr_f)
    cw_tensor = torch.tensor(weights, dtype=torch.float32).to(DEVICE)

    model     = build_model().to(DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=1e-4)
    criterion = nn.BCELoss(reduction='none')
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=3, min_lr=1e-7)

    if fold_idx == 0:
        n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
        print(f"  Parámetros entrenables: {n_params:,}\n")

    best_loss, patience_cnt, PATIENCE, best_state = float('inf'), 0, 5, None
    for epoch in range(EPOCHS):
        loss = train_one_epoch(model, train_dl, optimizer, criterion, cw_tensor)
        scheduler.step(loss)
        if loss < best_loss:
            best_loss = loss; patience_cnt = 0
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            mark = "✓"
        else:
            patience_cnt += 1
            mark = f"({patience_cnt}/{PATIENCE})"
        print(f"  Epoch {epoch+1:02d}/{EPOCHS} — loss: {loss:.4f}  best: {best_loss:.4f}  {mark}")
        if patience_cnt >= PATIENCE:
            print("  ► Early stopping activado."); break

    model.load_state_dict(best_state)

    for i, fp in enumerate(Xte_f):
        avg_prob = predict_file(model, fp)
        if avg_prob is None: continue
        global_y_true.append(int(yte_f[i]))
        global_y_pred.append(1 if avg_prob > 0.33 else 0)
        global_y_prob.append(avg_prob)

    del model, train_ds, train_dl, best_state, cw_tensor
    gc.collect(); torch.cuda.empty_cache()

print("\n" + "="*60)
print(f"RESULTADOS {MODEL_NAME} | {SEGMENT_DURATION}s  ({len(global_y_true)} pacientes)")
print("="*60)

cm = confusion_matrix(global_y_true, global_y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='RdPu',
            xticklabels=['Sin Ansiedad', 'Con Ansiedad'],
            yticklabels=['Sin Ansiedad', 'Con Ansiedad'])
plt.title(f'Confusión {MODEL_NAME} {SEGMENT_DURATION}s ({N_FOLDS}-Fold CV)')
plt.ylabel('Real'); plt.xlabel('Predicción')
plt.tight_layout()
plt.savefig(f'confusion_{MODEL_NAME.replace(" ","_")}_{SEGMENT_DURATION}s.png', dpi=150)
plt.show()

acc      = accuracy_score(global_y_true, global_y_pred)
prec     = precision_score(global_y_true, global_y_pred, zero_division=0)
rec      = recall_score(global_y_true, global_y_pred, zero_division=0)
f1       = f1_score(global_y_true, global_y_pred, zero_division=0)
f1_macro = f1_score(global_y_true, global_y_pred, average='macro', zero_division=0)
print(f"\nAccuracy :  {acc:.4f}")
print(f"Precision:  {prec:.4f}")
print(f"Recall   :  {rec:.4f}")
print(f"F1-Ans   :  {f1:.4f}  (clase positiva)")
print(f"F1-Macro :  {f1_macro:.4f}")
print()
print(classification_report(global_y_true, global_y_pred,
                             target_names=['Sin Ansiedad', 'Con Ansiedad']))

fpr, tpr, _ = roc_curve(global_y_true, global_y_prob)
roc_auc = auc(fpr, tpr)
plt.figure(figsize=(7, 6))
plt.plot(fpr, tpr, lw=2, label=f'AUC = {roc_auc:.2f}')
plt.plot([0,1],[0,1],'navy',lw=2,linestyle='--')
plt.xlabel('Falsos Positivos'); plt.ylabel('Verdaderos Positivos')
plt.title(f'ROC — {MODEL_NAME} {SEGMENT_DURATION}s')
plt.legend(loc='lower right'); plt.tight_layout()
plt.savefig(f'roc_{MODEL_NAME.replace(" ","_")}_{SEGMENT_DURATION}s.png', dpi=150)
plt.show()


## 2 s | Overlap 50%

In [ ]:
import os
import gc
import numpy as np
import librosa
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit
from sklearn.metrics import (confusion_matrix, classification_report,
                             accuracy_score, precision_score, recall_score,
                             f1_score, roc_curve, auc)
from sklearn.utils import class_weight

# ══════════════════════════════════════════════════════════

BASE_PATH_0 = "../Clasificación Final/Ansiedad/wav/0"
BASE_PATH_1 = "../Clasificación Final/Ansiedad/wav/1"

SAMPLE_RATE         = 16000
SEGMENT_DURATION    = 2
SAMPLES_PER_SEGMENT = int(SEGMENT_DURATION * SAMPLE_RATE)  # 32000
HOP_SIZE            = SAMPLES_PER_SEGMENT // 2  # 50% overlap → 16000
N_MFCC              = 40
HOP_LENGTH          = 512
N_FFT               = 1024
N_FEATURES          = N_MFCC * 3  # MFCC + Δ + ΔΔ = 120
EXPECTED_FRAMES     = int(np.ceil(SAMPLES_PER_SEGMENT / HOP_LENGTH))  # ≈ 63
N_FOLDS             = 5
BATCH_SIZE          = 128
EPOCHS              = 25

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Usando dispositivo: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
    torch.backends.cudnn.benchmark = True

def get_file_lists():
    files_0 = [os.path.join(BASE_PATH_0, f)
               for f in sorted(os.listdir(BASE_PATH_0)) if f.endswith('.wav')]
    files_1 = [os.path.join(BASE_PATH_1, f)
               for f in sorted(os.listdir(BASE_PATH_1)) if f.endswith('.wav')]
    files  = np.array(files_0 + files_1)
    labels = np.array([0]*len(files_0) + [1]*len(files_1), dtype=np.int64)
    return files, labels

all_files, all_labels = get_file_lists()
print(f"\nTotal archivos : {len(all_files)}")
print(f" - Sin Ansiedad (0): {np.sum(all_labels == 0)}")
print(f" - Con Ansiedad (1): {np.sum(all_labels == 1)}")
print(f"Segment: {SEGMENT_DURATION}s | Features: ({EXPECTED_FRAMES}, {N_FEATURES})")

class MFCCDataset(Dataset):
    """Extrae MFCC+Δ+ΔΔ on-the-fly. Lazy loading."""
    def __init__(self, file_list, label_list):
        self.samples = []
        for fp, lbl in zip(file_list, label_list):
            try:
                total = librosa.get_duration(path=fp)
                n_segs = max(1, int((total * SAMPLE_RATE - SAMPLES_PER_SEGMENT) / HOP_SIZE) + 1)
                for i in range(n_segs):
                    offset = i * HOP_SIZE / SAMPLE_RATE
                    self.samples.append((fp, offset, int(lbl)))
            except Exception as e:
                print(f"[AVISO] {fp}: {e}")

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        fp, offset, label = self.samples[idx]
        audio, _ = librosa.load(fp, sr=SAMPLE_RATE, offset=offset,
                                duration=SEGMENT_DURATION)
        if len(audio) < SAMPLES_PER_SEGMENT:
            audio = np.pad(audio, (0, SAMPLES_PER_SEGMENT - len(audio)))
        audio = audio[:SAMPLES_PER_SEGMENT]

        mfcc = librosa.feature.mfcc(y=audio, sr=SAMPLE_RATE, n_mfcc=N_MFCC,
                                    n_fft=N_FFT, hop_length=HOP_LENGTH)
        d1   = librosa.feature.delta(mfcc)
        d2   = librosa.feature.delta(mfcc, order=2)
        feat = np.concatenate([mfcc, d1, d2], axis=0).T.astype(np.float32)  # (T, F)
        feat = (feat - feat.mean(0)) / (feat.std(0) + 1e-8)
        if feat.shape[0] < EXPECTED_FRAMES:
            feat = np.pad(feat, ((0, EXPECTED_FRAMES - feat.shape[0]), (0, 0)))
        else:
            feat = feat[:EXPECTED_FRAMES]
        return torch.from_numpy(feat), torch.tensor(label, dtype=torch.long)

def oversample_files(files, labels):
    f1 = files[labels == 1]; l1 = labels[labels == 1]
    diff = np.sum(labels == 0) - np.sum(labels == 1)
    if diff > 0:
        idx    = np.random.choice(len(f1), size=diff, replace=True)
        files  = np.concatenate([files,  f1[idx]])
        labels = np.concatenate([labels, l1[idx]])
    return files, labels

MODEL_NAME = "Transformer-Waveform"

# ── Bloques auxiliares ────────────────────────────────────────────────────────
class TCNBlock(nn.Module):
    def __init__(self, channels, kernel_size, dilation, dropout=0.2):
        super().__init__()
        pad = (kernel_size - 1) * dilation
        self.net = nn.Sequential(
            nn.ConstantPad1d((pad, 0), 0),
            nn.Conv1d(channels, channels, kernel_size, dilation=dilation),
            nn.BatchNorm1d(channels), nn.ReLU(inplace=True), nn.Dropout(dropout),
            nn.ConstantPad1d((pad, 0), 0),
            nn.Conv1d(channels, channels, kernel_size, dilation=dilation),
            nn.BatchNorm1d(channels), nn.ReLU(inplace=True), nn.Dropout(dropout),
        )

    def forward(self, x): return x + self.net(x)

class TransformerBlock(nn.Module):
    def __init__(self, d_model, num_heads, ff_dim, dropout=0.1):
        super().__init__()
        self.attn = nn.MultiheadAttention(d_model, num_heads, dropout=dropout,
                                          batch_first=True)
        self.ffn  = nn.Sequential(
            nn.Linear(d_model, ff_dim), nn.ReLU(inplace=True),
            nn.Linear(ff_dim, d_model)
        )
        self.ln1 = nn.LayerNorm(d_model); self.ln2 = nn.LayerNorm(d_model)
        self.dp1 = nn.Dropout(dropout);   self.dp2 = nn.Dropout(dropout)

    def forward(self, x):
        attn, _ = self.attn(x, x, x)
        x = self.ln1(x + self.dp1(attn))
        return self.ln2(x + self.dp2(self.ffn(x)))

def positional_encoding(seq_len, d_model, device):
    pos  = torch.arange(seq_len, device=device).unsqueeze(1).float()
    dims = torch.arange(d_model,  device=device).unsqueeze(0).float()
    angles = pos / (10000 ** (2 * (dims // 2) / d_model))
    angles[:, 0::2] = torch.sin(angles[:, 0::2])
    angles[:, 1::2] = torch.cos(angles[:, 1::2])
    return angles.unsqueeze(0)  # (1, seq_len, d_model)

D_MODEL = 64; NUM_HEADS = 4; FF_DIM = 128; NUM_BLOCKS = 2; DROPOUT_TR = 0.1

class TransformerWaveform(nn.Module):
    """TCN sobre raw waveform → Transformer → clasificación."""

    def __init__(self, input_length):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv1d(1, 32, 80, stride=4, padding=40),
            nn.BatchNorm1d(32), nn.ReLU(inplace=True),
            nn.MaxPool1d(4), nn.Dropout(0.15),
        )
        self.tcn = nn.Sequential(
            *[TCNBlock(32, 3, 2**d, 0.2) for d in range(4)]
        )
        self.proj = nn.Sequential(
            nn.MaxPool1d(4), nn.MaxPool1d(4), nn.MaxPool1d(2)
        )
        self.linear_proj = nn.LazyLinear(D_MODEL)
        self.tr_blocks = nn.Sequential(
            *[TransformerBlock(D_MODEL, NUM_HEADS, FF_DIM, DROPOUT_TR)
              for _ in range(NUM_BLOCKS)]
        )
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.head = nn.Sequential(
            nn.Linear(D_MODEL, 64), nn.ReLU(inplace=True), nn.Dropout(0.5),
            nn.Linear(64, 1), nn.Sigmoid()
        )

    def forward(self, x):                          # x: (B, 1, L)
        x = self.stem(x)                           # (B, 32, ~L/16)
        x = self.tcn(x)                            # (B, 32, ~L/16)
        x = self.proj(x)                           # (B, 32, small)
        x = x.permute(0, 2, 1)                    # (B, T, 32)
        x = self.linear_proj(x)                   # (B, T, D_MODEL)
        x = x + positional_encoding(x.size(1), D_MODEL, x.device)
        x = self.tr_blocks(x)                     # (B, T, D_MODEL)
        x = self.pool(x.permute(0,2,1)).squeeze(2) # (B, D_MODEL)
        return self.head(x).squeeze(1)

def build_model():
    return TransformerWaveform(SAMPLES_PER_SEGMENT)

def train_one_epoch(model, loader, optimizer, criterion, cw_tensor):
    model.train()
    total_loss = 0.0
    for xb, yb in loader:
        xb = xb.to(DEVICE, non_blocking=True)
        yb = yb.to(DEVICE, non_blocking=True).float()
        optimizer.zero_grad(set_to_none=True)
        preds = model(xb)
        sample_w = cw_tensor[yb.long()]
        loss = (criterion(preds, yb) * sample_w).mean()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

@torch.no_grad()
def predict_file(model, file_path):
    model.eval()
    segs = []
    try:
        audio, _ = librosa.load(file_path, sr=SAMPLE_RATE)
        for start in range(0, len(audio), HOP_SIZE):
            seg = audio[start:start + SAMPLES_PER_SEGMENT]
            if len(seg) < SAMPLES_PER_SEGMENT // 2: continue
            if len(seg) < SAMPLES_PER_SEGMENT:
                seg = np.pad(seg, (0, SAMPLES_PER_SEGMENT - len(seg)))
            seg = seg[:SAMPLES_PER_SEGMENT]
            mfcc = librosa.feature.mfcc(y=seg.astype(np.float32), sr=SAMPLE_RATE,
                                        n_mfcc=N_MFCC, n_fft=N_FFT, hop_length=HOP_LENGTH)
            d1   = librosa.feature.delta(mfcc)
            d2   = librosa.feature.delta(mfcc, order=2)
            feat = np.concatenate([mfcc, d1, d2], axis=0).T.astype(np.float32)
            feat = (feat - feat.mean(0)) / (feat.std(0) + 1e-8)
            if feat.shape[0] < EXPECTED_FRAMES:
                feat = np.pad(feat, ((0, EXPECTED_FRAMES - feat.shape[0]), (0, 0)))
            else:
                feat = feat[:EXPECTED_FRAMES]
            segs.append(feat)
    except Exception as e:
        print(f"  Error {file_path}: {e}"); return None
    if not segs: return None
    probs = []
    for b in range(0, len(segs), BATCH_SIZE):
        batch = np.stack(segs[b:b + BATCH_SIZE])
        bt = torch.from_numpy(batch).to(DEVICE)
        out = model(bt).cpu().numpy()
        probs.extend(out.tolist())
    return float(np.mean(probs))

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
global_y_true, global_y_pred, global_y_prob = [], [], []

print(f"\nINICIANDO CROSS-VALIDATION {MODEL_NAME} | {SEGMENT_DURATION}s | {N_FOLDS} FOLDS\n")

for fold_idx, (tr_idx, te_idx) in enumerate(skf.split(all_files, all_labels)):
    print(f"\n{'─'*55}\n  FOLD {fold_idx+1}/{N_FOLDS}\n{'─'*55}")

    Xtr_f, Xte_f = all_files[tr_idx], all_files[te_idx]
    ytr_f, yte_f = all_labels[tr_idx], all_labels[te_idx]
    Xtr_f, ytr_f = oversample_files(Xtr_f, ytr_f)

    train_ds = MFCCDataset(Xtr_f, ytr_f)
    train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=8, pin_memory=True, persistent_workers=True)
    print(f"  Segmentos train: {len(train_ds)}")

    weights   = class_weight.compute_class_weight('balanced',
                    classes=np.unique(ytr_f), y=ytr_f)
    cw_tensor = torch.tensor(weights, dtype=torch.float32).to(DEVICE)

    model     = build_model().to(DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=1e-4)
    criterion = nn.BCELoss(reduction='none')
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=3, min_lr=1e-7)

    if fold_idx == 0:
        n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
        print(f"  Parámetros entrenables: {n_params:,}\n")

    best_loss, patience_cnt, PATIENCE, best_state = float('inf'), 0, 5, None
    for epoch in range(EPOCHS):
        loss = train_one_epoch(model, train_dl, optimizer, criterion, cw_tensor)
        scheduler.step(loss)
        if loss < best_loss:
            best_loss = loss; patience_cnt = 0
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            mark = "✓"
        else:
            patience_cnt += 1
            mark = f"({patience_cnt}/{PATIENCE})"
        print(f"  Epoch {epoch+1:02d}/{EPOCHS} — loss: {loss:.4f}  best: {best_loss:.4f}  {mark}")
        if patience_cnt >= PATIENCE:
            print("  ► Early stopping activado."); break

    model.load_state_dict(best_state)

    for i, fp in enumerate(Xte_f):
        avg_prob = predict_file(model, fp)
        if avg_prob is None: continue
        global_y_true.append(int(yte_f[i]))
        global_y_pred.append(1 if avg_prob > 0.33 else 0)
        global_y_prob.append(avg_prob)

    del model, train_ds, train_dl, best_state, cw_tensor
    gc.collect(); torch.cuda.empty_cache()

print("\n" + "="*60)
print(f"RESULTADOS {MODEL_NAME} | {SEGMENT_DURATION}s  ({len(global_y_true)} pacientes)")
print("="*60)

cm = confusion_matrix(global_y_true, global_y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='RdPu',
            xticklabels=['Sin Ansiedad', 'Con Ansiedad'],
            yticklabels=['Sin Ansiedad', 'Con Ansiedad'])
plt.title(f'Confusión {MODEL_NAME} {SEGMENT_DURATION}s ({N_FOLDS}-Fold CV)')
plt.ylabel('Real'); plt.xlabel('Predicción')
plt.tight_layout()
plt.savefig(f'confusion_{MODEL_NAME.replace(" ","_")}_{SEGMENT_DURATION}s.png', dpi=150)
plt.show()

acc      = accuracy_score(global_y_true, global_y_pred)
prec     = precision_score(global_y_true, global_y_pred, zero_division=0)
rec      = recall_score(global_y_true, global_y_pred, zero_division=0)
f1       = f1_score(global_y_true, global_y_pred, zero_division=0)
f1_macro = f1_score(global_y_true, global_y_pred, average='macro', zero_division=0)
print(f"\nAccuracy :  {acc:.4f}")
print(f"Precision:  {prec:.4f}")
print(f"Recall   :  {rec:.4f}")
print(f"F1-Ans   :  {f1:.4f}  (clase positiva)")
print(f"F1-Macro :  {f1_macro:.4f}")
print()
print(classification_report(global_y_true, global_y_pred,
                             target_names=['Sin Ansiedad', 'Con Ansiedad']))

fpr, tpr, _ = roc_curve(global_y_true, global_y_prob)
roc_auc = auc(fpr, tpr)
plt.figure(figsize=(7, 6))
plt.plot(fpr, tpr, lw=2, label=f'AUC = {roc_auc:.2f}')
plt.plot([0,1],[0,1],'navy',lw=2,linestyle='--')
plt.xlabel('Falsos Positivos'); plt.ylabel('Verdaderos Positivos')
plt.title(f'ROC — {MODEL_NAME} {SEGMENT_DURATION}s')
plt.legend(loc='lower right'); plt.tight_layout()
plt.savefig(f'roc_{MODEL_NAME.replace(" ","_")}_{SEGMENT_DURATION}s.png', dpi=150)
plt.show()


## 3 s | Overlap 50%

In [ ]:
import os
import gc
import numpy as np
import librosa
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit
from sklearn.metrics import (confusion_matrix, classification_report,
                             accuracy_score, precision_score, recall_score,
                             f1_score, roc_curve, auc)
from sklearn.utils import class_weight

# ══════════════════════════════════════════════════════════

BASE_PATH_0 = "../Clasificación Final/Ansiedad/wav/0"
BASE_PATH_1 = "../Clasificación Final/Ansiedad/wav/1"

SAMPLE_RATE         = 16000
SEGMENT_DURATION    = 3
SAMPLES_PER_SEGMENT = int(SEGMENT_DURATION * SAMPLE_RATE)  # 48000
HOP_SIZE            = SAMPLES_PER_SEGMENT // 2  # 50% overlap → 24000
N_MFCC              = 40
HOP_LENGTH          = 512
N_FFT               = 1024
N_FEATURES          = N_MFCC * 3  # MFCC + Δ + ΔΔ = 120
EXPECTED_FRAMES     = int(np.ceil(SAMPLES_PER_SEGMENT / HOP_LENGTH))  # ≈ 94
N_FOLDS             = 5
BATCH_SIZE          = 128
EPOCHS              = 25

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Usando dispositivo: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
    torch.backends.cudnn.benchmark = True

def get_file_lists():
    files_0 = [os.path.join(BASE_PATH_0, f)
               for f in sorted(os.listdir(BASE_PATH_0)) if f.endswith('.wav')]
    files_1 = [os.path.join(BASE_PATH_1, f)
               for f in sorted(os.listdir(BASE_PATH_1)) if f.endswith('.wav')]
    files  = np.array(files_0 + files_1)
    labels = np.array([0]*len(files_0) + [1]*len(files_1), dtype=np.int64)
    return files, labels

all_files, all_labels = get_file_lists()
print(f"\nTotal archivos : {len(all_files)}")
print(f" - Sin Ansiedad (0): {np.sum(all_labels == 0)}")
print(f" - Con Ansiedad (1): {np.sum(all_labels == 1)}")
print(f"Segment: {SEGMENT_DURATION}s | Features: ({EXPECTED_FRAMES}, {N_FEATURES})")

class MFCCDataset(Dataset):
    """Extrae MFCC+Δ+ΔΔ on-the-fly. Lazy loading."""
    def __init__(self, file_list, label_list):
        self.samples = []
        for fp, lbl in zip(file_list, label_list):
            try:
                total = librosa.get_duration(path=fp)
                n_segs = max(1, int((total * SAMPLE_RATE - SAMPLES_PER_SEGMENT) / HOP_SIZE) + 1)
                for i in range(n_segs):
                    offset = i * HOP_SIZE / SAMPLE_RATE
                    self.samples.append((fp, offset, int(lbl)))
            except Exception as e:
                print(f"[AVISO] {fp}: {e}")

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        fp, offset, label = self.samples[idx]
        audio, _ = librosa.load(fp, sr=SAMPLE_RATE, offset=offset,
                                duration=SEGMENT_DURATION)
        if len(audio) < SAMPLES_PER_SEGMENT:
            audio = np.pad(audio, (0, SAMPLES_PER_SEGMENT - len(audio)))
        audio = audio[:SAMPLES_PER_SEGMENT]

        mfcc = librosa.feature.mfcc(y=audio, sr=SAMPLE_RATE, n_mfcc=N_MFCC,
                                    n_fft=N_FFT, hop_length=HOP_LENGTH)
        d1   = librosa.feature.delta(mfcc)
        d2   = librosa.feature.delta(mfcc, order=2)
        feat = np.concatenate([mfcc, d1, d2], axis=0).T.astype(np.float32)  # (T, F)
        feat = (feat - feat.mean(0)) / (feat.std(0) + 1e-8)
        if feat.shape[0] < EXPECTED_FRAMES:
            feat = np.pad(feat, ((0, EXPECTED_FRAMES - feat.shape[0]), (0, 0)))
        else:
            feat = feat[:EXPECTED_FRAMES]
        return torch.from_numpy(feat), torch.tensor(label, dtype=torch.long)

def oversample_files(files, labels):
    f1 = files[labels == 1]; l1 = labels[labels == 1]
    diff = np.sum(labels == 0) - np.sum(labels == 1)
    if diff > 0:
        idx    = np.random.choice(len(f1), size=diff, replace=True)
        files  = np.concatenate([files,  f1[idx]])
        labels = np.concatenate([labels, l1[idx]])
    return files, labels

MODEL_NAME = "Transformer-Waveform"

# ── Bloques auxiliares ────────────────────────────────────────────────────────
class TCNBlock(nn.Module):
    def __init__(self, channels, kernel_size, dilation, dropout=0.2):
        super().__init__()
        pad = (kernel_size - 1) * dilation
        self.net = nn.Sequential(
            nn.ConstantPad1d((pad, 0), 0),
            nn.Conv1d(channels, channels, kernel_size, dilation=dilation),
            nn.BatchNorm1d(channels), nn.ReLU(inplace=True), nn.Dropout(dropout),
            nn.ConstantPad1d((pad, 0), 0),
            nn.Conv1d(channels, channels, kernel_size, dilation=dilation),
            nn.BatchNorm1d(channels), nn.ReLU(inplace=True), nn.Dropout(dropout),
        )

    def forward(self, x): return x + self.net(x)

class TransformerBlock(nn.Module):
    def __init__(self, d_model, num_heads, ff_dim, dropout=0.1):
        super().__init__()
        self.attn = nn.MultiheadAttention(d_model, num_heads, dropout=dropout,
                                          batch_first=True)
        self.ffn  = nn.Sequential(
            nn.Linear(d_model, ff_dim), nn.ReLU(inplace=True),
            nn.Linear(ff_dim, d_model)
        )
        self.ln1 = nn.LayerNorm(d_model); self.ln2 = nn.LayerNorm(d_model)
        self.dp1 = nn.Dropout(dropout);   self.dp2 = nn.Dropout(dropout)

    def forward(self, x):
        attn, _ = self.attn(x, x, x)
        x = self.ln1(x + self.dp1(attn))
        return self.ln2(x + self.dp2(self.ffn(x)))

def positional_encoding(seq_len, d_model, device):
    pos  = torch.arange(seq_len, device=device).unsqueeze(1).float()
    dims = torch.arange(d_model,  device=device).unsqueeze(0).float()
    angles = pos / (10000 ** (2 * (dims // 2) / d_model))
    angles[:, 0::2] = torch.sin(angles[:, 0::2])
    angles[:, 1::2] = torch.cos(angles[:, 1::2])
    return angles.unsqueeze(0)  # (1, seq_len, d_model)

D_MODEL = 64; NUM_HEADS = 4; FF_DIM = 128; NUM_BLOCKS = 2; DROPOUT_TR = 0.1

class TransformerWaveform(nn.Module):
    """TCN sobre raw waveform → Transformer → clasificación."""

    def __init__(self, input_length):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv1d(1, 32, 80, stride=4, padding=40),
            nn.BatchNorm1d(32), nn.ReLU(inplace=True),
            nn.MaxPool1d(4), nn.Dropout(0.15),
        )
        self.tcn = nn.Sequential(
            *[TCNBlock(32, 3, 2**d, 0.2) for d in range(4)]
        )
        self.proj = nn.Sequential(
            nn.MaxPool1d(4), nn.MaxPool1d(4), nn.MaxPool1d(2)
        )
        self.linear_proj = nn.LazyLinear(D_MODEL)
        self.tr_blocks = nn.Sequential(
            *[TransformerBlock(D_MODEL, NUM_HEADS, FF_DIM, DROPOUT_TR)
              for _ in range(NUM_BLOCKS)]
        )
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.head = nn.Sequential(
            nn.Linear(D_MODEL, 64), nn.ReLU(inplace=True), nn.Dropout(0.5),
            nn.Linear(64, 1), nn.Sigmoid()
        )

    def forward(self, x):                          # x: (B, 1, L)
        x = self.stem(x)                           # (B, 32, ~L/16)
        x = self.tcn(x)                            # (B, 32, ~L/16)
        x = self.proj(x)                           # (B, 32, small)
        x = x.permute(0, 2, 1)                    # (B, T, 32)
        x = self.linear_proj(x)                   # (B, T, D_MODEL)
        x = x + positional_encoding(x.size(1), D_MODEL, x.device)
        x = self.tr_blocks(x)                     # (B, T, D_MODEL)
        x = self.pool(x.permute(0,2,1)).squeeze(2) # (B, D_MODEL)
        return self.head(x).squeeze(1)

def build_model():
    return TransformerWaveform(SAMPLES_PER_SEGMENT)

def train_one_epoch(model, loader, optimizer, criterion, cw_tensor):
    model.train()
    total_loss = 0.0
    for xb, yb in loader:
        xb = xb.to(DEVICE, non_blocking=True)
        yb = yb.to(DEVICE, non_blocking=True).float()
        optimizer.zero_grad(set_to_none=True)
        preds = model(xb)
        sample_w = cw_tensor[yb.long()]
        loss = (criterion(preds, yb) * sample_w).mean()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

@torch.no_grad()
def predict_file(model, file_path):
    model.eval()
    segs = []
    try:
        audio, _ = librosa.load(file_path, sr=SAMPLE_RATE)
        for start in range(0, len(audio), HOP_SIZE):
            seg = audio[start:start + SAMPLES_PER_SEGMENT]
            if len(seg) < SAMPLES_PER_SEGMENT // 2: continue
            if len(seg) < SAMPLES_PER_SEGMENT:
                seg = np.pad(seg, (0, SAMPLES_PER_SEGMENT - len(seg)))
            seg = seg[:SAMPLES_PER_SEGMENT]
            mfcc = librosa.feature.mfcc(y=seg.astype(np.float32), sr=SAMPLE_RATE,
                                        n_mfcc=N_MFCC, n_fft=N_FFT, hop_length=HOP_LENGTH)
            d1   = librosa.feature.delta(mfcc)
            d2   = librosa.feature.delta(mfcc, order=2)
            feat = np.concatenate([mfcc, d1, d2], axis=0).T.astype(np.float32)
            feat = (feat - feat.mean(0)) / (feat.std(0) + 1e-8)
            if feat.shape[0] < EXPECTED_FRAMES:
                feat = np.pad(feat, ((0, EXPECTED_FRAMES - feat.shape[0]), (0, 0)))
            else:
                feat = feat[:EXPECTED_FRAMES]
            segs.append(feat)
    except Exception as e:
        print(f"  Error {file_path}: {e}"); return None
    if not segs: return None
    probs = []
    for b in range(0, len(segs), BATCH_SIZE):
        batch = np.stack(segs[b:b + BATCH_SIZE])
        bt = torch.from_numpy(batch).to(DEVICE)
        out = model(bt).cpu().numpy()
        probs.extend(out.tolist())
    return float(np.mean(probs))

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
global_y_true, global_y_pred, global_y_prob = [], [], []

print(f"\nINICIANDO CROSS-VALIDATION {MODEL_NAME} | {SEGMENT_DURATION}s | {N_FOLDS} FOLDS\n")

for fold_idx, (tr_idx, te_idx) in enumerate(skf.split(all_files, all_labels)):
    print(f"\n{'─'*55}\n  FOLD {fold_idx+1}/{N_FOLDS}\n{'─'*55}")

    Xtr_f, Xte_f = all_files[tr_idx], all_files[te_idx]
    ytr_f, yte_f = all_labels[tr_idx], all_labels[te_idx]
    Xtr_f, ytr_f = oversample_files(Xtr_f, ytr_f)

    train_ds = MFCCDataset(Xtr_f, ytr_f)
    train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=8, pin_memory=True, persistent_workers=True)
    print(f"  Segmentos train: {len(train_ds)}")

    weights   = class_weight.compute_class_weight('balanced',
                    classes=np.unique(ytr_f), y=ytr_f)
    cw_tensor = torch.tensor(weights, dtype=torch.float32).to(DEVICE)

    model     = build_model().to(DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=1e-4)
    criterion = nn.BCELoss(reduction='none')
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=3, min_lr=1e-7)

    if fold_idx == 0:
        n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
        print(f"  Parámetros entrenables: {n_params:,}\n")

    best_loss, patience_cnt, PATIENCE, best_state = float('inf'), 0, 5, None
    for epoch in range(EPOCHS):
        loss = train_one_epoch(model, train_dl, optimizer, criterion, cw_tensor)
        scheduler.step(loss)
        if loss < best_loss:
            best_loss = loss; patience_cnt = 0
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            mark = "✓"
        else:
            patience_cnt += 1
            mark = f"({patience_cnt}/{PATIENCE})"
        print(f"  Epoch {epoch+1:02d}/{EPOCHS} — loss: {loss:.4f}  best: {best_loss:.4f}  {mark}")
        if patience_cnt >= PATIENCE:
            print("  ► Early stopping activado."); break

    model.load_state_dict(best_state)

    for i, fp in enumerate(Xte_f):
        avg_prob = predict_file(model, fp)
        if avg_prob is None: continue
        global_y_true.append(int(yte_f[i]))
        global_y_pred.append(1 if avg_prob > 0.33 else 0)
        global_y_prob.append(avg_prob)

    del model, train_ds, train_dl, best_state, cw_tensor
    gc.collect(); torch.cuda.empty_cache()

print("\n" + "="*60)
print(f"RESULTADOS {MODEL_NAME} | {SEGMENT_DURATION}s  ({len(global_y_true)} pacientes)")
print("="*60)

cm = confusion_matrix(global_y_true, global_y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='RdPu',
            xticklabels=['Sin Ansiedad', 'Con Ansiedad'],
            yticklabels=['Sin Ansiedad', 'Con Ansiedad'])
plt.title(f'Confusión {MODEL_NAME} {SEGMENT_DURATION}s ({N_FOLDS}-Fold CV)')
plt.ylabel('Real'); plt.xlabel('Predicción')
plt.tight_layout()
plt.savefig(f'confusion_{MODEL_NAME.replace(" ","_")}_{SEGMENT_DURATION}s.png', dpi=150)
plt.show()

acc      = accuracy_score(global_y_true, global_y_pred)
prec     = precision_score(global_y_true, global_y_pred, zero_division=0)
rec      = recall_score(global_y_true, global_y_pred, zero_division=0)
f1       = f1_score(global_y_true, global_y_pred, zero_division=0)
f1_macro = f1_score(global_y_true, global_y_pred, average='macro', zero_division=0)
print(f"\nAccuracy :  {acc:.4f}")
print(f"Precision:  {prec:.4f}")
print(f"Recall   :  {rec:.4f}")
print(f"F1-Ans   :  {f1:.4f}  (clase positiva)")
print(f"F1-Macro :  {f1_macro:.4f}")
print()
print(classification_report(global_y_true, global_y_pred,
                             target_names=['Sin Ansiedad', 'Con Ansiedad']))

fpr, tpr, _ = roc_curve(global_y_true, global_y_prob)
roc_auc = auc(fpr, tpr)
plt.figure(figsize=(7, 6))
plt.plot(fpr, tpr, lw=2, label=f'AUC = {roc_auc:.2f}')
plt.plot([0,1],[0,1],'navy',lw=2,linestyle='--')
plt.xlabel('Falsos Positivos'); plt.ylabel('Verdaderos Positivos')
plt.title(f'ROC — {MODEL_NAME} {SEGMENT_DURATION}s')
plt.legend(loc='lower right'); plt.tight_layout()
plt.savefig(f'roc_{MODEL_NAME.replace(" ","_")}_{SEGMENT_DURATION}s.png', dpi=150)
plt.show()


## 4 s | Overlap 50%

In [ ]:
import os
import gc
import numpy as np
import librosa
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit
from sklearn.metrics import (confusion_matrix, classification_report,
                             accuracy_score, precision_score, recall_score,
                             f1_score, roc_curve, auc)
from sklearn.utils import class_weight

# ══════════════════════════════════════════════════════════

BASE_PATH_0 = "../Clasificación Final/Ansiedad/wav/0"
BASE_PATH_1 = "../Clasificación Final/Ansiedad/wav/1"

SAMPLE_RATE         = 16000
SEGMENT_DURATION    = 4
SAMPLES_PER_SEGMENT = int(SEGMENT_DURATION * SAMPLE_RATE)  # 64000
HOP_SIZE            = SAMPLES_PER_SEGMENT // 2  # 50% overlap → 32000
N_MFCC              = 40
HOP_LENGTH          = 512
N_FFT               = 1024
N_FEATURES          = N_MFCC * 3  # MFCC + Δ + ΔΔ = 120
EXPECTED_FRAMES     = int(np.ceil(SAMPLES_PER_SEGMENT / HOP_LENGTH))  # ≈ 126
N_FOLDS             = 5
BATCH_SIZE          = 128
EPOCHS              = 25

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Usando dispositivo: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
    torch.backends.cudnn.benchmark = True

def get_file_lists():
    files_0 = [os.path.join(BASE_PATH_0, f)
               for f in sorted(os.listdir(BASE_PATH_0)) if f.endswith('.wav')]
    files_1 = [os.path.join(BASE_PATH_1, f)
               for f in sorted(os.listdir(BASE_PATH_1)) if f.endswith('.wav')]
    files  = np.array(files_0 + files_1)
    labels = np.array([0]*len(files_0) + [1]*len(files_1), dtype=np.int64)
    return files, labels

all_files, all_labels = get_file_lists()
print(f"\nTotal archivos : {len(all_files)}")
print(f" - Sin Ansiedad (0): {np.sum(all_labels == 0)}")
print(f" - Con Ansiedad (1): {np.sum(all_labels == 1)}")
print(f"Segment: {SEGMENT_DURATION}s | Features: ({EXPECTED_FRAMES}, {N_FEATURES})")

class MFCCDataset(Dataset):
    """Extrae MFCC+Δ+ΔΔ on-the-fly. Lazy loading."""
    def __init__(self, file_list, label_list):
        self.samples = []
        for fp, lbl in zip(file_list, label_list):
            try:
                total = librosa.get_duration(path=fp)
                n_segs = max(1, int((total * SAMPLE_RATE - SAMPLES_PER_SEGMENT) / HOP_SIZE) + 1)
                for i in range(n_segs):
                    offset = i * HOP_SIZE / SAMPLE_RATE
                    self.samples.append((fp, offset, int(lbl)))
            except Exception as e:
                print(f"[AVISO] {fp}: {e}")

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        fp, offset, label = self.samples[idx]
        audio, _ = librosa.load(fp, sr=SAMPLE_RATE, offset=offset,
                                duration=SEGMENT_DURATION)
        if len(audio) < SAMPLES_PER_SEGMENT:
            audio = np.pad(audio, (0, SAMPLES_PER_SEGMENT - len(audio)))
        audio = audio[:SAMPLES_PER_SEGMENT]

        mfcc = librosa.feature.mfcc(y=audio, sr=SAMPLE_RATE, n_mfcc=N_MFCC,
                                    n_fft=N_FFT, hop_length=HOP_LENGTH)
        d1   = librosa.feature.delta(mfcc)
        d2   = librosa.feature.delta(mfcc, order=2)
        feat = np.concatenate([mfcc, d1, d2], axis=0).T.astype(np.float32)  # (T, F)
        feat = (feat - feat.mean(0)) / (feat.std(0) + 1e-8)
        if feat.shape[0] < EXPECTED_FRAMES:
            feat = np.pad(feat, ((0, EXPECTED_FRAMES - feat.shape[0]), (0, 0)))
        else:
            feat = feat[:EXPECTED_FRAMES]
        return torch.from_numpy(feat), torch.tensor(label, dtype=torch.long)

def oversample_files(files, labels):
    f1 = files[labels == 1]; l1 = labels[labels == 1]
    diff = np.sum(labels == 0) - np.sum(labels == 1)
    if diff > 0:
        idx    = np.random.choice(len(f1), size=diff, replace=True)
        files  = np.concatenate([files,  f1[idx]])
        labels = np.concatenate([labels, l1[idx]])
    return files, labels

MODEL_NAME = "Transformer-Waveform"

# ── Bloques auxiliares ────────────────────────────────────────────────────────
class TCNBlock(nn.Module):
    def __init__(self, channels, kernel_size, dilation, dropout=0.2):
        super().__init__()
        pad = (kernel_size - 1) * dilation
        self.net = nn.Sequential(
            nn.ConstantPad1d((pad, 0), 0),
            nn.Conv1d(channels, channels, kernel_size, dilation=dilation),
            nn.BatchNorm1d(channels), nn.ReLU(inplace=True), nn.Dropout(dropout),
            nn.ConstantPad1d((pad, 0), 0),
            nn.Conv1d(channels, channels, kernel_size, dilation=dilation),
            nn.BatchNorm1d(channels), nn.ReLU(inplace=True), nn.Dropout(dropout),
        )

    def forward(self, x): return x + self.net(x)

class TransformerBlock(nn.Module):
    def __init__(self, d_model, num_heads, ff_dim, dropout=0.1):
        super().__init__()
        self.attn = nn.MultiheadAttention(d_model, num_heads, dropout=dropout,
                                          batch_first=True)
        self.ffn  = nn.Sequential(
            nn.Linear(d_model, ff_dim), nn.ReLU(inplace=True),
            nn.Linear(ff_dim, d_model)
        )
        self.ln1 = nn.LayerNorm(d_model); self.ln2 = nn.LayerNorm(d_model)
        self.dp1 = nn.Dropout(dropout);   self.dp2 = nn.Dropout(dropout)

    def forward(self, x):
        attn, _ = self.attn(x, x, x)
        x = self.ln1(x + self.dp1(attn))
        return self.ln2(x + self.dp2(self.ffn(x)))

def positional_encoding(seq_len, d_model, device):
    pos  = torch.arange(seq_len, device=device).unsqueeze(1).float()
    dims = torch.arange(d_model,  device=device).unsqueeze(0).float()
    angles = pos / (10000 ** (2 * (dims // 2) / d_model))
    angles[:, 0::2] = torch.sin(angles[:, 0::2])
    angles[:, 1::2] = torch.cos(angles[:, 1::2])
    return angles.unsqueeze(0)  # (1, seq_len, d_model)

D_MODEL = 64; NUM_HEADS = 4; FF_DIM = 128; NUM_BLOCKS = 2; DROPOUT_TR = 0.1

class TransformerWaveform(nn.Module):
    """TCN sobre raw waveform → Transformer → clasificación."""

    def __init__(self, input_length):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv1d(1, 32, 80, stride=4, padding=40),
            nn.BatchNorm1d(32), nn.ReLU(inplace=True),
            nn.MaxPool1d(4), nn.Dropout(0.15),
        )
        self.tcn = nn.Sequential(
            *[TCNBlock(32, 3, 2**d, 0.2) for d in range(4)]
        )
        self.proj = nn.Sequential(
            nn.MaxPool1d(4), nn.MaxPool1d(4), nn.MaxPool1d(2)
        )
        self.linear_proj = nn.LazyLinear(D_MODEL)
        self.tr_blocks = nn.Sequential(
            *[TransformerBlock(D_MODEL, NUM_HEADS, FF_DIM, DROPOUT_TR)
              for _ in range(NUM_BLOCKS)]
        )
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.head = nn.Sequential(
            nn.Linear(D_MODEL, 64), nn.ReLU(inplace=True), nn.Dropout(0.5),
            nn.Linear(64, 1), nn.Sigmoid()
        )

    def forward(self, x):                          # x: (B, 1, L)
        x = self.stem(x)                           # (B, 32, ~L/16)
        x = self.tcn(x)                            # (B, 32, ~L/16)
        x = self.proj(x)                           # (B, 32, small)
        x = x.permute(0, 2, 1)                    # (B, T, 32)
        x = self.linear_proj(x)                   # (B, T, D_MODEL)
        x = x + positional_encoding(x.size(1), D_MODEL, x.device)
        x = self.tr_blocks(x)                     # (B, T, D_MODEL)
        x = self.pool(x.permute(0,2,1)).squeeze(2) # (B, D_MODEL)
        return self.head(x).squeeze(1)

def build_model():
    return TransformerWaveform(SAMPLES_PER_SEGMENT)

def train_one_epoch(model, loader, optimizer, criterion, cw_tensor):
    model.train()
    total_loss = 0.0
    for xb, yb in loader:
        xb = xb.to(DEVICE, non_blocking=True)
        yb = yb.to(DEVICE, non_blocking=True).float()
        optimizer.zero_grad(set_to_none=True)
        preds = model(xb)
        sample_w = cw_tensor[yb.long()]
        loss = (criterion(preds, yb) * sample_w).mean()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

@torch.no_grad()
def predict_file(model, file_path):
    model.eval()
    segs = []
    try:
        audio, _ = librosa.load(file_path, sr=SAMPLE_RATE)
        for start in range(0, len(audio), HOP_SIZE):
            seg = audio[start:start + SAMPLES_PER_SEGMENT]
            if len(seg) < SAMPLES_PER_SEGMENT // 2: continue
            if len(seg) < SAMPLES_PER_SEGMENT:
                seg = np.pad(seg, (0, SAMPLES_PER_SEGMENT - len(seg)))
            seg = seg[:SAMPLES_PER_SEGMENT]
            mfcc = librosa.feature.mfcc(y=seg.astype(np.float32), sr=SAMPLE_RATE,
                                        n_mfcc=N_MFCC, n_fft=N_FFT, hop_length=HOP_LENGTH)
            d1   = librosa.feature.delta(mfcc)
            d2   = librosa.feature.delta(mfcc, order=2)
            feat = np.concatenate([mfcc, d1, d2], axis=0).T.astype(np.float32)
            feat = (feat - feat.mean(0)) / (feat.std(0) + 1e-8)
            if feat.shape[0] < EXPECTED_FRAMES:
                feat = np.pad(feat, ((0, EXPECTED_FRAMES - feat.shape[0]), (0, 0)))
            else:
                feat = feat[:EXPECTED_FRAMES]
            segs.append(feat)
    except Exception as e:
        print(f"  Error {file_path}: {e}"); return None
    if not segs: return None
    probs = []
    for b in range(0, len(segs), BATCH_SIZE):
        batch = np.stack(segs[b:b + BATCH_SIZE])
        bt = torch.from_numpy(batch).to(DEVICE)
        out = model(bt).cpu().numpy()
        probs.extend(out.tolist())
    return float(np.mean(probs))

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
global_y_true, global_y_pred, global_y_prob = [], [], []

print(f"\nINICIANDO CROSS-VALIDATION {MODEL_NAME} | {SEGMENT_DURATION}s | {N_FOLDS} FOLDS\n")

for fold_idx, (tr_idx, te_idx) in enumerate(skf.split(all_files, all_labels)):
    print(f"\n{'─'*55}\n  FOLD {fold_idx+1}/{N_FOLDS}\n{'─'*55}")

    Xtr_f, Xte_f = all_files[tr_idx], all_files[te_idx]
    ytr_f, yte_f = all_labels[tr_idx], all_labels[te_idx]
    Xtr_f, ytr_f = oversample_files(Xtr_f, ytr_f)

    train_ds = MFCCDataset(Xtr_f, ytr_f)
    train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=8, pin_memory=True, persistent_workers=True)
    print(f"  Segmentos train: {len(train_ds)}")

    weights   = class_weight.compute_class_weight('balanced',
                    classes=np.unique(ytr_f), y=ytr_f)
    cw_tensor = torch.tensor(weights, dtype=torch.float32).to(DEVICE)

    model     = build_model().to(DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=1e-4)
    criterion = nn.BCELoss(reduction='none')
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=3, min_lr=1e-7)

    if fold_idx == 0:
        n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
        print(f"  Parámetros entrenables: {n_params:,}\n")

    best_loss, patience_cnt, PATIENCE, best_state = float('inf'), 0, 5, None
    for epoch in range(EPOCHS):
        loss = train_one_epoch(model, train_dl, optimizer, criterion, cw_tensor)
        scheduler.step(loss)
        if loss < best_loss:
            best_loss = loss; patience_cnt = 0
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            mark = "✓"
        else:
            patience_cnt += 1
            mark = f"({patience_cnt}/{PATIENCE})"
        print(f"  Epoch {epoch+1:02d}/{EPOCHS} — loss: {loss:.4f}  best: {best_loss:.4f}  {mark}")
        if patience_cnt >= PATIENCE:
            print("  ► Early stopping activado."); break

    model.load_state_dict(best_state)

    for i, fp in enumerate(Xte_f):
        avg_prob = predict_file(model, fp)
        if avg_prob is None: continue
        global_y_true.append(int(yte_f[i]))
        global_y_pred.append(1 if avg_prob > 0.33 else 0)
        global_y_prob.append(avg_prob)

    del model, train_ds, train_dl, best_state, cw_tensor
    gc.collect(); torch.cuda.empty_cache()

print("\n" + "="*60)
print(f"RESULTADOS {MODEL_NAME} | {SEGMENT_DURATION}s  ({len(global_y_true)} pacientes)")
print("="*60)

cm = confusion_matrix(global_y_true, global_y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='RdPu',
            xticklabels=['Sin Ansiedad', 'Con Ansiedad'],
            yticklabels=['Sin Ansiedad', 'Con Ansiedad'])
plt.title(f'Confusión {MODEL_NAME} {SEGMENT_DURATION}s ({N_FOLDS}-Fold CV)')
plt.ylabel('Real'); plt.xlabel('Predicción')
plt.tight_layout()
plt.savefig(f'confusion_{MODEL_NAME.replace(" ","_")}_{SEGMENT_DURATION}s.png', dpi=150)
plt.show()

acc      = accuracy_score(global_y_true, global_y_pred)
prec     = precision_score(global_y_true, global_y_pred, zero_division=0)
rec      = recall_score(global_y_true, global_y_pred, zero_division=0)
f1       = f1_score(global_y_true, global_y_pred, zero_division=0)
f1_macro = f1_score(global_y_true, global_y_pred, average='macro', zero_division=0)
print(f"\nAccuracy :  {acc:.4f}")
print(f"Precision:  {prec:.4f}")
print(f"Recall   :  {rec:.4f}")
print(f"F1-Ans   :  {f1:.4f}  (clase positiva)")
print(f"F1-Macro :  {f1_macro:.4f}")
print()
print(classification_report(global_y_true, global_y_pred,
                             target_names=['Sin Ansiedad', 'Con Ansiedad']))

fpr, tpr, _ = roc_curve(global_y_true, global_y_prob)
roc_auc = auc(fpr, tpr)
plt.figure(figsize=(7, 6))
plt.plot(fpr, tpr, lw=2, label=f'AUC = {roc_auc:.2f}')
plt.plot([0,1],[0,1],'navy',lw=2,linestyle='--')
plt.xlabel('Falsos Positivos'); plt.ylabel('Verdaderos Positivos')
plt.title(f'ROC — {MODEL_NAME} {SEGMENT_DURATION}s')
plt.legend(loc='lower right'); plt.tight_layout()
plt.savefig(f'roc_{MODEL_NAME.replace(" ","_")}_{SEGMENT_DURATION}s.png', dpi=150)
plt.show()


# Transformer — MFCC + Δ + ΔΔ  (SR=16000)

## 10 s | Overlap 50%

In [ ]:
import os
import gc
import numpy as np
import librosa
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit
from sklearn.metrics import (confusion_matrix, classification_report,
                             accuracy_score, precision_score, recall_score,
                             f1_score, roc_curve, auc)
from sklearn.utils import class_weight

# ══════════════════════════════════════════════════════════

BASE_PATH_0 = "../Clasificación Final/Ansiedad/wav/0"
BASE_PATH_1 = "../Clasificación Final/Ansiedad/wav/1"

SAMPLE_RATE         = 16000
SEGMENT_DURATION    = 10
SAMPLES_PER_SEGMENT = int(SEGMENT_DURATION * SAMPLE_RATE)  # 160000
HOP_SIZE            = SAMPLES_PER_SEGMENT // 2  # 50% overlap → 80000
N_MFCC              = 40
HOP_LENGTH          = 512
N_FFT               = 1024
N_FEATURES          = N_MFCC * 3  # MFCC + Δ + ΔΔ = 120
EXPECTED_FRAMES     = int(np.ceil(SAMPLES_PER_SEGMENT / HOP_LENGTH))  # ≈ 313
N_FOLDS             = 5
BATCH_SIZE          = 128
EPOCHS              = 25

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Usando dispositivo: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
    torch.backends.cudnn.benchmark = True

def get_file_lists():
    files_0 = [os.path.join(BASE_PATH_0, f)
               for f in sorted(os.listdir(BASE_PATH_0)) if f.endswith('.wav')]
    files_1 = [os.path.join(BASE_PATH_1, f)
               for f in sorted(os.listdir(BASE_PATH_1)) if f.endswith('.wav')]
    files  = np.array(files_0 + files_1)
    labels = np.array([0]*len(files_0) + [1]*len(files_1), dtype=np.int64)
    return files, labels

all_files, all_labels = get_file_lists()
print(f"\nTotal archivos : {len(all_files)}")
print(f" - Sin Ansiedad (0): {np.sum(all_labels == 0)}")
print(f" - Con Ansiedad (1): {np.sum(all_labels == 1)}")
print(f"Segment: {SEGMENT_DURATION}s | Features: ({EXPECTED_FRAMES}, {N_FEATURES})")

class MFCCDataset(Dataset):
    """Extrae MFCC+Δ+ΔΔ on-the-fly. Lazy loading."""
    def __init__(self, file_list, label_list):
        self.samples = []
        for fp, lbl in zip(file_list, label_list):
            try:
                total = librosa.get_duration(path=fp)
                n_segs = max(1, int((total * SAMPLE_RATE - SAMPLES_PER_SEGMENT) / HOP_SIZE) + 1)
                for i in range(n_segs):
                    offset = i * HOP_SIZE / SAMPLE_RATE
                    self.samples.append((fp, offset, int(lbl)))
            except Exception as e:
                print(f"[AVISO] {fp}: {e}")

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        fp, offset, label = self.samples[idx]
        audio, _ = librosa.load(fp, sr=SAMPLE_RATE, offset=offset,
                                duration=SEGMENT_DURATION)
        if len(audio) < SAMPLES_PER_SEGMENT:
            audio = np.pad(audio, (0, SAMPLES_PER_SEGMENT - len(audio)))
        audio = audio[:SAMPLES_PER_SEGMENT]

        mfcc = librosa.feature.mfcc(y=audio, sr=SAMPLE_RATE, n_mfcc=N_MFCC,
                                    n_fft=N_FFT, hop_length=HOP_LENGTH)
        d1   = librosa.feature.delta(mfcc)
        d2   = librosa.feature.delta(mfcc, order=2)
        feat = np.concatenate([mfcc, d1, d2], axis=0).T.astype(np.float32)  # (T, F)
        feat = (feat - feat.mean(0)) / (feat.std(0) + 1e-8)
        if feat.shape[0] < EXPECTED_FRAMES:
            feat = np.pad(feat, ((0, EXPECTED_FRAMES - feat.shape[0]), (0, 0)))
        else:
            feat = feat[:EXPECTED_FRAMES]
        return torch.from_numpy(feat), torch.tensor(label, dtype=torch.long)

def oversample_files(files, labels):
    f1 = files[labels == 1]; l1 = labels[labels == 1]
    diff = np.sum(labels == 0) - np.sum(labels == 1)
    if diff > 0:
        idx    = np.random.choice(len(f1), size=diff, replace=True)
        files  = np.concatenate([files,  f1[idx]])
        labels = np.concatenate([labels, l1[idx]])
    return files, labels

MODEL_NAME = "Transformer-MFCC"

class TransformerBlock(nn.Module):
    def __init__(self, d_model, num_heads, ff_dim, dropout=0.1):
        super().__init__()
        self.attn = nn.MultiheadAttention(d_model, num_heads, dropout=dropout,
                                          batch_first=True)
        self.ffn  = nn.Sequential(
            nn.Linear(d_model, ff_dim), nn.ReLU(inplace=True),
            nn.Linear(ff_dim, d_model)
        )
        self.ln1 = nn.LayerNorm(d_model); self.ln2 = nn.LayerNorm(d_model)
        self.dp1 = nn.Dropout(dropout);   self.dp2 = nn.Dropout(dropout)

    def forward(self, x):
        attn, _ = self.attn(x, x, x)
        x = self.ln1(x + self.dp1(attn))
        return self.ln2(x + self.dp2(self.ffn(x)))

def positional_encoding(seq_len, d_model, device):
    pos  = torch.arange(seq_len, device=device).unsqueeze(1).float()
    dims = torch.arange(d_model,  device=device).unsqueeze(0).float()
    angles = pos / (10000 ** (2 * (dims // 2) / d_model))
    angles[:, 0::2] = torch.sin(angles[:, 0::2])
    angles[:, 1::2] = torch.cos(angles[:, 1::2])
    return angles.unsqueeze(0)

D_MODEL = 64; NUM_HEADS = 4; FF_DIM = 128; NUM_BLOCKS = 2; DROPOUT_TR = 0.1

class TransformerMFCC(nn.Module):
    """Transformer sobre MFCC + Δ + ΔΔ.
    Entrada: (batch, EXPECTED_FRAMES, N_FEATURES)."""

    def __init__(self):
        super().__init__()
        self.bn  = nn.BatchNorm1d(EXPECTED_FRAMES)
        self.proj = nn.Linear(N_FEATURES, D_MODEL)
        self.tr_blocks = nn.Sequential(
            *[TransformerBlock(D_MODEL, NUM_HEADS, FF_DIM, DROPOUT_TR)
              for _ in range(NUM_BLOCKS)]
        )
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.head = nn.Sequential(
            nn.Linear(D_MODEL, 64), nn.ReLU(inplace=True), nn.Dropout(0.5),
            nn.Linear(64, 1), nn.Sigmoid()
        )

    def forward(self, x):                          # (B, T, F)
        x = self.bn(x)
        x = self.proj(x)                           # (B, T, D_MODEL)
        x = x + positional_encoding(x.size(1), D_MODEL, x.device)
        x = self.tr_blocks(x)
        x = self.pool(x.permute(0,2,1)).squeeze(2) # (B, D_MODEL)
        return self.head(x).squeeze(1)

def build_model():
    return TransformerMFCC()

def train_one_epoch(model, loader, optimizer, criterion, cw_tensor):
    model.train()
    total_loss = 0.0
    for xb, yb in loader:
        xb = xb.to(DEVICE, non_blocking=True)
        yb = yb.to(DEVICE, non_blocking=True).float()
        optimizer.zero_grad(set_to_none=True)
        preds = model(xb)
        sample_w = cw_tensor[yb.long()]
        loss = (criterion(preds, yb) * sample_w).mean()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

@torch.no_grad()
def predict_file(model, file_path):
    model.eval()
    segs = []
    try:
        audio, _ = librosa.load(file_path, sr=SAMPLE_RATE)
        for start in range(0, len(audio), HOP_SIZE):
            seg = audio[start:start + SAMPLES_PER_SEGMENT]
            if len(seg) < SAMPLES_PER_SEGMENT // 2: continue
            if len(seg) < SAMPLES_PER_SEGMENT:
                seg = np.pad(seg, (0, SAMPLES_PER_SEGMENT - len(seg)))
            seg = seg[:SAMPLES_PER_SEGMENT]
            mfcc = librosa.feature.mfcc(y=seg.astype(np.float32), sr=SAMPLE_RATE,
                                        n_mfcc=N_MFCC, n_fft=N_FFT, hop_length=HOP_LENGTH)
            d1   = librosa.feature.delta(mfcc)
            d2   = librosa.feature.delta(mfcc, order=2)
            feat = np.concatenate([mfcc, d1, d2], axis=0).T.astype(np.float32)
            feat = (feat - feat.mean(0)) / (feat.std(0) + 1e-8)
            if feat.shape[0] < EXPECTED_FRAMES:
                feat = np.pad(feat, ((0, EXPECTED_FRAMES - feat.shape[0]), (0, 0)))
            else:
                feat = feat[:EXPECTED_FRAMES]
            segs.append(feat)
    except Exception as e:
        print(f"  Error {file_path}: {e}"); return None
    if not segs: return None
    probs = []
    for b in range(0, len(segs), BATCH_SIZE):
        batch = np.stack(segs[b:b + BATCH_SIZE])
        bt = torch.from_numpy(batch).to(DEVICE)
        out = model(bt).cpu().numpy()
        probs.extend(out.tolist())
    return float(np.mean(probs))

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
global_y_true, global_y_pred, global_y_prob = [], [], []

print(f"\nINICIANDO CROSS-VALIDATION {MODEL_NAME} | {SEGMENT_DURATION}s | {N_FOLDS} FOLDS\n")

for fold_idx, (tr_idx, te_idx) in enumerate(skf.split(all_files, all_labels)):
    print(f"\n{'─'*55}\n  FOLD {fold_idx+1}/{N_FOLDS}\n{'─'*55}")

    Xtr_f, Xte_f = all_files[tr_idx], all_files[te_idx]
    ytr_f, yte_f = all_labels[tr_idx], all_labels[te_idx]
    Xtr_f, ytr_f = oversample_files(Xtr_f, ytr_f)

    train_ds = MFCCDataset(Xtr_f, ytr_f)
    train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=8, pin_memory=True, persistent_workers=True)
    print(f"  Segmentos train: {len(train_ds)}")

    weights   = class_weight.compute_class_weight('balanced',
                    classes=np.unique(ytr_f), y=ytr_f)
    cw_tensor = torch.tensor(weights, dtype=torch.float32).to(DEVICE)

    model     = build_model().to(DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=1e-4)
    criterion = nn.BCELoss(reduction='none')
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=3, min_lr=1e-7)

    if fold_idx == 0:
        n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
        print(f"  Parámetros entrenables: {n_params:,}\n")

    best_loss, patience_cnt, PATIENCE, best_state = float('inf'), 0, 5, None
    for epoch in range(EPOCHS):
        loss = train_one_epoch(model, train_dl, optimizer, criterion, cw_tensor)
        scheduler.step(loss)
        if loss < best_loss:
            best_loss = loss; patience_cnt = 0
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            mark = "✓"
        else:
            patience_cnt += 1
            mark = f"({patience_cnt}/{PATIENCE})"
        print(f"  Epoch {epoch+1:02d}/{EPOCHS} — loss: {loss:.4f}  best: {best_loss:.4f}  {mark}")
        if patience_cnt >= PATIENCE:
            print("  ► Early stopping activado."); break

    model.load_state_dict(best_state)

    for i, fp in enumerate(Xte_f):
        avg_prob = predict_file(model, fp)
        if avg_prob is None: continue
        global_y_true.append(int(yte_f[i]))
        global_y_pred.append(1 if avg_prob > 0.33 else 0)
        global_y_prob.append(avg_prob)

    del model, train_ds, train_dl, best_state, cw_tensor
    gc.collect(); torch.cuda.empty_cache()

print("\n" + "="*60)
print(f"RESULTADOS {MODEL_NAME} | {SEGMENT_DURATION}s  ({len(global_y_true)} pacientes)")
print("="*60)

cm = confusion_matrix(global_y_true, global_y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='cool',
            xticklabels=['Sin Ansiedad', 'Con Ansiedad'],
            yticklabels=['Sin Ansiedad', 'Con Ansiedad'])
plt.title(f'Confusión {MODEL_NAME} {SEGMENT_DURATION}s ({N_FOLDS}-Fold CV)')
plt.ylabel('Real'); plt.xlabel('Predicción')
plt.tight_layout()
plt.savefig(f'confusion_{MODEL_NAME.replace(" ","_")}_{SEGMENT_DURATION}s.png', dpi=150)
plt.show()

acc      = accuracy_score(global_y_true, global_y_pred)
prec     = precision_score(global_y_true, global_y_pred, zero_division=0)
rec      = recall_score(global_y_true, global_y_pred, zero_division=0)
f1       = f1_score(global_y_true, global_y_pred, zero_division=0)
f1_macro = f1_score(global_y_true, global_y_pred, average='macro', zero_division=0)
print(f"\nAccuracy :  {acc:.4f}")
print(f"Precision:  {prec:.4f}")
print(f"Recall   :  {rec:.4f}")
print(f"F1-Ans   :  {f1:.4f}  (clase positiva)")
print(f"F1-Macro :  {f1_macro:.4f}")
print()
print(classification_report(global_y_true, global_y_pred,
                             target_names=['Sin Ansiedad', 'Con Ansiedad']))

fpr, tpr, _ = roc_curve(global_y_true, global_y_prob)
roc_auc = auc(fpr, tpr)
plt.figure(figsize=(7, 6))
plt.plot(fpr, tpr, lw=2, label=f'AUC = {roc_auc:.2f}')
plt.plot([0,1],[0,1],'navy',lw=2,linestyle='--')
plt.xlabel('Falsos Positivos'); plt.ylabel('Verdaderos Positivos')
plt.title(f'ROC — {MODEL_NAME} {SEGMENT_DURATION}s')
plt.legend(loc='lower right'); plt.tight_layout()
plt.savefig(f'roc_{MODEL_NAME.replace(" ","_")}_{SEGMENT_DURATION}s.png', dpi=150)
plt.show()


## 2 s | Overlap 50%

In [ ]:
import os
import gc
import numpy as np
import librosa
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit
from sklearn.metrics import (confusion_matrix, classification_report,
                             accuracy_score, precision_score, recall_score,
                             f1_score, roc_curve, auc)
from sklearn.utils import class_weight

# ══════════════════════════════════════════════════════════

BASE_PATH_0 = "../Clasificación Final/Ansiedad/wav/0"
BASE_PATH_1 = "../Clasificación Final/Ansiedad/wav/1"

SAMPLE_RATE         = 16000
SEGMENT_DURATION    = 2
SAMPLES_PER_SEGMENT = int(SEGMENT_DURATION * SAMPLE_RATE)  # 32000
HOP_SIZE            = SAMPLES_PER_SEGMENT // 2  # 50% overlap → 16000
N_MFCC              = 40
HOP_LENGTH          = 512
N_FFT               = 1024
N_FEATURES          = N_MFCC * 3  # MFCC + Δ + ΔΔ = 120
EXPECTED_FRAMES     = int(np.ceil(SAMPLES_PER_SEGMENT / HOP_LENGTH))  # ≈ 63
N_FOLDS             = 5
BATCH_SIZE          = 128
EPOCHS              = 25

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Usando dispositivo: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
    torch.backends.cudnn.benchmark = True

def get_file_lists():
    files_0 = [os.path.join(BASE_PATH_0, f)
               for f in sorted(os.listdir(BASE_PATH_0)) if f.endswith('.wav')]
    files_1 = [os.path.join(BASE_PATH_1, f)
               for f in sorted(os.listdir(BASE_PATH_1)) if f.endswith('.wav')]
    files  = np.array(files_0 + files_1)
    labels = np.array([0]*len(files_0) + [1]*len(files_1), dtype=np.int64)
    return files, labels

all_files, all_labels = get_file_lists()
print(f"\nTotal archivos : {len(all_files)}")
print(f" - Sin Ansiedad (0): {np.sum(all_labels == 0)}")
print(f" - Con Ansiedad (1): {np.sum(all_labels == 1)}")
print(f"Segment: {SEGMENT_DURATION}s | Features: ({EXPECTED_FRAMES}, {N_FEATURES})")

class MFCCDataset(Dataset):
    """Extrae MFCC+Δ+ΔΔ on-the-fly. Lazy loading."""
    def __init__(self, file_list, label_list):
        self.samples = []
        for fp, lbl in zip(file_list, label_list):
            try:
                total = librosa.get_duration(path=fp)
                n_segs = max(1, int((total * SAMPLE_RATE - SAMPLES_PER_SEGMENT) / HOP_SIZE) + 1)
                for i in range(n_segs):
                    offset = i * HOP_SIZE / SAMPLE_RATE
                    self.samples.append((fp, offset, int(lbl)))
            except Exception as e:
                print(f"[AVISO] {fp}: {e}")

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        fp, offset, label = self.samples[idx]
        audio, _ = librosa.load(fp, sr=SAMPLE_RATE, offset=offset,
                                duration=SEGMENT_DURATION)
        if len(audio) < SAMPLES_PER_SEGMENT:
            audio = np.pad(audio, (0, SAMPLES_PER_SEGMENT - len(audio)))
        audio = audio[:SAMPLES_PER_SEGMENT]

        mfcc = librosa.feature.mfcc(y=audio, sr=SAMPLE_RATE, n_mfcc=N_MFCC,
                                    n_fft=N_FFT, hop_length=HOP_LENGTH)
        d1   = librosa.feature.delta(mfcc)
        d2   = librosa.feature.delta(mfcc, order=2)
        feat = np.concatenate([mfcc, d1, d2], axis=0).T.astype(np.float32)  # (T, F)
        feat = (feat - feat.mean(0)) / (feat.std(0) + 1e-8)
        if feat.shape[0] < EXPECTED_FRAMES:
            feat = np.pad(feat, ((0, EXPECTED_FRAMES - feat.shape[0]), (0, 0)))
        else:
            feat = feat[:EXPECTED_FRAMES]
        return torch.from_numpy(feat), torch.tensor(label, dtype=torch.long)

def oversample_files(files, labels):
    f1 = files[labels == 1]; l1 = labels[labels == 1]
    diff = np.sum(labels == 0) - np.sum(labels == 1)
    if diff > 0:
        idx    = np.random.choice(len(f1), size=diff, replace=True)
        files  = np.concatenate([files,  f1[idx]])
        labels = np.concatenate([labels, l1[idx]])
    return files, labels

MODEL_NAME = "Transformer-MFCC"

class TransformerBlock(nn.Module):
    def __init__(self, d_model, num_heads, ff_dim, dropout=0.1):
        super().__init__()
        self.attn = nn.MultiheadAttention(d_model, num_heads, dropout=dropout,
                                          batch_first=True)
        self.ffn  = nn.Sequential(
            nn.Linear(d_model, ff_dim), nn.ReLU(inplace=True),
            nn.Linear(ff_dim, d_model)
        )
        self.ln1 = nn.LayerNorm(d_model); self.ln2 = nn.LayerNorm(d_model)
        self.dp1 = nn.Dropout(dropout);   self.dp2 = nn.Dropout(dropout)

    def forward(self, x):
        attn, _ = self.attn(x, x, x)
        x = self.ln1(x + self.dp1(attn))
        return self.ln2(x + self.dp2(self.ffn(x)))

def positional_encoding(seq_len, d_model, device):
    pos  = torch.arange(seq_len, device=device).unsqueeze(1).float()
    dims = torch.arange(d_model,  device=device).unsqueeze(0).float()
    angles = pos / (10000 ** (2 * (dims // 2) / d_model))
    angles[:, 0::2] = torch.sin(angles[:, 0::2])
    angles[:, 1::2] = torch.cos(angles[:, 1::2])
    return angles.unsqueeze(0)

D_MODEL = 64; NUM_HEADS = 4; FF_DIM = 128; NUM_BLOCKS = 2; DROPOUT_TR = 0.1

class TransformerMFCC(nn.Module):
    """Transformer sobre MFCC + Δ + ΔΔ.
    Entrada: (batch, EXPECTED_FRAMES, N_FEATURES)."""

    def __init__(self):
        super().__init__()
        self.bn  = nn.BatchNorm1d(EXPECTED_FRAMES)
        self.proj = nn.Linear(N_FEATURES, D_MODEL)
        self.tr_blocks = nn.Sequential(
            *[TransformerBlock(D_MODEL, NUM_HEADS, FF_DIM, DROPOUT_TR)
              for _ in range(NUM_BLOCKS)]
        )
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.head = nn.Sequential(
            nn.Linear(D_MODEL, 64), nn.ReLU(inplace=True), nn.Dropout(0.5),
            nn.Linear(64, 1), nn.Sigmoid()
        )

    def forward(self, x):                          # (B, T, F)
        x = self.bn(x)
        x = self.proj(x)                           # (B, T, D_MODEL)
        x = x + positional_encoding(x.size(1), D_MODEL, x.device)
        x = self.tr_blocks(x)
        x = self.pool(x.permute(0,2,1)).squeeze(2) # (B, D_MODEL)
        return self.head(x).squeeze(1)

def build_model():
    return TransformerMFCC()

def train_one_epoch(model, loader, optimizer, criterion, cw_tensor):
    model.train()
    total_loss = 0.0
    for xb, yb in loader:
        xb = xb.to(DEVICE, non_blocking=True)
        yb = yb.to(DEVICE, non_blocking=True).float()
        optimizer.zero_grad(set_to_none=True)
        preds = model(xb)
        sample_w = cw_tensor[yb.long()]
        loss = (criterion(preds, yb) * sample_w).mean()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

@torch.no_grad()
def predict_file(model, file_path):
    model.eval()
    segs = []
    try:
        audio, _ = librosa.load(file_path, sr=SAMPLE_RATE)
        for start in range(0, len(audio), HOP_SIZE):
            seg = audio[start:start + SAMPLES_PER_SEGMENT]
            if len(seg) < SAMPLES_PER_SEGMENT // 2: continue
            if len(seg) < SAMPLES_PER_SEGMENT:
                seg = np.pad(seg, (0, SAMPLES_PER_SEGMENT - len(seg)))
            seg = seg[:SAMPLES_PER_SEGMENT]
            mfcc = librosa.feature.mfcc(y=seg.astype(np.float32), sr=SAMPLE_RATE,
                                        n_mfcc=N_MFCC, n_fft=N_FFT, hop_length=HOP_LENGTH)
            d1   = librosa.feature.delta(mfcc)
            d2   = librosa.feature.delta(mfcc, order=2)
            feat = np.concatenate([mfcc, d1, d2], axis=0).T.astype(np.float32)
            feat = (feat - feat.mean(0)) / (feat.std(0) + 1e-8)
            if feat.shape[0] < EXPECTED_FRAMES:
                feat = np.pad(feat, ((0, EXPECTED_FRAMES - feat.shape[0]), (0, 0)))
            else:
                feat = feat[:EXPECTED_FRAMES]
            segs.append(feat)
    except Exception as e:
        print(f"  Error {file_path}: {e}"); return None
    if not segs: return None
    probs = []
    for b in range(0, len(segs), BATCH_SIZE):
        batch = np.stack(segs[b:b + BATCH_SIZE])
        bt = torch.from_numpy(batch).to(DEVICE)
        out = model(bt).cpu().numpy()
        probs.extend(out.tolist())
    return float(np.mean(probs))

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
global_y_true, global_y_pred, global_y_prob = [], [], []

print(f"\nINICIANDO CROSS-VALIDATION {MODEL_NAME} | {SEGMENT_DURATION}s | {N_FOLDS} FOLDS\n")

for fold_idx, (tr_idx, te_idx) in enumerate(skf.split(all_files, all_labels)):
    print(f"\n{'─'*55}\n  FOLD {fold_idx+1}/{N_FOLDS}\n{'─'*55}")

    Xtr_f, Xte_f = all_files[tr_idx], all_files[te_idx]
    ytr_f, yte_f = all_labels[tr_idx], all_labels[te_idx]
    Xtr_f, ytr_f = oversample_files(Xtr_f, ytr_f)

    train_ds = MFCCDataset(Xtr_f, ytr_f)
    train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=8, pin_memory=True, persistent_workers=True)
    print(f"  Segmentos train: {len(train_ds)}")

    weights   = class_weight.compute_class_weight('balanced',
                    classes=np.unique(ytr_f), y=ytr_f)
    cw_tensor = torch.tensor(weights, dtype=torch.float32).to(DEVICE)

    model     = build_model().to(DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=1e-4)
    criterion = nn.BCELoss(reduction='none')
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=3, min_lr=1e-7)

    if fold_idx == 0:
        n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
        print(f"  Parámetros entrenables: {n_params:,}\n")

    best_loss, patience_cnt, PATIENCE, best_state = float('inf'), 0, 5, None
    for epoch in range(EPOCHS):
        loss = train_one_epoch(model, train_dl, optimizer, criterion, cw_tensor)
        scheduler.step(loss)
        if loss < best_loss:
            best_loss = loss; patience_cnt = 0
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            mark = "✓"
        else:
            patience_cnt += 1
            mark = f"({patience_cnt}/{PATIENCE})"
        print(f"  Epoch {epoch+1:02d}/{EPOCHS} — loss: {loss:.4f}  best: {best_loss:.4f}  {mark}")
        if patience_cnt >= PATIENCE:
            print("  ► Early stopping activado."); break

    model.load_state_dict(best_state)

    for i, fp in enumerate(Xte_f):
        avg_prob = predict_file(model, fp)
        if avg_prob is None: continue
        global_y_true.append(int(yte_f[i]))
        global_y_pred.append(1 if avg_prob > 0.33 else 0)
        global_y_prob.append(avg_prob)

    del model, train_ds, train_dl, best_state, cw_tensor
    gc.collect(); torch.cuda.empty_cache()

print("\n" + "="*60)
print(f"RESULTADOS {MODEL_NAME} | {SEGMENT_DURATION}s  ({len(global_y_true)} pacientes)")
print("="*60)

cm = confusion_matrix(global_y_true, global_y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='cool',
            xticklabels=['Sin Ansiedad', 'Con Ansiedad'],
            yticklabels=['Sin Ansiedad', 'Con Ansiedad'])
plt.title(f'Confusión {MODEL_NAME} {SEGMENT_DURATION}s ({N_FOLDS}-Fold CV)')
plt.ylabel('Real'); plt.xlabel('Predicción')
plt.tight_layout()
plt.savefig(f'confusion_{MODEL_NAME.replace(" ","_")}_{SEGMENT_DURATION}s.png', dpi=150)
plt.show()

acc      = accuracy_score(global_y_true, global_y_pred)
prec     = precision_score(global_y_true, global_y_pred, zero_division=0)
rec      = recall_score(global_y_true, global_y_pred, zero_division=0)
f1       = f1_score(global_y_true, global_y_pred, zero_division=0)
f1_macro = f1_score(global_y_true, global_y_pred, average='macro', zero_division=0)
print(f"\nAccuracy :  {acc:.4f}")
print(f"Precision:  {prec:.4f}")
print(f"Recall   :  {rec:.4f}")
print(f"F1-Ans   :  {f1:.4f}  (clase positiva)")
print(f"F1-Macro :  {f1_macro:.4f}")
print()
print(classification_report(global_y_true, global_y_pred,
                             target_names=['Sin Ansiedad', 'Con Ansiedad']))

fpr, tpr, _ = roc_curve(global_y_true, global_y_prob)
roc_auc = auc(fpr, tpr)
plt.figure(figsize=(7, 6))
plt.plot(fpr, tpr, lw=2, label=f'AUC = {roc_auc:.2f}')
plt.plot([0,1],[0,1],'navy',lw=2,linestyle='--')
plt.xlabel('Falsos Positivos'); plt.ylabel('Verdaderos Positivos')
plt.title(f'ROC — {MODEL_NAME} {SEGMENT_DURATION}s')
plt.legend(loc='lower right'); plt.tight_layout()
plt.savefig(f'roc_{MODEL_NAME.replace(" ","_")}_{SEGMENT_DURATION}s.png', dpi=150)
plt.show()


## 3 s | Overlap 50%

In [ ]:
import os
import gc
import numpy as np
import librosa
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit
from sklearn.metrics import (confusion_matrix, classification_report,
                             accuracy_score, precision_score, recall_score,
                             f1_score, roc_curve, auc)
from sklearn.utils import class_weight

# ══════════════════════════════════════════════════════════

BASE_PATH_0 = "../Clasificación Final/Ansiedad/wav/0"
BASE_PATH_1 = "../Clasificación Final/Ansiedad/wav/1"

SAMPLE_RATE         = 16000
SEGMENT_DURATION    = 3
SAMPLES_PER_SEGMENT = int(SEGMENT_DURATION * SAMPLE_RATE)  # 48000
HOP_SIZE            = SAMPLES_PER_SEGMENT // 2  # 50% overlap → 24000
N_MFCC              = 40
HOP_LENGTH          = 512
N_FFT               = 1024
N_FEATURES          = N_MFCC * 3  # MFCC + Δ + ΔΔ = 120
EXPECTED_FRAMES     = int(np.ceil(SAMPLES_PER_SEGMENT / HOP_LENGTH))  # ≈ 94
N_FOLDS             = 5
BATCH_SIZE          = 128
EPOCHS              = 25

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Usando dispositivo: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
    torch.backends.cudnn.benchmark = True

def get_file_lists():
    files_0 = [os.path.join(BASE_PATH_0, f)
               for f in sorted(os.listdir(BASE_PATH_0)) if f.endswith('.wav')]
    files_1 = [os.path.join(BASE_PATH_1, f)
               for f in sorted(os.listdir(BASE_PATH_1)) if f.endswith('.wav')]
    files  = np.array(files_0 + files_1)
    labels = np.array([0]*len(files_0) + [1]*len(files_1), dtype=np.int64)
    return files, labels

all_files, all_labels = get_file_lists()
print(f"\nTotal archivos : {len(all_files)}")
print(f" - Sin Ansiedad (0): {np.sum(all_labels == 0)}")
print(f" - Con Ansiedad (1): {np.sum(all_labels == 1)}")
print(f"Segment: {SEGMENT_DURATION}s | Features: ({EXPECTED_FRAMES}, {N_FEATURES})")

class MFCCDataset(Dataset):
    """Extrae MFCC+Δ+ΔΔ on-the-fly. Lazy loading."""
    def __init__(self, file_list, label_list):
        self.samples = []
        for fp, lbl in zip(file_list, label_list):
            try:
                total = librosa.get_duration(path=fp)
                n_segs = max(1, int((total * SAMPLE_RATE - SAMPLES_PER_SEGMENT) / HOP_SIZE) + 1)
                for i in range(n_segs):
                    offset = i * HOP_SIZE / SAMPLE_RATE
                    self.samples.append((fp, offset, int(lbl)))
            except Exception as e:
                print(f"[AVISO] {fp}: {e}")

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        fp, offset, label = self.samples[idx]
        audio, _ = librosa.load(fp, sr=SAMPLE_RATE, offset=offset,
                                duration=SEGMENT_DURATION)
        if len(audio) < SAMPLES_PER_SEGMENT:
            audio = np.pad(audio, (0, SAMPLES_PER_SEGMENT - len(audio)))
        audio = audio[:SAMPLES_PER_SEGMENT]

        mfcc = librosa.feature.mfcc(y=audio, sr=SAMPLE_RATE, n_mfcc=N_MFCC,
                                    n_fft=N_FFT, hop_length=HOP_LENGTH)
        d1   = librosa.feature.delta(mfcc)
        d2   = librosa.feature.delta(mfcc, order=2)
        feat = np.concatenate([mfcc, d1, d2], axis=0).T.astype(np.float32)  # (T, F)
        feat = (feat - feat.mean(0)) / (feat.std(0) + 1e-8)
        if feat.shape[0] < EXPECTED_FRAMES:
            feat = np.pad(feat, ((0, EXPECTED_FRAMES - feat.shape[0]), (0, 0)))
        else:
            feat = feat[:EXPECTED_FRAMES]
        return torch.from_numpy(feat), torch.tensor(label, dtype=torch.long)

def oversample_files(files, labels):
    f1 = files[labels == 1]; l1 = labels[labels == 1]
    diff = np.sum(labels == 0) - np.sum(labels == 1)
    if diff > 0:
        idx    = np.random.choice(len(f1), size=diff, replace=True)
        files  = np.concatenate([files,  f1[idx]])
        labels = np.concatenate([labels, l1[idx]])
    return files, labels

MODEL_NAME = "Transformer-MFCC"

class TransformerBlock(nn.Module):
    def __init__(self, d_model, num_heads, ff_dim, dropout=0.1):
        super().__init__()
        self.attn = nn.MultiheadAttention(d_model, num_heads, dropout=dropout,
                                          batch_first=True)
        self.ffn  = nn.Sequential(
            nn.Linear(d_model, ff_dim), nn.ReLU(inplace=True),
            nn.Linear(ff_dim, d_model)
        )
        self.ln1 = nn.LayerNorm(d_model); self.ln2 = nn.LayerNorm(d_model)
        self.dp1 = nn.Dropout(dropout);   self.dp2 = nn.Dropout(dropout)

    def forward(self, x):
        attn, _ = self.attn(x, x, x)
        x = self.ln1(x + self.dp1(attn))
        return self.ln2(x + self.dp2(self.ffn(x)))

def positional_encoding(seq_len, d_model, device):
    pos  = torch.arange(seq_len, device=device).unsqueeze(1).float()
    dims = torch.arange(d_model,  device=device).unsqueeze(0).float()
    angles = pos / (10000 ** (2 * (dims // 2) / d_model))
    angles[:, 0::2] = torch.sin(angles[:, 0::2])
    angles[:, 1::2] = torch.cos(angles[:, 1::2])
    return angles.unsqueeze(0)

D_MODEL = 64; NUM_HEADS = 4; FF_DIM = 128; NUM_BLOCKS = 2; DROPOUT_TR = 0.1

class TransformerMFCC(nn.Module):
    """Transformer sobre MFCC + Δ + ΔΔ.
    Entrada: (batch, EXPECTED_FRAMES, N_FEATURES)."""

    def __init__(self):
        super().__init__()
        self.bn  = nn.BatchNorm1d(EXPECTED_FRAMES)
        self.proj = nn.Linear(N_FEATURES, D_MODEL)
        self.tr_blocks = nn.Sequential(
            *[TransformerBlock(D_MODEL, NUM_HEADS, FF_DIM, DROPOUT_TR)
              for _ in range(NUM_BLOCKS)]
        )
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.head = nn.Sequential(
            nn.Linear(D_MODEL, 64), nn.ReLU(inplace=True), nn.Dropout(0.5),
            nn.Linear(64, 1), nn.Sigmoid()
        )

    def forward(self, x):                          # (B, T, F)
        x = self.bn(x)
        x = self.proj(x)                           # (B, T, D_MODEL)
        x = x + positional_encoding(x.size(1), D_MODEL, x.device)
        x = self.tr_blocks(x)
        x = self.pool(x.permute(0,2,1)).squeeze(2) # (B, D_MODEL)
        return self.head(x).squeeze(1)

def build_model():
    return TransformerMFCC()

def train_one_epoch(model, loader, optimizer, criterion, cw_tensor):
    model.train()
    total_loss = 0.0
    for xb, yb in loader:
        xb = xb.to(DEVICE, non_blocking=True)
        yb = yb.to(DEVICE, non_blocking=True).float()
        optimizer.zero_grad(set_to_none=True)
        preds = model(xb)
        sample_w = cw_tensor[yb.long()]
        loss = (criterion(preds, yb) * sample_w).mean()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

@torch.no_grad()
def predict_file(model, file_path):
    model.eval()
    segs = []
    try:
        audio, _ = librosa.load(file_path, sr=SAMPLE_RATE)
        for start in range(0, len(audio), HOP_SIZE):
            seg = audio[start:start + SAMPLES_PER_SEGMENT]
            if len(seg) < SAMPLES_PER_SEGMENT // 2: continue
            if len(seg) < SAMPLES_PER_SEGMENT:
                seg = np.pad(seg, (0, SAMPLES_PER_SEGMENT - len(seg)))
            seg = seg[:SAMPLES_PER_SEGMENT]
            mfcc = librosa.feature.mfcc(y=seg.astype(np.float32), sr=SAMPLE_RATE,
                                        n_mfcc=N_MFCC, n_fft=N_FFT, hop_length=HOP_LENGTH)
            d1   = librosa.feature.delta(mfcc)
            d2   = librosa.feature.delta(mfcc, order=2)
            feat = np.concatenate([mfcc, d1, d2], axis=0).T.astype(np.float32)
            feat = (feat - feat.mean(0)) / (feat.std(0) + 1e-8)
            if feat.shape[0] < EXPECTED_FRAMES:
                feat = np.pad(feat, ((0, EXPECTED_FRAMES - feat.shape[0]), (0, 0)))
            else:
                feat = feat[:EXPECTED_FRAMES]
            segs.append(feat)
    except Exception as e:
        print(f"  Error {file_path}: {e}"); return None
    if not segs: return None
    probs = []
    for b in range(0, len(segs), BATCH_SIZE):
        batch = np.stack(segs[b:b + BATCH_SIZE])
        bt = torch.from_numpy(batch).to(DEVICE)
        out = model(bt).cpu().numpy()
        probs.extend(out.tolist())
    return float(np.mean(probs))

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
global_y_true, global_y_pred, global_y_prob = [], [], []

print(f"\nINICIANDO CROSS-VALIDATION {MODEL_NAME} | {SEGMENT_DURATION}s | {N_FOLDS} FOLDS\n")

for fold_idx, (tr_idx, te_idx) in enumerate(skf.split(all_files, all_labels)):
    print(f"\n{'─'*55}\n  FOLD {fold_idx+1}/{N_FOLDS}\n{'─'*55}")

    Xtr_f, Xte_f = all_files[tr_idx], all_files[te_idx]
    ytr_f, yte_f = all_labels[tr_idx], all_labels[te_idx]
    Xtr_f, ytr_f = oversample_files(Xtr_f, ytr_f)

    train_ds = MFCCDataset(Xtr_f, ytr_f)
    train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=8, pin_memory=True, persistent_workers=True)
    print(f"  Segmentos train: {len(train_ds)}")

    weights   = class_weight.compute_class_weight('balanced',
                    classes=np.unique(ytr_f), y=ytr_f)
    cw_tensor = torch.tensor(weights, dtype=torch.float32).to(DEVICE)

    model     = build_model().to(DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=1e-4)
    criterion = nn.BCELoss(reduction='none')
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=3, min_lr=1e-7)

    if fold_idx == 0:
        n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
        print(f"  Parámetros entrenables: {n_params:,}\n")

    best_loss, patience_cnt, PATIENCE, best_state = float('inf'), 0, 5, None
    for epoch in range(EPOCHS):
        loss = train_one_epoch(model, train_dl, optimizer, criterion, cw_tensor)
        scheduler.step(loss)
        if loss < best_loss:
            best_loss = loss; patience_cnt = 0
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            mark = "✓"
        else:
            patience_cnt += 1
            mark = f"({patience_cnt}/{PATIENCE})"
        print(f"  Epoch {epoch+1:02d}/{EPOCHS} — loss: {loss:.4f}  best: {best_loss:.4f}  {mark}")
        if patience_cnt >= PATIENCE:
            print("  ► Early stopping activado."); break

    model.load_state_dict(best_state)

    for i, fp in enumerate(Xte_f):
        avg_prob = predict_file(model, fp)
        if avg_prob is None: continue
        global_y_true.append(int(yte_f[i]))
        global_y_pred.append(1 if avg_prob > 0.33 else 0)
        global_y_prob.append(avg_prob)

    del model, train_ds, train_dl, best_state, cw_tensor
    gc.collect(); torch.cuda.empty_cache()

print("\n" + "="*60)
print(f"RESULTADOS {MODEL_NAME} | {SEGMENT_DURATION}s  ({len(global_y_true)} pacientes)")
print("="*60)

cm = confusion_matrix(global_y_true, global_y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='cool',
            xticklabels=['Sin Ansiedad', 'Con Ansiedad'],
            yticklabels=['Sin Ansiedad', 'Con Ansiedad'])
plt.title(f'Confusión {MODEL_NAME} {SEGMENT_DURATION}s ({N_FOLDS}-Fold CV)')
plt.ylabel('Real'); plt.xlabel('Predicción')
plt.tight_layout()
plt.savefig(f'confusion_{MODEL_NAME.replace(" ","_")}_{SEGMENT_DURATION}s.png', dpi=150)
plt.show()

acc      = accuracy_score(global_y_true, global_y_pred)
prec     = precision_score(global_y_true, global_y_pred, zero_division=0)
rec      = recall_score(global_y_true, global_y_pred, zero_division=0)
f1       = f1_score(global_y_true, global_y_pred, zero_division=0)
f1_macro = f1_score(global_y_true, global_y_pred, average='macro', zero_division=0)
print(f"\nAccuracy :  {acc:.4f}")
print(f"Precision:  {prec:.4f}")
print(f"Recall   :  {rec:.4f}")
print(f"F1-Ans   :  {f1:.4f}  (clase positiva)")
print(f"F1-Macro :  {f1_macro:.4f}")
print()
print(classification_report(global_y_true, global_y_pred,
                             target_names=['Sin Ansiedad', 'Con Ansiedad']))

fpr, tpr, _ = roc_curve(global_y_true, global_y_prob)
roc_auc = auc(fpr, tpr)
plt.figure(figsize=(7, 6))
plt.plot(fpr, tpr, lw=2, label=f'AUC = {roc_auc:.2f}')
plt.plot([0,1],[0,1],'navy',lw=2,linestyle='--')
plt.xlabel('Falsos Positivos'); plt.ylabel('Verdaderos Positivos')
plt.title(f'ROC — {MODEL_NAME} {SEGMENT_DURATION}s')
plt.legend(loc='lower right'); plt.tight_layout()
plt.savefig(f'roc_{MODEL_NAME.replace(" ","_")}_{SEGMENT_DURATION}s.png', dpi=150)
plt.show()


## 4 s | Overlap 50%

In [ ]:
import os
import gc
import numpy as np
import librosa
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit
from sklearn.metrics import (confusion_matrix, classification_report,
                             accuracy_score, precision_score, recall_score,
                             f1_score, roc_curve, auc)
from sklearn.utils import class_weight

# ══════════════════════════════════════════════════════════

BASE_PATH_0 = "../Clasificación Final/Ansiedad/wav/0"
BASE_PATH_1 = "../Clasificación Final/Ansiedad/wav/1"

SAMPLE_RATE         = 16000
SEGMENT_DURATION    = 4
SAMPLES_PER_SEGMENT = int(SEGMENT_DURATION * SAMPLE_RATE)  # 64000
HOP_SIZE            = SAMPLES_PER_SEGMENT // 2  # 50% overlap → 32000
N_MFCC              = 40
HOP_LENGTH          = 512
N_FFT               = 1024
N_FEATURES          = N_MFCC * 3  # MFCC + Δ + ΔΔ = 120
EXPECTED_FRAMES     = int(np.ceil(SAMPLES_PER_SEGMENT / HOP_LENGTH))  # ≈ 126
N_FOLDS             = 5
BATCH_SIZE          = 128
EPOCHS              = 25

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Usando dispositivo: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
    torch.backends.cudnn.benchmark = True

def get_file_lists():
    files_0 = [os.path.join(BASE_PATH_0, f)
               for f in sorted(os.listdir(BASE_PATH_0)) if f.endswith('.wav')]
    files_1 = [os.path.join(BASE_PATH_1, f)
               for f in sorted(os.listdir(BASE_PATH_1)) if f.endswith('.wav')]
    files  = np.array(files_0 + files_1)
    labels = np.array([0]*len(files_0) + [1]*len(files_1), dtype=np.int64)
    return files, labels

all_files, all_labels = get_file_lists()
print(f"\nTotal archivos : {len(all_files)}")
print(f" - Sin Ansiedad (0): {np.sum(all_labels == 0)}")
print(f" - Con Ansiedad (1): {np.sum(all_labels == 1)}")
print(f"Segment: {SEGMENT_DURATION}s | Features: ({EXPECTED_FRAMES}, {N_FEATURES})")

class MFCCDataset(Dataset):
    """Extrae MFCC+Δ+ΔΔ on-the-fly. Lazy loading."""
    def __init__(self, file_list, label_list):
        self.samples = []
        for fp, lbl in zip(file_list, label_list):
            try:
                total = librosa.get_duration(path=fp)
                n_segs = max(1, int((total * SAMPLE_RATE - SAMPLES_PER_SEGMENT) / HOP_SIZE) + 1)
                for i in range(n_segs):
                    offset = i * HOP_SIZE / SAMPLE_RATE
                    self.samples.append((fp, offset, int(lbl)))
            except Exception as e:
                print(f"[AVISO] {fp}: {e}")

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        fp, offset, label = self.samples[idx]
        audio, _ = librosa.load(fp, sr=SAMPLE_RATE, offset=offset,
                                duration=SEGMENT_DURATION)
        if len(audio) < SAMPLES_PER_SEGMENT:
            audio = np.pad(audio, (0, SAMPLES_PER_SEGMENT - len(audio)))
        audio = audio[:SAMPLES_PER_SEGMENT]

        mfcc = librosa.feature.mfcc(y=audio, sr=SAMPLE_RATE, n_mfcc=N_MFCC,
                                    n_fft=N_FFT, hop_length=HOP_LENGTH)
        d1   = librosa.feature.delta(mfcc)
        d2   = librosa.feature.delta(mfcc, order=2)
        feat = np.concatenate([mfcc, d1, d2], axis=0).T.astype(np.float32)  # (T, F)
        feat = (feat - feat.mean(0)) / (feat.std(0) + 1e-8)
        if feat.shape[0] < EXPECTED_FRAMES:
            feat = np.pad(feat, ((0, EXPECTED_FRAMES - feat.shape[0]), (0, 0)))
        else:
            feat = feat[:EXPECTED_FRAMES]
        return torch.from_numpy(feat), torch.tensor(label, dtype=torch.long)

def oversample_files(files, labels):
    f1 = files[labels == 1]; l1 = labels[labels == 1]
    diff = np.sum(labels == 0) - np.sum(labels == 1)
    if diff > 0:
        idx    = np.random.choice(len(f1), size=diff, replace=True)
        files  = np.concatenate([files,  f1[idx]])
        labels = np.concatenate([labels, l1[idx]])
    return files, labels

MODEL_NAME = "Transformer-MFCC"

class TransformerBlock(nn.Module):
    def __init__(self, d_model, num_heads, ff_dim, dropout=0.1):
        super().__init__()
        self.attn = nn.MultiheadAttention(d_model, num_heads, dropout=dropout,
                                          batch_first=True)
        self.ffn  = nn.Sequential(
            nn.Linear(d_model, ff_dim), nn.ReLU(inplace=True),
            nn.Linear(ff_dim, d_model)
        )
        self.ln1 = nn.LayerNorm(d_model); self.ln2 = nn.LayerNorm(d_model)
        self.dp1 = nn.Dropout(dropout);   self.dp2 = nn.Dropout(dropout)

    def forward(self, x):
        attn, _ = self.attn(x, x, x)
        x = self.ln1(x + self.dp1(attn))
        return self.ln2(x + self.dp2(self.ffn(x)))

def positional_encoding(seq_len, d_model, device):
    pos  = torch.arange(seq_len, device=device).unsqueeze(1).float()
    dims = torch.arange(d_model,  device=device).unsqueeze(0).float()
    angles = pos / (10000 ** (2 * (dims // 2) / d_model))
    angles[:, 0::2] = torch.sin(angles[:, 0::2])
    angles[:, 1::2] = torch.cos(angles[:, 1::2])
    return angles.unsqueeze(0)

D_MODEL = 64; NUM_HEADS = 4; FF_DIM = 128; NUM_BLOCKS = 2; DROPOUT_TR = 0.1

class TransformerMFCC(nn.Module):
    """Transformer sobre MFCC + Δ + ΔΔ.
    Entrada: (batch, EXPECTED_FRAMES, N_FEATURES)."""

    def __init__(self):
        super().__init__()
        self.bn  = nn.BatchNorm1d(EXPECTED_FRAMES)
        self.proj = nn.Linear(N_FEATURES, D_MODEL)
        self.tr_blocks = nn.Sequential(
            *[TransformerBlock(D_MODEL, NUM_HEADS, FF_DIM, DROPOUT_TR)
              for _ in range(NUM_BLOCKS)]
        )
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.head = nn.Sequential(
            nn.Linear(D_MODEL, 64), nn.ReLU(inplace=True), nn.Dropout(0.5),
            nn.Linear(64, 1), nn.Sigmoid()
        )

    def forward(self, x):                          # (B, T, F)
        x = self.bn(x)
        x = self.proj(x)                           # (B, T, D_MODEL)
        x = x + positional_encoding(x.size(1), D_MODEL, x.device)
        x = self.tr_blocks(x)
        x = self.pool(x.permute(0,2,1)).squeeze(2) # (B, D_MODEL)
        return self.head(x).squeeze(1)

def build_model():
    return TransformerMFCC()

def train_one_epoch(model, loader, optimizer, criterion, cw_tensor):
    model.train()
    total_loss = 0.0
    for xb, yb in loader:
        xb = xb.to(DEVICE, non_blocking=True)
        yb = yb.to(DEVICE, non_blocking=True).float()
        optimizer.zero_grad(set_to_none=True)
        preds = model(xb)
        sample_w = cw_tensor[yb.long()]
        loss = (criterion(preds, yb) * sample_w).mean()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

@torch.no_grad()
def predict_file(model, file_path):
    model.eval()
    segs = []
    try:
        audio, _ = librosa.load(file_path, sr=SAMPLE_RATE)
        for start in range(0, len(audio), HOP_SIZE):
            seg = audio[start:start + SAMPLES_PER_SEGMENT]
            if len(seg) < SAMPLES_PER_SEGMENT // 2: continue
            if len(seg) < SAMPLES_PER_SEGMENT:
                seg = np.pad(seg, (0, SAMPLES_PER_SEGMENT - len(seg)))
            seg = seg[:SAMPLES_PER_SEGMENT]
            mfcc = librosa.feature.mfcc(y=seg.astype(np.float32), sr=SAMPLE_RATE,
                                        n_mfcc=N_MFCC, n_fft=N_FFT, hop_length=HOP_LENGTH)
            d1   = librosa.feature.delta(mfcc)
            d2   = librosa.feature.delta(mfcc, order=2)
            feat = np.concatenate([mfcc, d1, d2], axis=0).T.astype(np.float32)
            feat = (feat - feat.mean(0)) / (feat.std(0) + 1e-8)
            if feat.shape[0] < EXPECTED_FRAMES:
                feat = np.pad(feat, ((0, EXPECTED_FRAMES - feat.shape[0]), (0, 0)))
            else:
                feat = feat[:EXPECTED_FRAMES]
            segs.append(feat)
    except Exception as e:
        print(f"  Error {file_path}: {e}"); return None
    if not segs: return None
    probs = []
    for b in range(0, len(segs), BATCH_SIZE):
        batch = np.stack(segs[b:b + BATCH_SIZE])
        bt = torch.from_numpy(batch).to(DEVICE)
        out = model(bt).cpu().numpy()
        probs.extend(out.tolist())
    return float(np.mean(probs))

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
global_y_true, global_y_pred, global_y_prob = [], [], []

print(f"\nINICIANDO CROSS-VALIDATION {MODEL_NAME} | {SEGMENT_DURATION}s | {N_FOLDS} FOLDS\n")

for fold_idx, (tr_idx, te_idx) in enumerate(skf.split(all_files, all_labels)):
    print(f"\n{'─'*55}\n  FOLD {fold_idx+1}/{N_FOLDS}\n{'─'*55}")

    Xtr_f, Xte_f = all_files[tr_idx], all_files[te_idx]
    ytr_f, yte_f = all_labels[tr_idx], all_labels[te_idx]
    Xtr_f, ytr_f = oversample_files(Xtr_f, ytr_f)

    train_ds = MFCCDataset(Xtr_f, ytr_f)
    train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=8, pin_memory=True, persistent_workers=True)
    print(f"  Segmentos train: {len(train_ds)}")

    weights   = class_weight.compute_class_weight('balanced',
                    classes=np.unique(ytr_f), y=ytr_f)
    cw_tensor = torch.tensor(weights, dtype=torch.float32).to(DEVICE)

    model     = build_model().to(DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=1e-4)
    criterion = nn.BCELoss(reduction='none')
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=3, min_lr=1e-7)

    if fold_idx == 0:
        n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
        print(f"  Parámetros entrenables: {n_params:,}\n")

    best_loss, patience_cnt, PATIENCE, best_state = float('inf'), 0, 5, None
    for epoch in range(EPOCHS):
        loss = train_one_epoch(model, train_dl, optimizer, criterion, cw_tensor)
        scheduler.step(loss)
        if loss < best_loss:
            best_loss = loss; patience_cnt = 0
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            mark = "✓"
        else:
            patience_cnt += 1
            mark = f"({patience_cnt}/{PATIENCE})"
        print(f"  Epoch {epoch+1:02d}/{EPOCHS} — loss: {loss:.4f}  best: {best_loss:.4f}  {mark}")
        if patience_cnt >= PATIENCE:
            print("  ► Early stopping activado."); break

    model.load_state_dict(best_state)

    for i, fp in enumerate(Xte_f):
        avg_prob = predict_file(model, fp)
        if avg_prob is None: continue
        global_y_true.append(int(yte_f[i]))
        global_y_pred.append(1 if avg_prob > 0.33 else 0)
        global_y_prob.append(avg_prob)

    del model, train_ds, train_dl, best_state, cw_tensor
    gc.collect(); torch.cuda.empty_cache()

print("\n" + "="*60)
print(f"RESULTADOS {MODEL_NAME} | {SEGMENT_DURATION}s  ({len(global_y_true)} pacientes)")
print("="*60)

cm = confusion_matrix(global_y_true, global_y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='cool',
            xticklabels=['Sin Ansiedad', 'Con Ansiedad'],
            yticklabels=['Sin Ansiedad', 'Con Ansiedad'])
plt.title(f'Confusión {MODEL_NAME} {SEGMENT_DURATION}s ({N_FOLDS}-Fold CV)')
plt.ylabel('Real'); plt.xlabel('Predicción')
plt.tight_layout()
plt.savefig(f'confusion_{MODEL_NAME.replace(" ","_")}_{SEGMENT_DURATION}s.png', dpi=150)
plt.show()

acc      = accuracy_score(global_y_true, global_y_pred)
prec     = precision_score(global_y_true, global_y_pred, zero_division=0)
rec      = recall_score(global_y_true, global_y_pred, zero_division=0)
f1       = f1_score(global_y_true, global_y_pred, zero_division=0)
f1_macro = f1_score(global_y_true, global_y_pred, average='macro', zero_division=0)
print(f"\nAccuracy :  {acc:.4f}")
print(f"Precision:  {prec:.4f}")
print(f"Recall   :  {rec:.4f}")
print(f"F1-Ans   :  {f1:.4f}  (clase positiva)")
print(f"F1-Macro :  {f1_macro:.4f}")
print()
print(classification_report(global_y_true, global_y_pred,
                             target_names=['Sin Ansiedad', 'Con Ansiedad']))

fpr, tpr, _ = roc_curve(global_y_true, global_y_prob)
roc_auc = auc(fpr, tpr)
plt.figure(figsize=(7, 6))
plt.plot(fpr, tpr, lw=2, label=f'AUC = {roc_auc:.2f}')
plt.plot([0,1],[0,1],'navy',lw=2,linestyle='--')
plt.xlabel('Falsos Positivos'); plt.ylabel('Verdaderos Positivos')
plt.title(f'ROC — {MODEL_NAME} {SEGMENT_DURATION}s')
plt.legend(loc='lower right'); plt.tight_layout()
plt.savefig(f'roc_{MODEL_NAME.replace(" ","_")}_{SEGMENT_DURATION}s.png', dpi=150)
plt.show()
